
# PIF engine stress tests

This notebook starts after expand_pif.ipynb has already produced reusable artifacts. The goal is to avoid re-reading or rebuilding the same ENPG, mortality, and AAF inputs in this second pass.

Working order:

1. Locate __andres_control from the current Positron session.
2. Find and load the latest dated artifacts of each type.
3. Expose the saved exposure and AAF objects produced by expand_pif.ipynb.
4. Define PIF counterfactual scenarios for consumption volume and HED prevalence.
5. Run optional smoke tests before any full Monte Carlo run.
6. Combine PIF outputs with cached YPLL.
7. Run assertion checks before exporting any policy-counterfactual result.


In [1]:

#| label: pif2-setup
#| results: "hold"

.t0 <- Sys.time()
# Keep the second-stage notebook gentle in an interactive Positron session.
# Set this to TRUE only when you intentionally want to remove objects from memory.
pif2_clear_workspace <- FALSE
if (isTRUE(pif2_clear_workspace)) {
  rm(list = ls())
  invisible(gc())
}
# The notebook is verbose by default so users can see which files are selected,
# why they are selected, and which objects become available downstream.
pif2_verbose <- TRUE
# Use explicit namespace calls instead of library() so the notebook remains clear
# about where each helper comes from.
pif2_required_packages <- c("dplyr", "tidyr", "purrr", "tibble", "readxl", "knitr")
pif2_missing_packages <- pif2_required_packages[
  !vapply(pif2_required_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(pif2_missing_packages) > 0) {
  stop("Missing required package(s): ", paste(pif2_missing_packages, collapse = ", "))
}
message(sprintf(
  "pif2-setup elapsed minutes: %.2f",
  as.numeric(difftime(Sys.time(), .t0, units = "mins"))
))


pif2-setup elapsed minutes: 0.00



## Latest-artifact helper

The helper below is intentionally verbose. It searches by file type or pattern, extracts dates from filenames when possible, and selects the newest candidate by embedded date first and modification time second. If no filename date exists, it falls back to modification time.


In [2]:

#| label: pif2-latest-artifact-helpers
#| results: "hold"

.t0 <- Sys.time()

pif2_elapsed_min <- function(.start = .t0) {
  as.numeric(difftime(Sys.time(), .start, units = "mins"))
}
pif2_message <- function(..., verbose = pif2_verbose) {
  if (isTRUE(verbose)) {
    message(sprintf(...))
  }
}
pif2_find_control_dir <- function(verbose = pif2_verbose) {
  # Positron can run a notebook from the project root, from __andres_control,
  # or from another working directory. This helper tries all common locations
  # and returns the directory that contains the PIF/AAF engine files.
  candidates <- unique(c(
    normalizePath(".", winslash = "/", mustWork = FALSE),
    normalizePath(file.path(".", "__andres_control"), winslash = "/", mustWork = FALSE),
    normalizePath(file.path("..", "__andres_control"), winslash = "/", mustWork = FALSE)
  ))
  hits <- candidates[
    file.exists(file.path(candidates, "aaf_unified.R")) &
      file.exists(file.path(candidates, "rr_registry_adam.R"))
  ]
  if (!length(hits)) {
    stop("Could not find __andres_control with aaf_unified.R and rr_registry_adam.R from getwd(): ", getwd())
  }
  control_dir <- normalizePath(hits[[1L]], winslash = "/", mustWork = TRUE)
  pif2_message("[control-dir] Using: %s", control_dir, verbose = verbose)
  control_dir
}
# Define project root
pif2_project_root <- function(control_dir) {
  normalizePath(file.path(control_dir, ".."), winslash = "/", mustWork = TRUE)
}
pif2_extract_embedded_date <- function(file_name, date_regex = "(\\d{8})") {
  # Return a Date parsed from YYYYMMDD inside the filename. If the filename does
  # not carry an 8-digit date, return NA so the caller can fall back to mtime.
  hit <- regmatches(file_name, regexpr(date_regex, file_name, perl = TRUE))
  if (!length(hit) || is.na(hit) || !nzchar(hit)) {
    return(as.Date(NA))
  }
  as.Date(hit, format = "%Y%m%d")
}
pif2_latest_matching_file <- function(directory,
                                      pattern,
                                      date_regex = "(\\d{8})",
                                      recursive = FALSE,
                                      prefer_embedded_date = TRUE,
                                      verbose = pif2_verbose) {
  # This is the generic selector. It is deliberately not tied to one artifact
  # name, so it can be reused for any future output type from expand_pif.ipynb.
  directory <- normalizePath(directory, winslash = "/", mustWork = TRUE)
  pif2_message("[latest-file] Directory: %s", directory, verbose = verbose)
  pif2_message("[latest-file] Pattern: %s", pattern, verbose = verbose)
  files <- list.files(
    path = directory,
    pattern = pattern,
    full.names = TRUE,
    recursive = recursive,
    ignore.case = FALSE
  )
  if (!length(files)) {
    stop("No files matched pattern '", pattern, "' under ", directory)
  }
  info <- file.info(files)
  candidates <- tibble::tibble(
    path = normalizePath(files, winslash = "/", mustWork = TRUE),
    file_name = basename(files),
    embedded_date = as.Date(vapply(basename(files), pif2_extract_embedded_date, as.Date(NA), date_regex = date_regex)),
    modified_time = info$mtime,
    size_bytes = info$size
  )
  has_dates <- any(!is.na(candidates$embedded_date))
  if (isTRUE(prefer_embedded_date) && has_dates) {
    candidates <- candidates |>
      dplyr::arrange(dplyr::desc(.data$embedded_date), dplyr::desc(.data$modified_time), dplyr::desc(.data$size_bytes))
    pif2_message("[latest-file] Selection rule: embedded YYYYMMDD date, then modified time.", verbose = verbose)
  } else {
    candidates <- candidates |>
      dplyr::arrange(dplyr::desc(.data$modified_time), dplyr::desc(.data$size_bytes))
    pif2_message("[latest-file] Selection rule: modified time fallback.", verbose = verbose)
  }

  pif2_message("[latest-file] Candidate count: %s", nrow(candidates), verbose = verbose)
  pif2_message("[latest-file] Selected: %s", candidates$file_name[[1L]], verbose = verbose)
  attr(candidates$path[[1L]], "candidates") <- candidates
  candidates$path[[1L]]
}
pif2_latest_dated_file <- function(directory,
                                   stem,
                                   extension = "rds",
                                   verbose = pif2_verbose) {
  # Convenience wrapper for artifacts named like stem_YYYYMMDD.extension.
  escaped_stem <- gsub("([.|()\\^{}+$*?]|\\[|\\])", "\\\\\\1", stem, perl = TRUE)
  escaped_extension <- gsub("([.|()\\^{}+$*?]|\\[|\\])", "\\\\\\1", extension, perl = TRUE)
  pattern <- paste0("^", escaped_stem, "_\\d{8}\\.", escaped_extension, "$")
  pif2_latest_matching_file(directory, pattern = pattern, verbose = verbose)
}
pif2_read_latest_artifact <- function(directory,
                                      pattern = NULL,
                                      stem = NULL,
                                      extension = "rds",
                                      reader = NULL,
                                      verbose = pif2_verbose) {
  # Find the latest file and read it with the right parser. The return value keeps
  # the selected path in attr(x, "path") so later cells can report provenance.
  selected_path <- if (!is.null(pattern)) {
    pif2_latest_matching_file(directory, pattern = pattern, verbose = verbose)
  } else {
    pif2_latest_dated_file(directory, stem = stem, extension = extension, verbose = verbose)
  }
  if (is.null(reader)) {
    ext <- tolower(tools::file_ext(selected_path))
    reader <- switch(
      ext,
      rds = base::readRDS,
      xlsx = readxl::read_xlsx,
      csv = utils::read.csv,
      stop("No default reader for extension: ", ext)
    )
  }
  pif2_message("[read-artifact] Reading: %s", selected_path, verbose = verbose)
  out <- reader(selected_path)
  attr(out, "path") <- selected_path
  out
}
message(sprintf(
  "pif2-latest-artifact-helpers elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

pif2-latest-artifact-helpers elapsed minutes: 0.00



## Load saved outputs from expand_pif.ipynb

This section loads the latest saved bundles and exposes the objects needed by the PIF engine. It sources engine code, but it does not rebuild ENPG exposure objects or mortality tables from raw files.


In [3]:
#| label: pif2-load-latest-artifacts
#| results: "hold"

.t0 <- Sys.time()
# define directoriess
pif2_control_dir <- pif2_find_control_dir()
pif2_root_dir <- pif2_project_root(pif2_control_dir)

# Source code helpers only. These are lightweight compared with rebuilding ENPG,
# mortality, and AAF objects from raw data.
source(file.path(pif2_control_dir, "aaf_unified.R"))
source(file.path(pif2_control_dir, "rr_registry_adam.R"))
pif2_message("[engine] Loaded aaf_unified.R and rr_registry_adam.R")
# Loaded latest PIF artifacts, mortality results, previous YPLLs
pif2_input_bundle <- pif2_read_latest_artifact(
  directory = pif2_control_dir,
  stem = "aaf_engine_inputs_bundle",
  extension = "rds"
)
pif2_nested_bundle <- pif2_read_latest_artifact(
  directory = pif2_control_dir,
  stem = "aaf_nested_by_disease",
  extension = "rds"
)
pif2_mortality_results <- pif2_read_latest_artifact(
  directory = pif2_control_dir,
  pattern   = "^Mortality Estimates WHO 2024(_[0-9]{8})?\\.xlsx$",
  reader    = readxl::read_xlsx
)
pif2_ypll_cache <- tryCatch(
  pif2_read_latest_artifact(
    directory = file.path(pif2_root_dir, "Mortalidad", "Matrices"),
    pattern = "^YPLL.*\\.rds$",
    reader = base::readRDS
  ),
  error = function(e) {
    pif2_message("[YPLL] No cached YPLL file loaded: %s", conditionMessage(e))
    NULL
  }
)
pif2_message("[bundle] Exposure input bundle: %s", basename(attr(pif2_input_bundle, "path")))
pif2_message("[bundle] Nested AAF bundle: %s", basename(attr(pif2_nested_bundle, "path")))
pif2_message("[bundle] Mortality table: %s", basename(attr(pif2_mortality_results, "path")))
if (!is.null(pif2_ypll_cache)) {
  pif2_message("[bundle] YPLL cache: %s", basename(attr(pif2_ypll_cache, "path")))
}
message(sprintf(
  "pif2-load-latest-artifacts elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[control-dir] Using: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control
[engine] Loaded aaf_unified.R and rr_registry_adam.R
[latest-file] Directory: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control
[latest-file] Pattern: ^aaf_engine_inputs_bundle_\d{8}\.rds$
[latest-file] Selection rule: embedded YYYYMMDD date, then modified time.
[latest-file] Candidate count: 4
[latest-file] Selected: aaf_engine_inputs_bundle_20260715.rds
[read-artifact] Reading: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control/aaf_engine_inputs_bundle_20260715.rds
[latest-file] Directory: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control
[latest-file] Pattern: ^aaf_nested_by_disease_\d{8}\.rds$
[latest-file] Selection rule: embedded YYYYMMDD date, then modified time.
[latest-file] Candidate count: 4
[latest-file] Selected: aaf_nested_by_disease_20260715.rds
[read-artifact] Reading: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control/aaf_nested_by_disease_20260715.rds
[latest-file] 

Then, we loaded the registries for PIF scenarios.

In [4]:
#| label: pif2-expose-saved-objects
#| results: "hold"

.t0 <- Sys.time()
# The input bundle stores the exposure objects that were already built in
# expand_pif.ipynb. We expose them both as a named list and as ordinary objects
# because the existing engine functions expect names like g_fem_hed_list.
pif2_exposure_inputs <- pif2_input_bundle$exposure_inputs
base::list2env(pif2_exposure_inputs, envir = .GlobalEnv)
# Get the survey years
pif2_years <- as.integer(pif2_input_bundle$metadata$survey_years)
if (!length(pif2_years)) {
  pif2_years <- as.integer(names(pif2_exposure_inputs$g_fem_hed_list))
}
# Reuse the EXACT Monte Carlo settings and design closures the PAF run used, so
# the PIF is congruent with the validated PAF by construction:
#   * aaf_mc          -> list(n_sim, n_pca, seed, n_cores)
#   * aaf_uncertainty -> prev_method + neff/design_factor/neff_consumption/
#                        design_factor_consumption as function(year, group, sex)
#                        closures that read the survey design table (Kish n and
#                        PSU/region cluster factor). We do NOT rebuild them here.
pif2_aaf_mc          <- pif2_nested_bundle$inputs$aaf_mc
pif2_aaf_uncertainty <- pif2_nested_bundle$inputs$aaf_uncertainty
pif2_aaf_audit       <- pif2_nested_bundle$audits$aaf_adam_rr_audit
pif2_aaf_errors      <- pif2_nested_bundle$audits$aaf_adam_rr_errors

# ---------------------------------------------------------------------------
# Load the COMPLETE validated RR registry: ALL SIX scopes (not the former
# three-scope subset). These are the same RR-curve objects the PAF used, so the
# PIF can cover every one of the 23 pipeline diseases the AAF covers.
# ---------------------------------------------------------------------------
pif2_rr_registries <- list(
  cancer           = load_adam_rr_registry(scope = "cancer",   control_dir = pif2_control_dir, verbose = FALSE),
  hhd              = load_adam_rr_registry(scope = "hhd",      control_dir = pif2_control_dir, verbose = FALSE),
  general          = load_adam_rr_registry(scope = "general",  control_dir = pif2_control_dir, verbose = FALSE),
  ihd              = load_adam_rr_registry(scope = "ihd",      control_dir = pif2_control_dir, verbose = FALSE),
  ischaemic_stroke = load_adam_rr_registry(scope = "is",       control_dir = pif2_control_dir, verbose = FALSE),
  injuries         = load_adam_rr_registry(scope = "injuries", control_dir = pif2_control_dir, verbose = FALSE)
)

# Validate every registry BEFORE it is used: RR curves must be finite and
# non-negative, beta/covariance dimensions must match, covariance must be
# symmetric with a non-negative diagonal (this is validate_adam_rr_registry()).
pif2_registry_validation <- vapply(names(pif2_rr_registries), function(nm) {
  tryCatch({ validate_adam_rr_registry(pif2_rr_registries[[nm]]); "ok" },
           error = function(e) conditionMessage(e))
}, character(1))
if (any(pif2_registry_validation != "ok")) {
  stop("RR registry validation failed: ",
       paste(names(which(pif2_registry_validation != "ok")), collapse = ", "))
}

# Flat index for O(1) record lookup by (source_object, sex) for the non-age-banded
# families (cancer / hhd / general / injuries). CV (ihd / ischaemic_stroke) is
# age-banded and looked up separately by (sex, adam_age_band).
pif2_record_index <- local({
  idx <- list()
  for (scope in c("cancer", "hhd", "general", "injuries")) {
    for (rec in pif2_rr_registries[[scope]]) {
      idx[[paste(rec$source_object, rec$sex, sep = "::")]] <- rec
    }
  }
  idx
})

# ---------------------------------------------------------------------------
# Disease x sex x mode grid, DERIVED from the AAF audit (nb$audits$aaf_adam_rr_audit,
# 45 output tables). Driving the PIF from the PAF's own audit guarantees that the
# PIF covers EXACTLY the same output tables, modes (none/cap/explicit), and
# former-drinker settings the PAF used -- congruence by construction.
# ---------------------------------------------------------------------------
pif2_build_output_spec <- function(aaf_audit) {
  mode <- dplyr::recode(aaf_audit$hed_mode,
                        "none" = "nohed", "cap" = "cap", "explicit" = "explicit")
  tibble::tibble(
    output_name    = aaf_audit$output_name,
    disease        = aaf_audit$disease,            # pipeline label (e.g. "Other Pharyngeal Cancer")
    sex            = aaf_audit$sex,
    source_object  = aaf_audit$source_object,
    mode           = mode,                         # nohed | cap | explicit
    uses_hed       = mode %in% c("cap", "explicit"),
    age_banded     = mode == "cap",
    fd_uncertainty = as.logical(aaf_audit$fd_uncertainty),
    registry_scope = dplyr::case_when(
      mode == "explicit" ~ "injuries",
      mode == "cap" & grepl("Heart", aaf_audit$disease) ~ "ihd",
      mode == "cap" ~ "ischaemic_stroke",
      TRUE ~ NA_character_                         # nohed: resolved via pif2_record_index
    ),
    prefix = ifelse(aaf_audit$sex == "female", "Fem", "Male")
  )
}
pif2_output_spec <- pif2_build_output_spec(pif2_aaf_audit)

# Resolve the RR record for one spec row and one age group. CV records are chosen
# by Adam age band (15_64 scope: group 4 = 60-64 folds into the 35-64 band);
# every other family reuses one record across the four groups.
pif2_lookup_record <- function(spec_row, group, age_scope = "15_64") {
  if (isTRUE(spec_row$age_banded)) {
    band_map <- aaf_age_band_mapping(age_scope)
    band <- band_map$adam_age_band[band_map$group == group]
    return(.aaf_find_one(pif2_rr_registries[[spec_row$registry_scope]],
                         sex = spec_row$sex, adam_age_band = band))
  }
  rec <- pif2_record_index[[paste(spec_row$source_object, spec_row$sex, sep = "::")]]
  if (is.null(rec)) stop("No RR record for ", spec_row$source_object, " / ", spec_row$sex)
  rec$pipeline_disease <- spec_row$disease   # keep the AAF pipeline label (oral vs other-pharyngeal share one RR)
  rec
}

pif2_message("[objects] Exposed exposure objects: %s", paste(names(pif2_exposure_inputs), collapse = ", "))
pif2_message("[objects] Survey years: %s", paste(pif2_years, collapse = ", "))
pif2_message("[objects] Loaded + validated RR registries: %s", paste(names(pif2_rr_registries), collapse = ", "))
pif2_message("[objects] Registry record counts: %s",
             paste(names(pif2_rr_registries), vapply(pif2_rr_registries, length, integer(1)), sep = "=", collapse = ", "))
pif2_message("[objects] Output spec rows (PIF output tables): %s", nrow(pif2_output_spec))
pif2_message("[objects] Distinct pipeline diseases in spec: %s", dplyr::n_distinct(pif2_output_spec$disease))
pif2_message("[objects] Modes: %s",
             paste(names(table(pif2_output_spec$mode)), as.integer(table(pif2_output_spec$mode)), sep = "=", collapse = ", "))
pif2_message("[objects] MC settings reused from PAF: n_sim=%s n_pca=%s seed=%s",
             pif2_aaf_mc$n_sim, pif2_aaf_mc$n_pca, pif2_aaf_mc$seed)
pif2_message("[objects] Design closures reused: neff/design_factor are function(year,group,sex) = %s",
             is.function(pif2_aaf_uncertainty$neff))
message(sprintf(
  "pif2-expose-saved-objects elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


[objects] Exposed exposure objects: g_fem_list, g_male_list, g_fem_hed_list, g_male_hed_list, p_abs_list_fem, p_abs_list_male, p_form_list_fem, p_form_list_male, p_hed_list_fem, p_hed_list_male, x_vals, x_vals_nhed, x_vals_hed
[objects] Survey years: 2012, 2014, 2016, 2018, 2020, 2022, 2024
[objects] Loaded + validated RR registries: cancer, hhd, general, ihd, ischaemic_stroke, injuries
[objects] Registry record counts: cancer=15, hhd=2, general=16, ihd=6, ischaemic_stroke=6, injuries=6
[objects] Output spec rows (PIF output tables): 45
[objects] Distinct pipeline diseases in spec: 23
[objects] Modes: cap=4, explicit=6, nohed=35
[objects] MC settings reused from PAF: n_sim=10000 n_pca=1000 seed=2125
[objects] Design closures reused: neff/design_factor are function(year,group,sex) = TRUE
pif2-expose-saved-objects elapsed minutes: 0.00


## AAF long table from saved bundle

The nested bundle stores the AAF output tables by family. This cell reconstructs a long AAF table directly from that saved bundle, avoiding the old display tables and avoiding any raw-data rebuild.


In [5]:
#| label: pif2-aaf-long-from-bundle
#| results: "hold"

.t0 <- Sys.time()

pif2_wide_aaf_to_long <- function(wide_tbl, output_name = NA_character_) {
  # Convert one saved wide AAF table into the long schema used by downstream joins.
  # The saved tables use Fem1_point/Fem1_lower/Fem1_upper or Male1_* columns.
  wide_tbl <- tibble::as_tibble(wide_tbl)
  prefix <- if (any(grepl("^Fem\\d+_point$", names(wide_tbl)))) {
    "Fem"
  } else if (any(grepl("^Male\\d+_point$", names(wide_tbl)))) {
    "Male"
  } else {
    stop("Could not detect sex prefix in AAF table: ", output_name)
  }
# Format gender and age groups to numbers
  gender_value <- if (identical(prefix, "Fem")) "Mujer" else "Hombre"
  age_groups <- sort(as.integer(gsub(
    paste0("^", prefix, "(\\d+)_point$"),
    "\\1",
    grep(paste0("^", prefix, "\\d+_point$"), names(wide_tbl), value = TRUE)
  )))
# Format table
  purrr::map_dfr(age_groups, function(age_group_value) {
    tibble::tibble(
      year = wide_tbl$Year,
      age_group = age_group_value,
      gender = gender_value,
      disease = wide_tbl$disease,
      point = wide_tbl[[paste0(prefix, age_group_value, "_point")]],
      lower = wide_tbl[[paste0(prefix, age_group_value, "_lower")]],
      upper = wide_tbl[[paste0(prefix, age_group_value, "_upper")]],
      output_name = output_name
    )
  })
}
# function: wide to long
pif2_aaf_long_from_nested <- function(nested_bundle) {
  tables <- purrr::imap(
    nested_bundle$family_bundles,
    function(family_bundle, family_name) {
      purrr::imap_dfr(
        family_bundle$raw_result$tables,
        function(wide_tbl, output_name) {
          pif2_wide_aaf_to_long(wide_tbl, output_name = output_name) |>
            dplyr::mutate(family = family_name)
        }
      )
    }
  )
  dplyr::bind_rows(tables) |>
    dplyr::distinct(.data$year, .data$age_group, .data$gender, .data$disease, .data$output_name, .keep_all = TRUE) |>
    dplyr::arrange(.data$disease, .data$year, .data$age_group, .data$gender)
}
# deploy wide to long
pif2_aaf_long <- pif2_aaf_long_from_nested(pif2_nested_bundle)

pif2_message("[aaf-long] Rows: %s", nrow(pif2_aaf_long))
pif2_message("[aaf-long] Diseases: %s", dplyr::n_distinct(pif2_aaf_long$disease))
pif2_message("[aaf-long] Year range: %s-%s", min(pif2_aaf_long$year), max(pif2_aaf_long$year))

message(sprintf(
  "pif2-aaf-long-from-bundle elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[aaf-long] Rows: 1260
[aaf-long] Diseases: 23
[aaf-long] Year range: 2012-2024
pif2-aaf-long-from-bundle elapsed minutes: 0.00


## Result tables

Every table below only FORMATS objects that the chunks above already computed: no PIF,
AAF or mortality quantity is recomputed. The idiom is the one used throughout
`expand_pif.ipynb` -- `knitr::kable(format = "html")` wrapped in `htmltools::browsable()`
inside a scrollable div -- so large tables scroll instead of overflowing the page.

The full 12,600-row PIF grid is deliberately NOT printed: it would bloat the
self-contained HTML and no reader can use it. It stays in `pif2_pif_results_*.rds`. The
headline table (one wave) and the death-weighted cause-group tables carry the result.

In [6]:
#| label: pif2-table-helpers
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# Display helpers for the result tables.
#
# These helpers only FORMAT objects that the other chunks already built: no PIF,
# AAF or mortality quantity is recomputed here. The idiom is the same one used
# throughout expand_pif.ipynb: knitr::kable(format = "html") wrapped in
# htmltools::browsable() inside a scrollable div, with a bold caption and an
# optional explanatory note underneath it.
# ---------------------------------------------------------------------------

# Reporting year for the headline tables. The PIF grid covers every survey wave;
# printing all of them at once would make the rendered HTML unreadable, so the
# headline tables default to the latest wave.
pif2_report_year <- max(pif2_years)

# Age-group labels. edad_tramo 4 is 60-65: expand_pif.ipynb builds the age bands
# with between(edad, 60, 65), and the deaths and ENPG frames are both 15-65.
pif2_age_labels <- c("1" = "15-29", "2" = "30-44", "3" = "45-59", "4" = "60-65")
pif2_sex_labels <- c("male" = "Men", "female" = "Women")

# Spanish gender labels, as used by the mortality table (Hombre/Mujer). Defined
# here under its own name so the table chunks never depend on the YPLL bridge cell
# further down (which defines a similar helper for its own joins).
pif2_gender_es <- function(sex) {
  ifelse(sex %in% c("male", "Men", "Hombre"), "Hombre", "Mujer")
}

# Cause groups: the SAME five groups the figures of expand_pif.ipynb use
# (mortality_results_cat). "Uncategorized" fires only if a disease escapes the map.
pif2_cause_category <- function(disease) {
  dplyr::case_when(
    disease %in% c("Breast Cancer", "Colon and rectum Cancer", "Larynx Cancer",
                   "Lip and Oral Cavity Cancer", "Liver Cancer", "Oesophagus Cancer",
                   "Other Pharyngeal Cancer", "Oral Cavity and Pharynx Cancer",
                   "Stomach Cancer", "Pancreatic Cancer") ~ "Cancer",
    disease %in% c("Intentional Injuries", "Road Injuries", "Unintentional Injuries") ~ "Injuries",
    disease %in% c("Hypertensive Heart Disease", "Intracerebral Haemorrhage",
                   "Ischaemic Heart Disease", "Ischaemic Stroke") ~ "Cardiovascular",
    disease %in% c("DM2", "Liver Cirrhosis", "Lower Respiratory Infection",
                   "Tuberculosis", "Acute Pancreatitis", "HIV") ~ "Other Causes",
    disease == "Epilepsy" ~ "Neuropsychiatric",
    TRUE ~ "Uncategorized"
  )
}

# "point (lower, upper)" in a single cell, NA-safe.
pif2_fmt_ci <- function(point, lower, upper, digits = 3) {
  fmt <- sprintf("%%.%df (%%.%df, %%.%df)", digits, digits, digits)
  out <- sprintf(fmt, point, lower, upper)
  out[is.na(point)] <- NA_character_
  out
}

# Static HTML table: bold caption, optional note, scrollable body.
# knitr.kable.NA is forced to "" so that an NA cell renders as a BLANK. The default
# is NULL, which makes kable() print the literal string "NA" -- which would read as
# a value rather than as "this scenario does not apply to this cause".
pif2_html_table <- function(tbl,
                            caption,
                            note = NULL,
                            max_height = 500,
                            digits = 3,
                            round_numeric = TRUE) {
  tbl <- tibble::as_tibble(tbl)
  if (isTRUE(round_numeric)) {
    tbl <- dplyr::mutate(tbl, dplyr::across(dplyr::where(is.numeric), ~ round(.x, digits)))
  }
  old_na <- options(knitr.kable.NA = "")
  on.exit(options(old_na), add = TRUE)
  body_html <- knitr::kable(
    tbl,
    format = "html",
    table.attr = paste0(
      "style='border-collapse: collapse; font-family: Arial, sans-serif; ",
      "font-size: 0.85em; border: 1px solid #ccc;'"
    )
  )
  caption_html <- htmltools::tags$p(
    style = "font-family: Arial, sans-serif; font-size: 0.95em; font-weight: bold; margin-bottom: 0.3em;",
    caption,
    if (!is.null(note)) htmltools::tags$br(),
    if (!is.null(note)) {
      htmltools::tags$span(style = "font-weight: normal; font-size: 0.9em;", note)
    }
  )
  htmltools::browsable(
    htmltools::tags$div(
      caption_html,
      htmltools::tags$div(
        style = sprintf(
          "max-height: %spx; overflow: auto; border: 0px solid #ddd;",
          format(max_height, scientific = FALSE)
        ),
        htmltools::HTML(body_html)
      )
    )
  )
}

# Several tables in ONE cell: bundle them into a single browsable tag list, so the
# cell has exactly one visible value and every table is guaranteed to render.
pif2_html_tables <- function(...) {
  htmltools::browsable(htmltools::tagList(...))
}

pif2_message("[tables] Helpers ready. Reporting year: %s | age group 4 is labelled %s",
             pif2_report_year, pif2_age_labels[["4"]])

message(sprintf(
  "pif2-table-helpers elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[tables] Helpers ready. Reporting year: 2024 | age group 4 is labelled 60-65
pif2-table-helpers elapsed minutes: 0.00


## PIF scenario grid

A **declarative scenario registry**. `shift_vol`/`shift_hed` are retained fractions (0.9 = 10% reduction), and all reductions are **relative** (recorded in `scale`). `engine_scenario` maps each row to the unified engine: `volume` reduces average consumption, `hed` reduces the HED fraction among current drinkers, and `both` combines them (the additive engine superset). Baseline is a no-change reference (PIF ≡ 0). `requires_hed` marks scenarios that only apply to HED-capable causes; applicability is decided per disease from the RR structure so a volume-only cause never shows a spurious HED effect.


In [7]:
#| label: pif2-scenario-grid
#| results: "hold"

.t0 <- Sys.time()

# Keep expensive simulation off by default. Turn this on only after the loading
# cells above complete and you want to test one small PIF run.
pif2_run_smoke_tests <- TRUE

# ---------------------------------------------------------------------------
# Declarative PIF scenario registry (Phase 5). One row per policy counterfactual.
# All reductions are RELATIVE (a fraction of the current level), recorded in
# `scale`. `shift_vol` / `shift_hed` are the RETAINED fractions (0.9 = -10%).
# engine_scenario maps to the unified engine:
#   "volume" -> reduce average consumption (everyone: x -> x*shift_vol)
#   "hed"    -> reduce HED prevalence (a fraction of binge drinkers become NHED)
#   "both"   -> combined volume + HED (the additive engine superset)
# Baseline is a volume scenario with shift 1 (no change) -> PIF == 0 by construction.
# `requires_hed` marks scenarios that only make sense for HED-capable diseases.
# ---------------------------------------------------------------------------
pif2_scenario_grid <- tibble::tribble(
  ~scenario_id,          ~scenario_label,                     ~scenario_family,     ~engine_scenario, ~volume_reduction_pct, ~hed_reduction_pct, ~shift_vol, ~shift_hed, ~requires_hed,
  "baseline",            "Baseline (no change)",              "Baseline",           "volume",         0,                     0,                  1.00,       1.00,       FALSE,
  "volume_reduction_10", "Average consumption -10%",          "Consumption volume", "volume",         10,                    0,                  0.90,       1.00,       FALSE,
  "volume_reduction_20", "Average consumption -20%",          "Consumption volume", "volume",         20,                    0,                  0.80,       1.00,       FALSE,
  "volume_reduction_30", "Average consumption -30%",          "Consumption volume", "volume",         30,                    0,                  0.70,       1.00,       FALSE,
  "hed_reduction_10",    "HED prevalence -10%",               "HED prevalence",     "hed",            0,                     10,                 1.00,       0.90,       TRUE,
  "hed_reduction_25",    "HED prevalence -25%",               "HED prevalence",     "hed",            0,                     25,                 1.00,       0.75,       TRUE,
  "hed_reduction_50",    "HED prevalence -50%",               "HED prevalence",     "hed",            0,                     50,                 1.00,       0.50,       TRUE,
  "combined_v10_h25",    "Combined: volume -10%, HED -25%",   "Combined",           "both",           10,                    25,                 0.90,       0.75,       TRUE,
  "combined_v20_h50",    "Combined: volume -20%, HED -50%",   "Combined",           "both",           20,                    50,                 0.80,       0.50,       TRUE,
  "combined_v30_h50",    "Combined: volume -30%, HED -50%",   "Combined",           "both",           30,                    50,                 0.70,       0.50,       TRUE
)
pif2_scenario_grid$scale <- "relative"   # reductions are relative fractions, not percentage points

# Applicability of a (disease, scenario) pair, decided from the RR structure:
# a HED/combined scenario is NOT applicable to a volume-only (no-HED) disease, so
# such cells are reported as NA with a machine-readable reason rather than a
# spurious zero effect.
pif2_scenario_applicability <- function(spec_row, scenario_row) {
  if (isTRUE(scenario_row$requires_hed) && !isTRUE(spec_row$uses_hed)) {
    return(list(applicable = FALSE,
                reason = "no HED RR component (volume-only cause); HED/combined scenario not applicable"))
  }
  list(applicable = TRUE, reason = NA_character_)
}

pif2_message("[scenarios] Defined %s scenarios (%s volume, %s HED, %s combined, 1 baseline).",
             nrow(pif2_scenario_grid),
             sum(pif2_scenario_grid$scenario_family == "Consumption volume"),
             sum(pif2_scenario_grid$scenario_family == "HED prevalence"),
             sum(pif2_scenario_grid$scenario_family == "Combined"))
pif2_message("[scenarios] Reduction scale: %s (retained fraction shift_vol/shift_hed).",
             unique(pif2_scenario_grid$scale))

message(sprintf(
  "pif2-scenario-grid elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


[scenarios] Defined 10 scenarios (3 volume, 3 HED, 3 combined, 1 baseline).
[scenarios] Reduction scale: relative (retained fraction shift_vol/shift_hed).
pif2-scenario-grid elapsed minutes: 0.00


In [8]:
#| label: pif2-tbl-spec-and-scenarios
#| results: "hold"

.t0 <- Sys.time()

# Two reference tables: WHAT the PIF covers (the output spec, derived from the AAF
# audit) and WHICH counterfactuals it runs (the scenario registry).
pif2_spec_display <- pif2_output_spec |>
  dplyr::mutate(
    sex = unname(pif2_sex_labels[sex]),
    mode = dplyr::recode(mode,
                         "nohed"    = "nohed (volume only)",
                         "cap"      = "cap (J-curve, age-banded)",
                         "explicit" = "explicit (two-component HED)"),
    registry_scope = tidyr::replace_na(registry_scope, "record index (cancer/hhd/general)")
  ) |>
  dplyr::select(
    `Output table`     = output_name,
    Disease            = disease,
    Sex                = sex,
    `RR source object` = source_object,
    `Engine mode`      = mode,
    `Uses HED`         = uses_hed,
    `Age-banded`       = age_banded,
    `FD uncertainty`   = fd_uncertainty,
    `RR registry`      = registry_scope
  ) |>
  dplyr::arrange(Disease, Sex)

pif2_scenario_display <- pif2_scenario_grid |>
  dplyr::select(
    Scenario               = scenario_id,
    Label                  = scenario_label,
    Family                 = scenario_family,
    `Engine path`          = engine_scenario,
    `Volume cut (%)`       = volume_reduction_pct,
    `HED cut (%)`          = hed_reduction_pct,
    `shift_vol (retained)` = shift_vol,
    `shift_hed (retained)` = shift_hed,
    `Needs HED cause`      = requires_hed,
    Scale                  = scale
  )

message(sprintf(
  "pif2-tbl-spec-and-scenarios elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

pif2_html_tables(
  pif2_html_table(
    pif2_spec_display,
    caption = sprintf("PIF output spec: %s output tables, %s distinct diseases",
                      nrow(pif2_output_spec), dplyr::n_distinct(pif2_output_spec$disease)),
    note = paste(
      "Derived from the AAF audit (aaf_adam_rr_audit), so the PIF covers exactly the same output",
      "tables, engine modes and former-drinker settings as the validated PAF. A volume-only cause",
      "(Uses HED = FALSE) can never receive an HED or combined scenario."
    ),
    max_height = 420
  ),
  pif2_html_table(
    pif2_scenario_display,
    caption = sprintf("PIF scenario registry (%s counterfactuals)", nrow(pif2_scenario_grid)),
    note = paste(
      "All reductions are relative; shift_vol / shift_hed are the RETAINED fractions (0.9 = -10%).",
      "Baseline is a no-change reference (PIF = 0 by construction). Scenarios flagged",
      "'Needs HED cause' are reported as NA with a reason on volume-only causes, never as a zero."
    ),
    max_height = 380,
    digits = 2
  )
)

pif2-tbl-spec-and-scenarios elapsed minutes: 0.00


Output table,Disease,Sex,RR source object,Engine mode,Uses HED,Age-banded,FD uncertainty,RR registry
panc_male,Acute Pancreatitis,Men,pancreatitismale,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
panc_fem,Acute Pancreatitis,Women,pancreatitisfemale,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
bcan_female,Breast Cancer,Women,Breastcancer_female,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
crcan_male,Colon and rectum Cancer,Men,colorectalcancer_male,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
crcan_female,Colon and rectum Cancer,Women,colorectalcancer_female,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
dm_male,DM2,Men,diabetesmale,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
dm_fem,DM2,Women,diabetesfemale,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
epi_male,Epilepsy,Men,epilepsymale,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
epi_female,Epilepsy,Women,epilepsyfemale,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)
hiv_male,HIV,Men,HIVmale,nohed (volume only),FALSE,FALSE,TRUE,record index (cancer/hhd/general)


## Per-cell PIF resolution

These helpers resolve one PIF cell's inputs **exactly as the validated AAF families do** — same gamma/prevalence pickers, same design closures — and build the `pif_confint()` argument list. Three modes are covered: `nohed` (cancer / HHD / general — volume scenarios only), `cap` (IHD / ischaemic stroke — J-curve HED, age-banded), and `explicit` (injuries — two-component HED with shared β₁). Multi-β RR uncertainty is drawn jointly from the covariance matrix inside the engine; former-drinker variance enters through `var_ln_rr_fd`.


In [9]:
#| label: pif2-pif-engine-wrapper
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# Resolve one PIF cell's inputs EXACTLY as the AAF families do (same picker
# helpers, same gamma/prevalence objects), so the numerator and denominator of
# each PIF use the same population and age support as the validated PAF.
#   * nohed families (cancer/hhd/general): NHED gamma g_{sex}_list, no p_hed.
#   * cap/explicit families (ihd/is/injuries): HED-split gamma (nhed/hed) and p_hed.
# Returns list(ok=TRUE, gamma, gamma_hed, p_abs, p_form, p_hed) or ok=FALSE + reason.
# ---------------------------------------------------------------------------
pif2_resolve_cell_inputs <- function(spec_row, record, year, group, exposure) {
  sex <- spec_row$sex
  g_list      <- if (sex == "female") exposure$g_fem_list      else exposure$g_male_list
  g_hed_list  <- if (sex == "female") exposure$g_fem_hed_list  else exposure$g_male_hed_list
  p_abs_list  <- if (sex == "female") exposure$p_abs_list_fem  else exposure$p_abs_list_male
  p_form_list <- if (sex == "female") exposure$p_form_list_fem else exposure$p_form_list_male
  p_hed_list  <- if (sex == "female") exposure$p_hed_list_fem  else exposure$p_hed_list_male

  p_abs  <- .aaf_pick_prop(p_abs_list, year, group)
  p_form <- .aaf_pick_prop(p_form_list, year, group)

  if (identical(spec_row$mode, "nohed")) {
    gamma     <- g_list[[as.character(year)]][[group]]
    gamma_hed <- NULL; p_hed <- NULL
  } else {
    hed_entry  <- g_hed_list[[as.character(year)]][[group]]
    gamma      <- if (!is.null(hed_entry)) hed_entry$nhed else NULL
    gamma_hed  <- if (!is.null(hed_entry)) hed_entry$hed  else NULL
    year_index <- which(pif2_years == year)
    p_hed <- .aaf_prop_from_group_list(p_hed_list, year_index, group, year)
  }

  bad <- c(
    if (is.null(gamma)) "gamma_null",
    if (spec_row$uses_hed && is.null(gamma_hed)) "gamma_hed_null",
    if (is.na(p_abs)) "p_abs_NA",
    if (is.na(p_form)) "p_form_NA",
    if (spec_row$uses_hed && (is.null(p_hed) || is.na(p_hed))) "p_hed_NA"
  )
  if (length(bad)) return(list(ok = FALSE, reason = paste(bad, collapse = ",")))
  list(ok = TRUE, gamma = gamma, gamma_hed = gamma_hed,
       p_abs = p_abs, p_form = p_form, p_hed = p_hed)
}

# ---------------------------------------------------------------------------
# Build the pif_confint() argument list for one applicable cell + scenario. The
# design knobs (Kish n and cluster factor per survey question, plus the separate
# consumption-question design) are resolved from the SAME closures the PAF used
# (pif2_aaf_uncertainty), so the survey design is applied exactly once. Multi-beta
# RR uncertainty is carried by cov_beta (drawn jointly via mvrnorm inside the
# engine); former-drinker variance is carried by var_ln_rr_fd.
# ---------------------------------------------------------------------------
pif2_build_pif_args <- function(spec_row, record, inputs, scenario_row,
                                exposure, unc, mc, year, group, run_cfg) {
  sex <- spec_row$sex
  args <- list(
    gamma        = inputs$gamma,
    rr_fun       = record$RRCurrent,
    beta         = record$betaCurrent,
    p_abs        = inputs$p_abs,
    p_form       = inputs$p_form,
    ln_rr_fd     = record$lnRRFormer,
    var_ln_rr_fd = if (is.null(record$varLnRRFormer)) 0 else record$varLnRRFormer,
    fd_uncertainty = isTRUE(spec_row$fd_uncertainty),
    x            = exposure$x_vals,
    scenario     = scenario_row$engine_scenario,
    # Engine convention: `shift` is the retained fraction of the lever the scenario
    # acts on -> for "hed" it is the HED retained fraction (shift_hed); for
    # "volume"/"both" it is the volume retained fraction. `shift_hed` is read by
    # the engine only when scenario == "both".
    shift        = if (identical(scenario_row$engine_scenario, "hed")) scenario_row$shift_hed else scenario_row$shift_vol,
    shift_hed    = scenario_row$shift_hed,
    prev_method  = unc$prev_method,
    neff_prev                 = .aaf_resolve_cell(unc$neff, year, group, sex),
    design_factor             = .aaf_resolve_cell(unc$design_factor, year, group, sex),
    neff_consumption          = .aaf_resolve_cell(unc$neff_consumption, year, group, sex),
    design_factor_consumption = .aaf_resolve_cell(unc$design_factor_consumption, year, group, sex),
    n_sim = run_cfg$n_sim, n_pca = run_cfg$n_pca, seed = mc$seed,
    n_cores = run_cfg$n_cores, use_parallel = run_cfg$inner_parallel
  )
  if (spec_row$mode %in% c("nohed", "cap")) {
    args$cov_beta <- record$covBetaCurrent      # multi-beta joint draw (J-curve betas for CV)
    args$hed_mode <- "cap"
    if (spec_row$mode == "cap") { args$gamma_hed <- inputs$gamma_hed; args$p_hed <- inputs$p_hed }
  } else {  # explicit injuries: NHED beta1 shared, HED has its own beta2 + covariance
    args$cov_beta     <- NULL
    args$gamma_hed    <- inputs$gamma_hed
    args$p_hed        <- inputs$p_hed
    args$hed_mode     <- "explicit"
    args$rr_fun_hed   <- record$RRCurrent_binge
    args$beta_hed     <- record$betaCurrent_binge
    args$cov_beta_hed <- record$covBetaCurrent_binge
    args$share_beta1  <- TRUE
  }
  args
}

# Per-cell uncertainty-source flags (for the disease x scenario audit table).
pif2_uncertainty_flags <- function(spec_row, record) {
  ci_method <- .adam_ci_method(record)   # fixed | scalar | vcov (from the covariance rank)
  list(
    uses_volume_rr   = TRUE,                                   # every record exposes RRCurrent(x, beta)
    uses_hed_rr      = isTRUE(spec_row$uses_hed),
    uses_fd_rr       = is.finite(record$lnRRFormer) && abs(record$lnRRFormer) > 0,
    uses_beta_vcov   = identical(ci_method, "vcov") ||
                       (spec_row$mode == "explicit" && !is.null(record$covBetaCurrent_binge)),
    uses_weighted_gamma  = TRUE,                               # bundle 20260709: fit_gamma_weighted (see gamma audit)
    uses_design_variance = is.function(pif2_aaf_uncertainty$neff),
    ci_method = ci_method
  )
}

pif2_message("[pif-wrapper] Cell resolvers ready: pif2_resolve_cell_inputs / pif2_build_pif_args / pif2_uncertainty_flags.")

message(sprintf(
  "pif2-pif-engine-wrapper elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


[pif-wrapper] Cell resolvers ready: pif2_resolve_cell_inputs / pif2_build_pif_args / pif2_uncertainty_flags.
pif2-pif-engine-wrapper elapsed minutes: 0.00


## Ex-HED exit rule: two scenarios (lambda = 0 vs lambda = 1)

Until now, in the `hed` and `both` scenarios, the mass leaving binge drinking **kept its own drinking density** `d_hed` and only switched risk curves (adopting `RR_NHED`). In other words: quitting binge drinking removed the binge-specific excess risk **but did not touch a single gram of alcohol**. Hence, the HED scenarios were, by construction, volume-neutral.

The engine (`aaf_unified.R`, 2026-07-14) now parameterizes that assumption. The counterfactual risk of the exiting mass is

$$R_{\text{exit}} = (1-\lambda)\, R\!\left(d_{hed},\, RR_{NHED}(x\rho)\right) + \lambda\, R_{nhed}$$

Additionally, **two rules** were reported, not a parameter space:

| Rule | lambda | What it assumes | Rows |
|---|---|---|---|
| **Conservative** (ours) | **0** | the ex-HED quits the binge but **keeps drinking the same volume**; they only lose the binge-specific excess RR | `hed_reduction_*`, `combined_*` |
| **Ruiz-Tagle** (JRT) | **1** | the ex-HED **becomes an average NHED drinker**: they keep drinking, with the residual risk of that volume | `*_rt` |

`lambda = 0, rho = 1` reproduces **bit for bit** everything already computed. The `rho` knob (volume retained by the ex-HED) stays at 1 and is not used: it is the calibration socket.

The knobs are **cell-level** scalars (year x sex x age band x cause). Today the engine receives a single number that is broadcast to all cells. Once the **microsimulation** delivers transitions by year/age/sex, that number can be replaced by a `function(year, group, sex)` or a per-year/band list, and **nothing else needs to change**: `resolve_hed_exit()` already handles both forms. For example, if evidence shows that men aged 30–44 reduce their volume by 10% when quitting binge drinking (rho = 0.9), while men aged 45–59 reduce it by 50% (rho = 0.5), we can specify this explicitly per cell.

> **Warning for IHD and ischemic stroke (`hed_mode = "cap"`).** The project's actual J-curve is **protective across the whole band below 60 g/day** (nadir RR = 0.78 at ~31 g/day). There, lambda=1 can **lower** the PIF instead of raising it, and a rho sweep is not monotonic. For the remaining causes (increasing RR), lambda=0 is the conservative bound and lambda=1 the optimistic one. See `test_hed_exit_knobs.R`, section (H).

**Accounting:** with `lambda > 0` the HED scenarios **stop being volume-neutral**. The "implied volume" cell below computes, per cell, the consumption drop the counterfactual actually implies — which is what must be reported, and not the `volume_reduction_pct` of the grid (which is the *policy lever*, not the outcome).

In [10]:
#| label: pif2-hed-exit-wiring
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# HED-EXIT REASSIGNMENT: two exit rules, wired end to end.
#
#   hed_exit_mix   = lambda in [0,1]  share of the mass leaving binge that migrates to
#                                     the NHED consumption DISTRIBUTION.
#                                     0 = conservative (ours) | 1 = Ruiz-Tagle
#   hed_exit_shift = rho in (0,1]     volume that same mass RETAINS. Held at 1 (unused);
#                                     it is the calibration hook, not a lever we report.
#
# Legacy rows keep lambda = 0, rho = 1, so every number already computed is unchanged.
# ---------------------------------------------------------------------------

stopifnot(exists("pif2_scenario_grid"), exists("pif2_build_pif_args"),
          exists("resolve_hed_exit"))   # resolve_hed_exit() lives in aaf_unified.R

# ---- 1. Existing rows -> the conservative rule, declared EXPLICITLY -----------
# Until today lambda = 0 was hard-coded inside the engine and invisible in the grid.
# It is a MODELLING ASSUMPTION and it belongs in the registry, next to the shifts.
if (!"hed_exit_mix" %in% names(pif2_scenario_grid)) {
  pif2_scenario_grid$hed_exit_mix   <- 0    # ex-HED keeps his own consumption density
  pif2_scenario_grid$hed_exit_shift <- 1    # ...and his full volume
  pif2_scenario_grid$exit_rule      <- "conservative"
}

# ---- 2. The six Ruiz-Tagle twins (lambda = 1) --------------------------------
# One twin per row that actually has mass leaving binge: the 3 pure-HED rows and the 3
# combined rows. "volume" and "baseline" are NOT duplicated: with nobody exiting binge
# lambda is a mathematical no-op, so a twin would be a bit-identical duplicate that
# costs a full run for nothing. Same policy levers as the row each one mirrors --
# the ONLY thing that changes is where the ex-binge drinker lands.
pif2_hed_exit_twins <- tibble::tribble(
  ~scenario_id,          ~scenario_label,                                   ~scenario_family,   ~engine_scenario, ~volume_reduction_pct, ~hed_reduction_pct, ~shift_vol, ~shift_hed, ~requires_hed, ~hed_exit_mix, ~hed_exit_shift, ~exit_rule,
  "hed_reduction_10_rt", "HED -10% (ex-HED -> NHED, JRT)",                  "HED prevalence",   "hed",            0,                     10,                 1.00,       0.90,       TRUE,          1,             1,               "ruiz_tagle",
  "hed_reduction_25_rt", "HED -25% (ex-HED -> NHED, JRT)",                  "HED prevalence",   "hed",            0,                     25,                 1.00,       0.75,       TRUE,          1,             1,               "ruiz_tagle",
  "hed_reduction_50_rt", "HED -50% (ex-HED -> NHED, JRT)",                  "HED prevalence",   "hed",            0,                     50,                 1.00,       0.50,       TRUE,          1,             1,               "ruiz_tagle",
  "combined_v10_h25_rt", "Combined v-10% / HED -25% (ex-HED -> NHED, JRT)", "Combined",         "both",           10,                    25,                 0.90,       0.75,       TRUE,          1,             1,               "ruiz_tagle",
  "combined_v20_h50_rt", "Combined v-20% / HED -50% (ex-HED -> NHED, JRT)", "Combined",         "both",           20,                    50,                 0.80,       0.50,       TRUE,          1,             1,               "ruiz_tagle",
  "combined_v30_h50_rt", "Combined v-30% / HED -50% (ex-HED -> NHED, JRT)", "Combined",         "both",           30,                    50,                 0.70,       0.50,       TRUE,          1,             1,               "ruiz_tagle"
)
pif2_hed_exit_twins$scale <- "relative"

pif2_scenario_grid <- dplyr::bind_rows(
  pif2_scenario_grid,
  pif2_hed_exit_twins[!pif2_hed_exit_twins$scenario_id %in% pif2_scenario_grid$scenario_id, ]
)

# Every twin must mirror its original EXACTLY except for the exit rule. If a shift ever
# drifts apart, the "lambda moved the PIF by X%" reading becomes a lie. Assert it.
for (.tw in pif2_hed_exit_twins$scenario_id) {
  .orig <- sub("_rt$", "", .tw)
  .a <- pif2_scenario_grid[pif2_scenario_grid$scenario_id == .orig, ]
  .b <- pif2_scenario_grid[pif2_scenario_grid$scenario_id == .tw, ]
  stopifnot(nrow(.a) == 1L, nrow(.b) == 1L,
            identical(.a$engine_scenario, .b$engine_scenario),
            isTRUE(all.equal(.a$shift_vol, .b$shift_vol)),
            isTRUE(all.equal(.a$shift_hed, .b$shift_hed)),
            .a$hed_exit_mix == 0, .b$hed_exit_mix == 1)
}

# ---- 3. THE SAFE DISCARD LIST (whitelist, not blacklist) ---------------------
# Four places in this notebook turn a pif_confint() arg list into an aaf_confint() one by
# BLACKLISTING the PIF-only names:
#     args[setdiff(names(args), c("scenario", "shift", "shift_hed"))]
# That is fragile by construction: aaf_confint() has no `...`, so the day the engine
# grows ANY new PIF-only argument (today: hed_exit_mix / hed_exit_shift), every one of
# those four sites breaks with "unused argument" -- and the blacklist has to be hunted
# down and edited in four places, forever.
#
# Invert it. WHITELIST against what aaf_confint() actually accepts, read from the
# function itself. A new PIF-only argument is then dropped automatically and no site
# ever needs editing again. The four call sites are patched to use this.
pif2_as_aaf_args <- function(args) {
  keep <- intersect(names(args), names(formals(aaf_confint)))
  dropped <- setdiff(names(args), keep)
  if (length(dropped)) {
    pif2_message("[aaf-args] Dropped %d PIF-only argument(s) before calling aaf_confint(): %s.",
                 length(dropped), paste(dropped, collapse = ", "))
  }
  args[keep]
}

# Self-check: it must drop exactly the PIF-only names and keep everything else.
local({
  .probe <- list(gamma = 1, rr_fun = 1, beta = 1, p_abs = 1, p_form = 1, x = 1,
                 scenario = "hed", shift = 0.9, shift_hed = 0.9,
                 hed_exit_mix = 1, hed_exit_shift = 1)
  .kept <- suppressMessages(names(pif2_as_aaf_args(.probe)))
  stopifnot(!any(c("scenario", "shift", "shift_hed", "hed_exit_mix", "hed_exit_shift") %in% .kept),
            all(c("gamma", "rr_fun", "beta", "p_abs", "p_form", "x") %in% .kept))
})

# ---- 4. Wire the knobs into the engine call ----------------------------------
# Wrap (do NOT edit) pif2_build_pif_args: call the original, then append the knobs.
# Guarded against double-wrapping if this cell is re-run.
if (!exists(".pif2_build_pif_args_base")) {
  .pif2_build_pif_args_base <- pif2_build_pif_args
}
pif2_build_pif_args <- function(spec_row, record, inputs, scenario_row,
                                exposure, unc, mc, year, group, run_cfg) {
  args <- .pif2_build_pif_args_base(spec_row, record, inputs, scenario_row,
                                    exposure, unc, mc, year, group, run_cfg)

  # Only "hed"/"both" have a mass leaving binge. Under "volume" the knobs are a
  # mathematical no-op, so we do not even attach them.
  if (!scenario_row$engine_scenario %in% c("hed", "both")) return(args)

  # A hand-built scenario row (as in the Phase 7 isolation tests) may not carry the new
  # columns -> fall back to the legacy constants rather than erroring.
  lam_spec <- if (is.null(scenario_row$hed_exit_mix))   0 else scenario_row$hed_exit_mix
  rho_spec <- if (is.null(scenario_row$hed_exit_shift)) 1 else scenario_row$hed_exit_shift

  # resolve_hed_exit() turns a SPEC into this cell's scalar. Today the spec is a bare
  # number and this is just a broadcast. The day the microsimulation delivers lambda by
  # year/age/sex, put a function(year, group, sex) -- or a list keyed by year -> age band
  # -- in the grid cell instead, and nothing else in the pipeline changes. It ERRORS on
  # any cell a spec fails to cover: it never silently falls back.
  lam <- resolve_hed_exit(lam_spec, year, group, spec_row$sex)
  rho <- resolve_hed_exit(rho_spec, year, group, spec_row$sex)

  # Legacy values -> attach nothing: the engine reuses the pre-2026-07-14 code path and
  # the result is bit-for-bit identical to what is already published.
  if (isTRUE(lam == 0) && isTRUE(rho == 1)) return(args)

  args$hed_exit_mix   <- lam
  args$hed_exit_shift <- rho
  args
}

pif2_message("[hed-exit] Grid: %d scenarios (%d conservative, %d Ruiz-Tagle). Safe AAF arg whitelist installed.",
             nrow(pif2_scenario_grid),
             sum(pif2_scenario_grid$exit_rule == "conservative"),
             sum(pif2_scenario_grid$exit_rule == "ruiz_tagle"))

message(sprintf("pif2-hed-exit-wiring elapsed minutes: %.2f", pif2_elapsed_min(.t0)))

[hed-exit] Grid: 16 scenarios (10 conservative, 6 Ruiz-Tagle). Safe AAF arg whitelist installed.
pif2-hed-exit-wiring elapsed minutes: 0.00


In [11]:
#| label: pif2-hed-exit-implied-volume
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# WHY volume_reduction_pct = 0 IS A LIE ON THE JRT ROWS, AND WHY THE FIX IS NOT A
# CONSTANT IN THE GRID.
#
# With lambda = 1 the ex-binge drinker moves onto the NHED consumption distribution, so
# he really does drink LESS. The HED scenarios stop being volume-neutral. But the size
# of that drop is NOT a property of the scenario: it depends on p_hed, E[d_hed] and
# E[d_nhed], which change with year, sex and age band. It is a PER-CELL derived
# quantity, not a grid constant -- so it cannot be written into the tribble.
#
# The fix therefore has two parts:
#   (a) volume_reduction_pct KEEPS its meaning and is read honestly: it is the POLICY
#       volume lever (what the intervention does to everyone's consumption by decree),
#       not the total change in grams. On a pure HED row that lever is 0, and that is
#       TRUE -- the policy does not cut anyone's volume by decree.
#   (b) a NEW per-cell column reports the change in mean consumption the counterfactual
#       actually IMPLIES, lever plus behavioural reallocation. THAT is the number that
#       must never be quoted as 0.
#
# Mean consumption among CURRENT DRINKERS:
#   baseline   E0   = (1-p_hed)*E_nhed + p_hed*E_hed
#   counterfac E_cf = s_v * [ (1-p_hed)*E_nhed
#                             + s_h*p_hed*E_hed
#                             + (1-s_h)*p_hed*( (1-lambda)*rho*E_hed + lambda*E_nhed ) ]
# with s_v = shift_vol (1 on pure-HED rows) and s_h = shift_hed (1 on volume rows).
# Reported as the percent change (E_cf - E0)/E0, i.e. NEGATIVE = a real drop in grams.
#
# LOOP BOUNDS: see the pif2_run_cfg / fallback block below -- this cell runs BEFORE the
# pif2_run_pif_grid() is handed -- so this table covers exactly the cells the engine
# runs. (Do NOT use pif2_years here: it exists, but it is not what the grid runner loops
# over, and a silent mismatch would make this summary describe a different population
# than the PIFs it is meant to annotate.)
# ---------------------------------------------------------------------------

stopifnot(exists("pif2_scenario_grid"), exists("pif2_resolve_cell_inputs"),
          exists("pif2_output_spec"), exists("pif2_years"))

# LOOP BOUNDS -- and an ordering subtlety worth stating.
# The grid runner iterates pif2_run_cfg$years / pif2_run_cfg$groups. But pif2_run_cfg is
# only defined DOWNSTREAM, in the run-grid cell -- this cell runs BEFORE it, on purpose:
# you want to see the implied consumption change BEFORE launching a ~314-minute run.
# So: use pif2_run_cfg when it exists (a re-run, or an out-of-order execution), and
# otherwise fall back to the FULL wave set, which is exactly what pif2_run_cfg$years
# resolves to in "full" mode (pif2_pif_run_config$full$years == pif2_years, groups == 1:4).
# In "demo" mode the grid only runs the last wave, so this table would then cover MORE
# cells than the run -- which is fine (it is a property of the scenarios, not of the run)
# but it is said out loud rather than left to be discovered.
if (exists("pif2_run_cfg")) {
  .iv_years  <- pif2_run_cfg$years
  .iv_groups <- pif2_run_cfg$groups
  pif2_message("[implied-vol] Using pif2_run_cfg: %d year(s), %d age band(s).", length(.iv_years), length(.iv_groups))
} else {
  .iv_years  <- pif2_years
  .iv_groups <- 1:4
  pif2_message("[implied-vol] pif2_run_cfg not defined yet (it lives in the run-grid cell downstream). Falling back to the FULL wave set: %d years x 4 bands.", length(.iv_years))
}

# Mean of a fitted gamma = shape/rate. .aaf_gamma_pars() normalises the two conventions.
pif2_gamma_mean <- function(g) {
  p <- .aaf_gamma_pars(g)
  if (is.finite(p$rate)) p$shape / p$rate else p$shape * p$scale
}

# Implied change in MEAN CONSUMPTION AMONG CURRENT DRINKERS, for one cell x scenario.
# Returns percent change (negative = drop).
pif2_implied_vol_change_pct <- function(inputs, scenario_row) {
  s_v <- if (is.null(scenario_row$shift_vol))      1 else scenario_row$shift_vol
  s_h <- if (is.null(scenario_row$shift_hed))      1 else scenario_row$shift_hed
  lam <- if (is.null(scenario_row$hed_exit_mix))   0 else scenario_row$hed_exit_mix
  rho <- if (is.null(scenario_row$hed_exit_shift)) 1 else scenario_row$hed_exit_shift

  E_nhed <- pif2_gamma_mean(inputs$gamma)
  has_hed <- !is.null(inputs$gamma_hed) && !is.null(inputs$p_hed) &&
             is.finite(inputs$p_hed) && inputs$p_hed > 0
  if (!has_hed) return(100 * (s_v - 1))   # volume-only cause: only the lever moves grams

  E_hed <- pif2_gamma_mean(inputs$gamma_hed)
  p     <- inputs$p_hed
  E0    <- (1 - p) * E_nhed + p * E_hed
  E_cf  <- s_v * ((1 - p) * E_nhed +
                  s_h * p * E_hed +
                  (1 - s_h) * p * ((1 - lam) * rho * E_hed + lam * E_nhed))
  if (!is.finite(E0) || E0 <= 0) return(NA_real_)
  100 * (E_cf / E0 - 1)
}

# ---- Compute it for EVERY (cause x scenario x year x band) --------------------
# Cheap: no Monte Carlo, no integrals -- just gamma means and the prevalences already
# resolved by pif2_resolve_cell_inputs().
pif2_implied_vol <- do.call(rbind, lapply(seq_len(nrow(pif2_output_spec)), function(i) {
  spec_row <- pif2_output_spec[i, ]
  do.call(rbind, lapply(.iv_years, function(year) {
    do.call(rbind, lapply(.iv_groups, function(group) {
      rec <- tryCatch(pif2_lookup_record(spec_row, group), error = function(e) NULL)
      if (is.null(rec)) return(NULL)
      inp <- pif2_resolve_cell_inputs(spec_row, rec, year, group, pif2_exposure_inputs)
      if (!isTRUE(inp$ok)) return(NULL)
      do.call(rbind, lapply(seq_len(nrow(pif2_scenario_grid)), function(j) {
        scn <- pif2_scenario_grid[j, ]
        if (!isTRUE(pif2_scenario_applicability(spec_row, scn)$applicable)) return(NULL)
        data.frame(output_name = spec_row$output_name, sex = spec_row$sex,
                   year = year, age_group = group,
                   scenario_id = scn$scenario_id,
                   exit_rule = if (is.null(scn$exit_rule)) "conservative" else scn$exit_rule,
                   policy_vol_lever_pct = -scn$volume_reduction_pct,
                   implied_vol_change_pct = pif2_implied_vol_change_pct(inp, scn),
                   stringsAsFactors = FALSE)
      }))
    }))
  }))
}))

# ---- The headline: what the grid CLAIMS vs what the counterfactual IMPLIES -----
pif2_implied_vol_summary <- pif2_implied_vol |>
  dplyr::group_by(scenario_id, exit_rule) |>
  dplyr::summarise(policy_lever_pct = unique(policy_vol_lever_pct),
                   implied_mean_pct = mean(implied_vol_change_pct, na.rm = TRUE),
                   implied_min_pct  = min(implied_vol_change_pct,  na.rm = TRUE),
                   implied_max_pct  = max(implied_vol_change_pct,  na.rm = TRUE),
                   .groups = "drop") |>
  dplyr::arrange(scenario_id)

print(as.data.frame(pif2_implied_vol_summary))

.liars <- pif2_implied_vol_summary[
  pif2_implied_vol_summary$policy_lever_pct == 0 &
  abs(pif2_implied_vol_summary$implied_mean_pct) > 0.01, ]
if (nrow(.liars)) {
  pif2_message(paste0("[implied-vol] %d scenario(s) declare a POLICY volume lever of 0%% but IMPLY a real ",
                      "change in mean consumption: %s. Report `implied_vol_change_pct`, NOT ",
                      "`volume_reduction_pct`, whenever you describe what the policy does to grams."),
               nrow(.liars),
               paste(sprintf("%s (%.2f%%)", .liars$scenario_id, .liars$implied_mean_pct), collapse = "; "))
}

# Sanity: the conservative rows (lambda = 0, rho = 1) must imply EXACTLY the policy lever
# -- no reallocation, no hidden grams. If this ever fails, the wiring is wrong.
.cons <- pif2_implied_vol_summary[pif2_implied_vol_summary$exit_rule == "conservative", ]
stopifnot(all(abs(.cons$implied_mean_pct - .cons$policy_lever_pct) < 1e-8))
pif2_message("[implied-vol] Check passed: every conservative row implies exactly its policy lever (lambda=0 moves no grams).")

message(sprintf("pif2-hed-exit-implied-volume elapsed minutes: %.2f", pif2_elapsed_min(.t0)))

[implied-vol] pif2_run_cfg not defined yet (it lives in the run-grid cell downstream). Falling back to the FULL wave set: 7 years x 4 bands.
           scenario_id    exit_rule policy_lever_pct implied_mean_pct
1             baseline conservative                0     0.000000e+00
2     combined_v10_h25 conservative              -10    -1.000000e+01
3  combined_v10_h25_rt   ruiz_tagle              -10    -2.192257e+01
4     combined_v20_h50 conservative              -20    -2.000000e+01
5  combined_v20_h50_rt   ruiz_tagle              -20    -4.119567e+01
6     combined_v30_h50 conservative              -30    -3.000000e+01
7  combined_v30_h50_rt   ruiz_tagle              -30    -4.854621e+01
8     hed_reduction_10 conservative                0    -9.912706e-16
9  hed_reduction_10_rt   ruiz_tagle                0    -5.298918e+00
10    hed_reduction_25 conservative                0    -1.387779e-15
11 hed_reduction_25_rt   ruiz_tagle                0    -1.324730e+01
12    hed_reduction

In [12]:
#| label: pif2-hed-exit-smoke
#| results: "hold"

.t0 <- Sys.time()

# One real cell, run twice through the SAME arg builder, changing ONLY the exit rule.
#
# THE CELL HAS TO BE HED-CAPABLE. Most causes in pif2_output_spec are mode = "nohed"
# (uses_hed = FALSE) -- a "hed" scenario on them errors with "requiere p_hed>0", because
# there is no binge component to reduce. The HED-capable ones are the cap family
# (ihd_*, is_*) and the explicit family (ri_*, injuries_*, violence_*).
#
# We use ri_male (Road Injuries, male, 2022, band 30-44): explicit mode, HED-capable, and
# a MONOTONICALLY INCREASING RR -- which is what makes the direction check below
# meaningful. IHD/IS are HED-capable too, but their cardioprotective J-curve makes the
# direction of lambda genuinely ambiguous (lambda=1 can LOWER the PIF there), so they are
# the wrong cell to assert a direction on. See test_hed_exit_knobs.R, section (H).
.he_spec <- pif2_output_spec[pif2_output_spec$output_name == "ri_male", ]
.he_rec  <- pif2_lookup_record(.he_spec, 2L)
.he_in   <- pif2_resolve_cell_inputs(.he_spec, .he_rec, 2022L, 2L, pif2_exposure_inputs)
stopifnot(isTRUE(.he_in$ok))
.he_cfg  <- list(n_sim = 400L, n_pca = 400L, n_cores = 1L,
                 inner_parallel = FALSE, outer_parallel = FALSE)

.he_run <- function(scenario_id) {
  row  <- pif2_scenario_grid[pif2_scenario_grid$scenario_id == scenario_id, ]
  args <- pif2_build_pif_args(.he_spec, .he_rec, .he_in, row, pif2_exposure_inputs,
                              pif2_aaf_uncertainty, pif2_aaf_mc, 2022L, 2L, .he_cfg)
  res  <- do.call(pif_confint, args)
  data.frame(
    scenario = scenario_id,
    exit_rule = if (is.null(row$exit_rule)) "conservative" else row$exit_rule,
    lambda = if (is.null(args$hed_exit_mix)) 0 else args$hed_exit_mix,
    pif = res$point_estimate, lower = res$lower_ci, upper = res$upper_ci,
    implied_vol_pct = pif2_implied_vol_change_pct(.he_in, row),
    stringsAsFactors = FALSE)
}

.he_cmp <- do.call(rbind, lapply(
  c("hed_reduction_50", "hed_reduction_50_rt",
    "combined_v20_h50", "combined_v20_h50_rt"), .he_run))
print(.he_cmp)

pif2_message("[hed-exit] ri_male (Road Injuries) 2022 / 30-44, HED -50%%: conservative PIF = %.4f -> Ruiz-Tagle PIF = %.4f (%+.1f%%). Implied mean-consumption change: %.2f%% vs %.2f%%.",
             .he_cmp$pif[1], .he_cmp$pif[2],
             100 * (.he_cmp$pif[2] / .he_cmp$pif[1] - 1),
             .he_cmp$implied_vol_pct[1], .he_cmp$implied_vol_pct[2])

# Guard: on a monotone-RR cause, JRT must not come out BELOW the conservative rule. If it
# does, the wiring is inverted. (Deliberately NOT applied to IHD/IS, where a lower PIF
# under lambda=1 is a real property of the cardioprotective J-curve.)
stopifnot(.he_cmp$pif[2] >= .he_cmp$pif[1])
pif2_message("[hed-exit] Direction check passed on a monotone-RR cause (JRT >= conservative).")

message(sprintf("pif2-hed-exit-smoke elapsed minutes: %.2f", pif2_elapsed_min(.t0)))

aaf MC: use_parallel=FALSE; running Monte Carlo sequentially.
             scenario    exit_rule lambda       pif      lower     upper
1    hed_reduction_50 conservative      0 0.1763206 0.09975717 0.2538748
2 hed_reduction_50_rt   ruiz_tagle      1 0.1800360 0.10507582 0.2565546
3    combined_v20_h50 conservative      0 0.1827731 0.10749703 0.2603635
4 combined_v20_h50_rt   ruiz_tagle      1 0.1856811 0.11164522 0.2629333
  implied_vol_pct
1         0.00000
2       -21.60916
3       -20.00000
4       -37.28733
[hed-exit] ri_male (Road Injuries) 2022 / 30-44, HED -50%: conservative PIF = 0.1763 -> Ruiz-Tagle PIF = 0.1800 (+2.1%). Implied mean-consumption change: 0.00% vs -21.61%.
[hed-exit] Direction check passed on a monotone-RR cause (JRT >= conservative).
pif2-hed-exit-smoke elapsed minutes: 0.05


## Run the PIF grid (all diseases × scenarios)

The orchestrator loops every output table × scenario × year × age group, resolves each cell's inputs exactly as the PAF families do, and calls `pif_confint()`. Non-applicable combinations (e.g. a HED scenario on a volume-only cancer) become `NA` rows with a machine-readable reason — never a spurious zero. `mode = "demo"` runs the latest wave at reduced `n_sim` so the notebook renders; set `pif2_pif_run_mode <- "full"` to reproduce the PAF's Monte Carlo depth over every wave. Results and the disease×scenario audit are saved with the dated project convention.


In [13]:
#| label: pif2-run-grid
#| results: "hold"

.t0 <- Sys.time()
# ---------------------------------------------------------------------------
# Orchestrator (Phase 6). Loops output-table x scenario x year x age group.
# Non-applicable and invalid cells become NA rows carrying a machine-readable
# reason (never a spurious zero). Applicable non-baseline cells are collected as
# fully-resolved jobs and run through pif_confint(). Baselines validate their
# record and inputs, then return the deterministic identity PIF = 0 without
# spending Monte Carlo draws. Common random numbers make the remaining
# counterfactual comparisons reproducible across serial and parallel execution.
# Returns: long-format `results`, a disease x scenario `audit`, and the job count.
# ---------------------------------------------------------------------------
pif2_run_pif_grid <- function(spec, scenarios, exposure, unc, mc, years, groups, run_cfg) {
  long_rows  <- list(); jobs <- list(); job_meta <- list(); audit_rows <- list()
  na_row <- function(spec_row, scenario_row, year, group, applicable, reason, flags) {
    data.frame(
      disease = spec_row$disease, sex = spec_row$sex, age_group = group, year = year,
      output_name = spec_row$output_name,
      scenario_id = scenario_row$scenario_id, scenario_label = scenario_row$scenario_label,
      engine_scenario = scenario_row$engine_scenario,
      avg_consumption_change_pct = scenario_row$volume_reduction_pct,
      hed_prevalence_change_pct  = scenario_row$hed_reduction_pct,
      pif = NA_real_, pif_low = NA_real_, pif_up = NA_real_, n_used = NA_integer_,
      applicable = applicable, reason = reason,
      uses_volume_rr = flags$uses_volume_rr, uses_hed_rr = flags$uses_hed_rr,
      uses_fd_rr = flags$uses_fd_rr, uses_beta_vcov = flags$uses_beta_vcov,
      uses_weighted_gamma = flags$uses_weighted_gamma, uses_design_variance = flags$uses_design_variance,
      n_sim = run_cfg$n_sim, stringsAsFactors = FALSE)
  }
  baseline_row <- function(spec_row, scenario_row, year, group, flags) {
    data.frame(
      disease = spec_row$disease, sex = spec_row$sex, age_group = group, year = year,
      output_name = spec_row$output_name,
      scenario_id = scenario_row$scenario_id, scenario_label = scenario_row$scenario_label,
      engine_scenario = scenario_row$engine_scenario,
      avg_consumption_change_pct = scenario_row$volume_reduction_pct,
      hed_prevalence_change_pct = scenario_row$hed_reduction_pct,
      pif = 0, pif_low = 0, pif_up = 0, n_used = 0L,
      applicable = TRUE, reason = NA_character_,
      uses_volume_rr = flags$uses_volume_rr, uses_hed_rr = flags$uses_hed_rr,
      uses_fd_rr = flags$uses_fd_rr, uses_beta_vcov = flags$uses_beta_vcov,
      uses_weighted_gamma = flags$uses_weighted_gamma, uses_design_variance = flags$uses_design_variance,
      n_sim = run_cfg$n_sim, stringsAsFactors = FALSE)
  }
  for (si in seq_len(nrow(spec))) {
    spec_row <- spec[si, ]
    for (sc in seq_len(nrow(scenarios))) {
      scenario_row <- scenarios[sc, ]
      applic <- pif2_scenario_applicability(spec_row, scenario_row)
      for (year in years) for (group in groups) {
        record <- tryCatch(pif2_lookup_record(spec_row, group), error = function(e) NULL)
        flags  <- if (is.null(record))
          list(uses_volume_rr = NA, uses_hed_rr = spec_row$uses_hed, uses_fd_rr = NA,
               uses_beta_vcov = NA, uses_weighted_gamma = TRUE, uses_design_variance = is.function(unc$neff))
        else pif2_uncertainty_flags(spec_row, record)
        # one audit row per (disease x scenario), stamped on the first (year, group)
        if (year == years[[1]] && group == groups[[1]]) {
          audit_rows[[length(audit_rows) + 1L]] <- data.frame(
            disease_id = spec_row$output_name, disease = spec_row$disease, sex = spec_row$sex,
            scenario_id = scenario_row$scenario_id,
            uses_volume_rr = flags$uses_volume_rr, uses_hed_rr = flags$uses_hed_rr,
            uses_fd_rr = flags$uses_fd_rr, uses_beta_vcov = flags$uses_beta_vcov,
            uses_weighted_gamma = flags$uses_weighted_gamma, uses_design_variance = flags$uses_design_variance,
            age_support = "15-64 (edad_tramo 1-4)",
            applicable = applic$applicable, reason = applic$reason,
            warnings = NA_character_, stringsAsFactors = FALSE)
        }
        if (!applic$applicable) {
          long_rows[[length(long_rows) + 1L]] <- na_row(spec_row, scenario_row, year, group, FALSE, applic$reason, flags)
          next
        }
        if (is.null(record)) {
          long_rows[[length(long_rows) + 1L]] <- na_row(spec_row, scenario_row, year, group, TRUE, "record_lookup_failed", flags)
          next
        }
        inputs <- pif2_resolve_cell_inputs(spec_row, record, year, group, exposure)
        if (!isTRUE(inputs$ok)) {
          long_rows[[length(long_rows) + 1L]] <- na_row(spec_row, scenario_row, year, group, TRUE, paste0("invalid_inputs:", inputs$reason), flags)
          next
        }
        if (identical(scenario_row$scenario_id, "baseline")) {
          long_rows[[length(long_rows) + 1L]] <- baseline_row(spec_row, scenario_row, year, group, flags)
          next
        }
        args <- pif2_build_pif_args(spec_row, record, inputs, scenario_row, exposure, unc, mc, year, group, run_cfg)
        jobs[[length(jobs) + 1L]] <- args
        job_meta[[length(job_meta) + 1L]] <- list(spec_row = spec_row, scenario_row = scenario_row,
                                                  year = year, group = group, flags = flags)
      }
    }
  }
  # ---- run all applicable jobs (each pif_confint controls its own inner MC) ----
  run_one <- function(args) tryCatch(do.call(pif_confint, args), error = function(e) e)
  results <- if (isTRUE(run_cfg$outer_parallel) && length(jobs) > 1L) {
    cl <- parallel::makeCluster(run_cfg$n_cores)
    on.exit(try(parallel::stopCluster(cl), silent = TRUE), add = TRUE)
    engine_file <- normalizePath(file.path(pif2_control_dir, "aaf_unified.R"), winslash = "/")
    parallel::clusterExport(cl, "engine_file", envir = environment())
    parallel::clusterEvalQ(cl, source(engine_file))
    parallel::clusterApplyLB(cl, lapply(jobs, function(a) utils::modifyList(a, list(use_parallel = FALSE, n_cores = 1L))), run_one)
  } else {
    lapply(jobs, run_one)
  }
  for (i in seq_along(results)) {
    res <- results[[i]]; m <- job_meta[[i]]
    if (inherits(res, "error")) {
      long_rows[[length(long_rows) + 1L]] <- na_row(m$spec_row, m$scenario_row, m$year, m$group, TRUE,
                                                    paste0("engine_error:", conditionMessage(res)), m$flags)
      next
    }
    long_rows[[length(long_rows) + 1L]] <- data.frame(
      disease = m$spec_row$disease, sex = m$spec_row$sex, age_group = m$group, year = m$year,
      output_name = m$spec_row$output_name,
      scenario_id = m$scenario_row$scenario_id, scenario_label = m$scenario_row$scenario_label,
      engine_scenario = m$scenario_row$engine_scenario,
      avg_consumption_change_pct = m$scenario_row$volume_reduction_pct,
      hed_prevalence_change_pct  = m$scenario_row$hed_reduction_pct,
      pif = res$point_estimate, pif_low = res$lower_ci, pif_up = res$upper_ci, n_used = res$n_used,
      applicable = TRUE, reason = NA_character_,
      uses_volume_rr = m$flags$uses_volume_rr, uses_hed_rr = m$flags$uses_hed_rr,
      uses_fd_rr = m$flags$uses_fd_rr, uses_beta_vcov = m$flags$uses_beta_vcov,
      uses_weighted_gamma = m$flags$uses_weighted_gamma, uses_design_variance = m$flags$uses_design_variance,
      n_sim = run_cfg$n_sim, stringsAsFactors = FALSE)
  }
  list(results = dplyr::bind_rows(long_rows), audit = dplyr::bind_rows(audit_rows), n_jobs = length(jobs))
}
# ---------------------------------------------------------------------------
# Run configuration. "demo" is a fast, render-friendly pass over the latest wave;
# "full" reproduces the PAF's Monte Carlo depth (n_sim from aaf_mc) over all waves.
# Both use the SAME estimator and the SAME design closures; only breadth/depth differ.
# ---------------------------------------------------------------------------
pif2_pif_run_mode <- "full"   # set to "full" for the complete multi-year, n_sim=10000 run
pif2_pif_run_config <- list(
  demo = list(years = as.integer(tail(pif2_years, 1L)), groups = 1:4,
              n_sim = 1000L, n_pca = 500L, n_cores = as.integer(pif2_aaf_mc$n_cores),
              inner_parallel = FALSE, outer_parallel = TRUE),
  full = list(years = pif2_years, groups = 1:4,
              n_sim = as.integer(pif2_aaf_mc$n_sim), n_pca = as.integer(pif2_aaf_mc$n_pca),
              n_cores = as.integer(pif2_aaf_mc$n_cores),
              inner_parallel = FALSE, outer_parallel = TRUE)
)
pif2_run_cfg <- pif2_pif_run_config[[pif2_pif_run_mode]]

pif2_pif_grid <- pif2_run_pif_grid(
  spec = pif2_output_spec, scenarios = pif2_scenario_grid,
  exposure = pif2_exposure_inputs, unc = pif2_aaf_uncertainty, mc = pif2_aaf_mc,
  years = pif2_run_cfg$years, groups = pif2_run_cfg$groups, run_cfg = pif2_run_cfg)
pif2_pif_results <- pif2_pif_grid$results
pif2_pif_audit   <- pif2_pif_grid$audit
# Attach the list of uncertainty components carried by each row (for provenance).
pif2_pif_results$uncertainty_components <- with(pif2_pif_results, paste(
  ifelse(uses_weighted_gamma, "weighted_gamma", NA),
  ifelse(uses_design_variance, "design_kish_cluster", NA),
  "dirichlet_prevalence",
  ifelse(uses_fd_rr, "fd_lognormal", NA),
  ifelse(uses_beta_vcov, "beta_mvrnorm", NA), sep = "+"))
pif2_pif_results$uncertainty_components <- gsub("(NA\\+)|(\\+NA)", "", pif2_pif_results$uncertainty_components)
# Save with the project's dated convention (does NOT overwrite the AAF bundles).
pif2_pif_stamp <- format(Sys.Date(), "%Y%m%d")
pif2_pif_results_path <- file.path(pif2_control_dir, paste0("pif2_pif_results_", pif2_pif_run_mode, "_", pif2_pif_stamp, ".rds"))
pif2_pif_audit_path   <- file.path(pif2_control_dir, paste0("pif2_pif_audit_",   pif2_pif_run_mode, "_", pif2_pif_stamp, ".rds"))
saveRDS(pif2_pif_results, pif2_pif_results_path)
saveRDS(pif2_pif_audit,   pif2_pif_audit_path)
pif2_message("[run-grid] mode=%s | years=%s | groups=%s | n_sim=%s",
             pif2_pif_run_mode, paste(pif2_run_cfg$years, collapse = ","),
             paste(pif2_run_cfg$groups, collapse = ","), pif2_run_cfg$n_sim)
pif2_message("[run-grid] MC jobs run: %s | deterministic baselines: %s | result rows: %s | applicable: %s | non-applicable: %s",
             pif2_pif_grid$n_jobs,
             sum(pif2_pif_results$applicable & pif2_pif_results$scenario_id == "baseline" & pif2_pif_results$n_used == 0L),
             nrow(pif2_pif_results),
             sum(pif2_pif_results$applicable & !is.na(pif2_pif_results$pif)),
             sum(!pif2_pif_results$applicable))
pif2_message("[run-grid] diseases covered: %s / %s output tables",
             dplyr::n_distinct(pif2_pif_results$output_name[pif2_pif_results$applicable & !is.na(pif2_pif_results$pif)]),
             nrow(pif2_output_spec))
pif2_message("[run-grid] saved: %s", basename(pif2_pif_results_path))

message(sprintf(
  "pif2-run-grid elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[run-grid] mode=full | years=2012,2014,2016,2018,2020,2022,2024 | groups=1,2,3,4 | n_sim=10000
[run-grid] MC jobs run: 7140 | deterministic baselines: 1260 | result rows: 20160 | applicable: 8400 | non-applicable: 11760
[run-grid] diseases covered: 45 / 45 output tables
[run-grid] saved: pif2_pif_results_full_20260720.rds
pif2-run-grid elapsed minutes: 263.40


~ 314 minutes

In [14]:
#| label: pif2-save-synchronised-monte-carlo-draws
#| results: "hold"

# --- Start timing this chunk ---
.t0 <- Sys.time()
# This chunk reruns every applicable non-baseline PIF cell with return_sims = TRUE.
# Baseline draw vectors are deterministic zeros and are constructed directly.
# Draw position i is preserved across cells as draw_id i.
# The existing summary-result RDS files are not modified or overwritten.
# --- Safety check: only allow synchronized draws in full run mode ---
if (!base::identical(pif2_pif_run_mode, "full")) {
  base::stop("Set pif2_pif_run_mode to 'full' before saving synchronized draws.")
}
# --- Function that collects one Monte Carlo draw per applicable PIF cell ---
pif2_collect_synchronised_draws <- function(spec, scenarios, exposure, unc, mc, years, groups, run_cfg) {
  # Lists to accumulate per-cell job arguments and descriptive metadata.
  jobs <- list()
  metadata_rows <- list()
  # Loop over each row of the output specification (cause / sex / measure).
  for (si in base::seq_len(base::nrow(spec))) {
    spec_row <- spec[si, , drop = FALSE]
    # Loop over each policy scenario defined in the scenario grid.
    for (sc in base::seq_len(base::nrow(scenarios))) {
      scenario_row <- scenarios[sc, , drop = FALSE]
      # Determine whether this scenario applies to the current spec row.
      applicability <- pif2_scenario_applicability(spec_row, scenario_row)
      if (!base::isTRUE(applicability$applicable)) {
        next
      }
      # Loop over the target years and age groups.
      for (year in years) {
        for (group in groups) {
          # Look up the precomputed AAF/PAF record for this cell.
          record <- base::tryCatch(pif2_lookup_record(spec_row, group), error = function(e) NULL)
          if (base::is.null(record)) {
            base::stop("Record lookup failed for ", spec_row$output_name, ", scenario ", scenario_row$scenario_id, ", year ", year, ", age group ", group, ".")
          }
          # Resolve exposure and other inputs for the current cell.
          inputs <- pif2_resolve_cell_inputs(spec_row, record, year, group, exposure)
          if (!base::isTRUE(inputs$ok)) {
            base::stop("Invalid inputs for ", spec_row$output_name, ", scenario ", scenario_row$scenario_id, ", year ", year, ", age group ", group, ": ", inputs$reason)
          }
          # Create a stable identifier for this draw set and store the job.
          job_number <- base::length(jobs) + 1L
          draw_key <- base::paste(spec_row$output_name, spec_row$sex, year, group, scenario_row$scenario_id, sep = "|")
          if (base::identical(scenario_row$scenario_id, "baseline")) {
            # The observed and counterfactual risks are identical at shift = 1.
            # Keep one zero draw per draw_id without calling the MC engine.
            jobs[[job_number]] <- list(kind = "baseline", n_sim = base::as.integer(run_cfg$n_sim))
          } else {
            # Force single-core execution inside each call so outer-level parallelism stays safe.
            args <- pif2_build_pif_args(spec_row, record, inputs, scenario_row, exposure, unc, mc, year, group, run_cfg)
            args$return_sims <- TRUE
            args$use_parallel <- FALSE
            args$n_cores <- 1L
            jobs[[job_number]] <- list(kind = "engine", args = args)
          }
          # Store metadata that describes this draw set.
          metadata_rows[[job_number]] <- base::data.frame(
            draw_key = draw_key,
            output_name = spec_row$output_name,
            disease = spec_row$disease,
            sex = spec_row$sex,
            year = base::as.integer(year),
            age_group = base::as.integer(group),
            scenario_id = scenario_row$scenario_id,
            scenario_label = scenario_row$scenario_label,
            engine_scenario = scenario_row$engine_scenario,
            stringsAsFactors = FALSE
          )
        }
      }
    }
  }
  # Combine all metadata rows into a single data frame.
  metadata <- dplyr::bind_rows(metadata_rows)
  # Ensure each cell has a unique key before drawing.
  if (base::anyDuplicated(metadata$draw_key)) {
    base::stop("The synchronized-draw metadata contains duplicated draw keys.")
  }
  # --- Inner helper that runs a single job and catches errors ---
  run_one <- function(job) {
    if (base::identical(job$kind, "baseline")) {
      return(list(ok = TRUE, draws = base::rep.int(0, job$n_sim),
                  n_used = job$n_sim, error = NA_character_))
    }
    base::tryCatch({
      result <- base::do.call(pif_confint, job$args)
      list(ok = TRUE, draws = result$simulated_pifs, n_used = result$n_used, error = NA_character_)
    }, error = function(e) {
      list(ok = FALSE, draws = NULL, n_used = 0L, error = base::conditionMessage(e))
    })
  }
  # --- Execute jobs: optional outer parallelism across cells ---
  if (base::isTRUE(run_cfg$outer_parallel) && base::length(jobs) > 1L) {
    # Set up a cluster and load the PIF engine on each worker.
    cluster <- parallel::makeCluster(run_cfg$n_cores)
    base::on.exit(base::try(parallel::stopCluster(cluster), silent = TRUE), add = TRUE)
    engine_file <- base::normalizePath(base::file.path(pif2_control_dir, "aaf_unified.R"), winslash = "/")
    parallel::clusterExport(cluster, "engine_file", envir = base::environment())
    parallel::clusterEvalQ(cluster, base::source(engine_file))
    # Run jobs in parallel with load balancing.
    results <- parallel::clusterApplyLB(cluster, jobs, run_one)
    parallel::stopCluster(cluster)
  } else {
    # Sequential fallback when parallelism is disabled or only one job exists.
    results <- base::lapply(jobs, run_one)
  }
  # --- Validate that every job succeeded ---
  successful <- base::vapply(results, function(x) base::isTRUE(x$ok), logical(1))
  if (!base::all(successful)) {
    failed <- base::which(!successful)
    failure_text <- base::paste0(
      metadata$draw_key[failed], ": ",
      base::vapply(results[failed], function(x) x$error, character(1))
    )
    base::stop("At least one synchronized PIF run failed:\n", base::paste(failure_text, collapse = "\n"))
  }
  # --- Validate that every cell produced the expected number of simulations ---
  n_used <- base::vapply(results, function(x) base::as.integer(x$n_used), integer(1))
  if (base::any(n_used != run_cfg$n_sim)) {
    bad <- base::which(n_used != run_cfg$n_sim)
    base::stop("Synchronization cannot be guaranteed because some cells lost simulations: ", base::paste(metadata$draw_key[bad], collapse = ", "))
  }
  # Extract the raw PIF draws and name them by their stable draw key.
  pif_draws <- base::lapply(results, function(x) x$draws)
  base::names(pif_draws) <- metadata$draw_key
  # Return metadata, draws, and run-level information.
  list(
    metadata = metadata,
    pif_draws = pif_draws,
    n_jobs = base::length(jobs),
    n_sim = base::as.integer(run_cfg$n_sim),
    seed = base::as.integer(mc$seed)
  )
}
# --- Run the collection function with project-level objects ---
pif2_synchronised_draw_bundle <- pif2_collect_synchronised_draws(
  spec = pif2_output_spec,
  scenarios = pif2_scenario_grid,
  exposure = pif2_exposure_inputs,
  unc = pif2_aaf_uncertainty,
  mc = pif2_aaf_mc,
  years = pif2_run_cfg$years,
  groups = pif2_run_cfg$groups,
  run_cfg = pif2_run_cfg
)
# --- Build a dated file path for the output artifact ---
pif2_draw_stamp <- base::format(base::Sys.Date(), "%Y%m%d")
pif2_synchronised_draw_path <- base::file.path(
  pif2_control_dir,
  base::paste0("pif2_pif_synchronised_draws_full_", pif2_draw_stamp, ".rds")
)
# --- Package everything into the export object, including draw identifiers ---
pif2_synchronised_draw_export <- list(
  schema_version = "1.0",
  created_at = base::format(base::Sys.time(), "%Y-%m-%d %H:%M:%S %Z"),
  seed = pif2_synchronised_draw_bundle$seed,
  n_sim = pif2_synchronised_draw_bundle$n_sim,
  n_jobs = pif2_synchronised_draw_bundle$n_jobs,
  draw_id = base::seq_len(pif2_synchronised_draw_bundle$n_sim),
  metadata = pif2_synchronised_draw_bundle$metadata,
  pif_draws = pif2_synchronised_draw_bundle$pif_draws
)
# --- Save the synchronized draws as a compressed RDS file ---
base::saveRDS(pif2_synchronised_draw_export, pif2_synchronised_draw_path, compress = "gzip")
# --- Post-save validation: reload and check structural integrity ---
pif2_saved_draws_check <- base::readRDS(pif2_synchronised_draw_path)
if (
  base::length(pif2_saved_draws_check$pif_draws) != pif2_synchronised_draw_export$n_jobs ||
  !base::all(base::vapply(pif2_saved_draws_check$pif_draws, base::length, integer(1)) == pif2_synchronised_draw_export$n_sim)
) {
  base::stop("The saved synchronized-draw artifact failed its post-save validation.")
}
# --- Report success and timing ---
pif2_message("[synchronized draws] Saved %s cells x %s Monte Carlo draws.",
             pif2_synchronised_draw_export$n_jobs, pif2_synchronised_draw_export$n_sim)
pif2_message("[synchronized draws] File: %s", pif2_synchronised_draw_path)
base::message(base::sprintf(
  "pif2-save-synchronised-monte-carlo-draws elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))
))

[synchronized draws] Saved 8400 cells x 10000 Monte Carlo draws.
[synchronized draws] File: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control/pif2_pif_synchronised_draws_full_20260721.rds
pif2-save-synchronised-monte-carlo-draws elapsed minutes: 259.75


### PIF result tables

In [15]:
#| label: pif2-tbl-pif-headline
#| results: "hold"

.t0 <- Sys.time()

# Headline PIF table for the reporting wave: one row per (disease, sex, age group),
# one column per scenario, each cell "PIF (lower, upper)".
#
# names_sort = TRUE is REQUIRED: pivot_wider() defaults to names_sort = FALSE, which
# orders the new columns by first appearance rather than by the factor levels. The
# non-applicable rows are appended first by the orchestrator, so without it the table
# would open with the HED scenarios and bury the baseline in the tenth column.
pif2_pif_headline <- pif2_pif_results |>
  dplyr::filter(year == pif2_report_year) |>
  dplyr::mutate(
    scenario_id = factor(scenario_id, levels = pif2_scenario_grid$scenario_id),
    cell        = pif2_fmt_ci(pif, pif_low, pif_up, digits = 3),
    cell        = dplyr::if_else(!applicable, "n/a", cell),
    Sex         = unname(pif2_sex_labels[sex]),
    `Age group` = unname(pif2_age_labels[as.character(age_group)])
  ) |>
  dplyr::select(Disease = disease, Sex, `Age group`, scenario_id, cell) |>
  tidyr::pivot_wider(names_from = scenario_id, values_from = cell, names_sort = TRUE) |>
  dplyr::arrange(Disease, Sex, `Age group`)

message(sprintf(
  "pif2-tbl-pif-headline elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

pif2_html_table(
  pif2_pif_headline,
  caption = sprintf(
    "Potential Impact Fraction by disease, sex and age group -- survey wave %s (point estimate and 95%% CI)",
    pif2_report_year
  ),
  note = paste(
    "'n/a' = the scenario does not apply to that cause (no HED RR component); it is never a zero.",
    "Negative PIFs are retained, not clipped: they arise on the cardioprotective (J-curve) causes,",
    "where a volume reduction removes protection.",
    sprintf("Monte Carlo depth: n_sim = %s. The full multi-wave grid stays in pif2_pif_results_*.rds.",
            paste(unique(pif2_pif_results$n_sim), collapse = "/"))
  ),
  max_height = 600,
  round_numeric = FALSE
)

pif2-tbl-pif-headline elapsed minutes: 0.00


Disease,Sex,Age group,baseline,volume_reduction_10,volume_reduction_20,volume_reduction_30,hed_reduction_10,hed_reduction_25,hed_reduction_50,combined_v10_h25,combined_v20_h50,combined_v30_h50,hed_reduction_10_rt,hed_reduction_25_rt,hed_reduction_50_rt,combined_v10_h25_rt,combined_v20_h50_rt,combined_v30_h50_rt
Acute Pancreatitis,Men,15-29,"0.000 (0.000, 0.000)","0.012 (0.005, 0.025)","0.023 (0.009, 0.047)","0.033 (0.014, 0.066)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Acute Pancreatitis,Men,30-44,"0.000 (0.000, 0.000)","0.022 (0.008, 0.048)","0.042 (0.016, 0.088)","0.059 (0.024, 0.122)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Acute Pancreatitis,Men,45-59,"0.000 (0.000, 0.000)","0.021 (0.008, 0.048)","0.040 (0.015, 0.087)","0.057 (0.022, 0.120)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Acute Pancreatitis,Men,60-65,"0.000 (0.000, 0.000)","0.015 (0.005, 0.036)","0.028 (0.010, 0.066)","0.040 (0.014, 0.090)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Acute Pancreatitis,Women,15-29,"0.000 (0.000, 0.000)","0.005 (-0.001, 0.027)","0.008 (-0.003, 0.050)","0.011 (-0.005, 0.067)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Acute Pancreatitis,Women,30-44,"0.000 (0.000, 0.000)","0.004 (-0.001, 0.023)","0.007 (-0.003, 0.040)","0.009 (-0.005, 0.053)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Acute Pancreatitis,Women,45-59,"0.000 (0.000, 0.000)","0.002 (-0.001, 0.010)","0.002 (-0.003, 0.018)","0.003 (-0.005, 0.023)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Acute Pancreatitis,Women,60-65,"0.000 (0.000, 0.000)","0.004 (-0.000, 0.021)","0.007 (-0.001, 0.040)","0.010 (-0.001, 0.056)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Breast Cancer,Women,15-29,"0.000 (0.000, 0.000)","0.002 (0.001, 0.002)","0.004 (0.003, 0.005)","0.006 (0.004, 0.007)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a
Breast Cancer,Women,30-44,"0.000 (0.000, 0.000)","0.002 (0.001, 0.002)","0.004 (0.003, 0.005)","0.006 (0.005, 0.007)",n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a


In [16]:
#| label: pif2-tbl-pif-summary
#| results: "hold"

.t0 <- Sys.time()

# (a) Distribution of the cell-level PIF, per scenario and sex. DIAGNOSTIC ONLY:
#     the unweighted mean of a set of PIFs is not itself a PIF. The aggregate
#     estimates are the death-weighted tables further down.
pif2_scenario_summary <- pif2_pif_results |>
  dplyr::filter(applicable, !is.na(pif)) |>
  dplyr::group_by(scenario_id, scenario_label, sex) |>
  dplyr::summarise(
    n_cells    = dplyr::n(),
    mean_pif   = mean(pif),
    median_pif = stats::median(pif),
    min_pif    = min(pif),
    max_pif    = max(pif),
    n_negative = sum(pif < -1e-9),
    .groups    = "drop"
  ) |>
  dplyr::mutate(
    scenario_id = factor(scenario_id, levels = pif2_scenario_grid$scenario_id),
    sex         = unname(pif2_sex_labels[sex])
  ) |>
  dplyr::arrange(scenario_id, sex) |>
  dplyr::select(
    Scenario = scenario_id, Label = scenario_label, Sex = sex, Cells = n_cells,
    `Mean PIF` = mean_pif, `Median PIF` = median_pif, `Min PIF` = min_pif,
    `Max PIF` = max_pif, `Negative PIFs` = n_negative
  )

# (b) Every cell that produced no estimate, with its machine-readable reason. The
#     'no HED RR component' rows are by design; an 'engine_error:' or
#     'invalid_inputs:' row would be a real defect.
pif2_pif_gaps <- pif2_pif_results |>
  dplyr::filter(!applicable | is.na(pif)) |>
  dplyr::count(scenario_id, disease, sex, reason, name = "n_cells") |>
  dplyr::mutate(sex = unname(pif2_sex_labels[sex])) |>
  dplyr::arrange(reason, disease, sex, scenario_id) |>
  dplyr::select(Scenario = scenario_id, Disease = disease, Sex = sex,
                Reason = reason, Cells = n_cells)

message(sprintf(
  "pif2-tbl-pif-summary elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

pif2_html_tables(
  pif2_html_table(
    pif2_scenario_summary,
    caption = "Distribution of the cell-level PIF by scenario and sex (all diseases, waves and age groups)",
    note = paste(
      "DIAGNOSTIC ONLY. These are unweighted summaries of the cell-level PIFs, and the mean of a set",
      "of PIFs is not itself a PIF. The aggregate estimates are the death-weighted tables below.",
      "Negative PIFs are a property of the cardioprotective (J-curve) RRs, not an engine error."
    ),
    max_height = 480,
    digits = 4
  ),
  pif2_html_table(
    pif2_pif_gaps,
    caption = sprintf("Cells without a PIF estimate (%s of %s rows)",
                      sum(pif2_pif_gaps$Cells), nrow(pif2_pif_results)),
    note = paste(
      "'no HED RR component' rows are by design: an HED or combined scenario cannot act on a",
      "volume-only cause. Any 'engine_error:' or 'invalid_inputs:' row is a real defect and must",
      "be investigated before the results are used."
    ),
    max_height = 400
  )
)

pif2-tbl-pif-summary elapsed minutes: 0.00


Scenario,Label,Sex,Cells,Mean PIF,Median PIF,Min PIF,Max PIF,Negative PIFs
baseline,Baseline (no change),Men,616,0.0000,0.0000,0.0000,0.0000,0
baseline,Baseline (no change),Women,644,0.0000,0.0000,0.0000,0.0000,0
volume_reduction_10,Average consumption -10%,Men,616,0.0123,0.0048,-0.0009,0.0839,14
volume_reduction_10,Average consumption -10%,Women,644,0.0040,0.0018,-0.0027,0.0351,47
volume_reduction_20,Average consumption -20%,Men,616,0.0233,0.0095,-0.0020,0.1479,20
volume_reduction_20,Average consumption -20%,Women,644,0.0077,0.0035,-0.0058,0.0685,56
volume_reduction_30,Average consumption -30%,Men,616,0.0333,0.0142,-0.0033,0.1993,25
volume_reduction_30,Average consumption -30%,Women,644,0.0112,0.0051,-0.0096,0.1005,64
hed_reduction_10,HED prevalence -10%,Men,140,0.0154,0.0163,0.0011,0.0402,0
hed_reduction_10,HED prevalence -10%,Women,140,0.0097,0.0084,0.0003,0.0299,0


In [17]:
#| label: pif2-tbl-pif-audit
#| results: "hold"

.t0 <- Sys.time()

# One row per (output table x scenario): which uncertainty sources that cell's PIF
# actually carries, and whether the scenario applies to the cause at all.
pif2_audit_display <- pif2_pif_audit |>
  dplyr::mutate(
    sex         = unname(pif2_sex_labels[sex]),
    scenario_id = factor(scenario_id, levels = pif2_scenario_grid$scenario_id)
  ) |>
  dplyr::arrange(disease, sex, scenario_id) |>
  dplyr::select(
    `Output table` = disease_id, Disease = disease, Sex = sex, Scenario = scenario_id,
    `Volume RR` = uses_volume_rr, `HED RR` = uses_hed_rr, `Former-drinker RR` = uses_fd_rr,
    `Beta covariance` = uses_beta_vcov, `Weighted gamma` = uses_weighted_gamma,
    `Survey design` = uses_design_variance, `Age support` = age_support,
    Applicable = applicable, Reason = reason
  )
message(sprintf(
  "pif2-tbl-pif-audit elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))
pif2_html_table(
  pif2_audit_display,
  caption = sprintf("Disease x scenario audit: uncertainty sources carried by each PIF (%s rows)",
                    nrow(pif2_audit_display)),
  note = paste(
    "'Survey design' = the Kish effective n and the PSU/region cluster factor, reused from the PAF's",
    "own design closures (so the design is applied exactly once). 'Beta covariance' = joint mvrnorm",
    "draw of the RR betas."
  ),
  max_height = 500
)

pif2-tbl-pif-audit elapsed minutes: 0.00


Output table,Disease,Sex,Scenario,Volume RR,HED RR,Former-drinker RR,Beta covariance,Weighted gamma,Survey design,Age support,Applicable,Reason
panc_male,Acute Pancreatitis,Men,baseline,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),TRUE,
panc_male,Acute Pancreatitis,Men,volume_reduction_10,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),TRUE,
panc_male,Acute Pancreatitis,Men,volume_reduction_20,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),TRUE,
panc_male,Acute Pancreatitis,Men,volume_reduction_30,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),TRUE,
panc_male,Acute Pancreatitis,Men,hed_reduction_10,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),FALSE,no HED RR component (volume-only cause); HED/combined scenario not applicable
panc_male,Acute Pancreatitis,Men,hed_reduction_25,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),FALSE,no HED RR component (volume-only cause); HED/combined scenario not applicable
panc_male,Acute Pancreatitis,Men,hed_reduction_50,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),FALSE,no HED RR component (volume-only cause); HED/combined scenario not applicable
panc_male,Acute Pancreatitis,Men,combined_v10_h25,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),FALSE,no HED RR component (volume-only cause); HED/combined scenario not applicable
panc_male,Acute Pancreatitis,Men,combined_v20_h50,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),FALSE,no HED RR component (volume-only cause); HED/combined scenario not applicable
panc_male,Acute Pancreatitis,Men,combined_v30_h50,TRUE,FALSE,TRUE,FALSE,TRUE,TRUE,15-64 (edad_tramo 1-4),FALSE,no HED RR component (volume-only cause); HED/combined scenario not applicable


### From PIF to averted deaths

The mortality table loaded at the top of this notebook carries **attributable** deaths
(`mort = AAF x n`). The next chunk recovers the **total** deaths of each cause cell
(`n = mort / AAF`, an exact recovery whose integrality also proves the mortality export and
the AAF bundle came from the same engine run) and applies the PIF to them.

In [18]:
#| label: pif2-deaths-bridge
#| results: "hold"

.t0 <- Sys.time()

# ===========================================================================
# From PIF to averted deaths.
#
# WHICH MULTIPLIER IS CORRECT, AND WHY.
# The engine defines BOTH fractions against the SAME population risk R_obs
# (aaf_unified.R): .aaf_core() returns AAF = num/(num+1) with num+1 == R_obs, and
# .pif_core() returns PIF = 1 - R_cf/R_obs from the same .aaf_pop_R() call. So
#       AAF = 1 - 1/R_obs        and        PIF = 1 - R_cf/R_obs
# are both shares of the TOTAL deaths of the cause, and therefore
#       averted deaths = TOTAL deaths x PIF
# NOT attributable deaths x PIF, which would understate the effect by a factor
# equal to the AAF itself.
# Verified numerically against this engine (2026-07-11): on a volume-only cause with
# RR_FD forced to 1, PIF(volume, shift = 0) reproduces the AAF to 2e-17; and the
# identity PIF = (AAF - AAF_cf)/(1 - AAF_cf) holds to 3e-16 over every nohed cause
# and every volume shift. The ratio PIF/AAF is the share of the ATTRIBUTABLE burden
# a scenario averts -- a ratio, not a count.
#
# RECOVERING TOTAL DEATHS.
# pif2_mortality_results carries ATTRIBUTABLE deaths only: expand_pif.ipynb builds
# them as mort = AAF x n, where n is the ICD-coded death count, and exports
# (year, age_group, gender, disease, mort, ll_mort, up_mort). The total n is therefore
# recovered exactly as n = mort / AAF, with the AAF already in pif2_aaf_long. The
# recovered n must be a whole number: that assertion also proves the mortality export
# and the AAF bundle came from the same engine run.
#
# THE GUARD IS abs(aaf) > 0, NOT aaf > 0.
# Where the AAF is NEGATIVE (the cardioprotective cells: ischaemic stroke, and DM2 in
# women) the attributable deaths are negative too, so mort/aaf is STILL the correct
# positive death count. An `aaf > 0` guard would silently turn those 68 cells into NA
# and delete ischaemic stroke from every table below.
# ===========================================================================

pif2_aaf_for_join <- pif2_aaf_long |>
  dplyr::distinct(year, age_group, gender, disease, .keep_all = TRUE) |>
  dplyr::select(year, age_group, gender, disease, aaf = point)

pif2_deaths <- tibble::as_tibble(pif2_mortality_results) |>
  dplyr::inner_join(pif2_aaf_for_join, by = c("year", "age_group", "gender", "disease")) |>
  dplyr::mutate(
    deaths_attributable = mort,
    deaths_total = dplyr::if_else(is.finite(aaf) & abs(aaf) > 1e-12, mort / aaf, NA_real_)
  )

pif2_deaths_residual <- max(
  abs(pif2_deaths$deaths_total - round(pif2_deaths$deaths_total)), na.rm = TRUE
)
pif2_message("[deaths] Total deaths recovered for %s cells | max non-integer residual: %.2e",
             sum(!is.na(pif2_deaths$deaths_total)), pif2_deaths_residual)
if (!is.finite(pif2_deaths_residual) || pif2_deaths_residual > 1e-6) {
  stop("Recovered total deaths are not whole numbers: the mortality export and the AAF bundle ",
       "are not from the same run. Re-export mortality_results from expand_pif.ipynb.")
}
# Averted deaths per (disease x sex x age group x wave x scenario).
#   avoided_deaths            -> engine-correct: TOTAL deaths x PIF
#   avoided_deaths_att_conv   -> attributable x PIF, kept ONLY so the two conventions
#                                can be compared in a table (it is NOT the estimate)
pif2_avoided_deaths <- pif2_pif_results |>
  dplyr::filter(applicable, !is.na(pif)) |>
  dplyr::mutate(gender = pif2_gender_es(sex)) |>
  dplyr::inner_join(
    dplyr::select(pif2_deaths, year, age_group, gender, disease,
                  aaf, deaths_total, deaths_attributable),
    by = c("year", "age_group", "gender", "disease")
  ) |>
  dplyr::mutate(
    avoided_deaths          = deaths_total * pif,
    avoided_deaths_low      = deaths_total * pif_low,
    avoided_deaths_up       = deaths_total * pif_up,
    avoided_deaths_att_conv = deaths_attributable * pif,
    category                = pif2_cause_category(disease)
  )
if (any(pif2_avoided_deaths$category == "Uncategorized")) {
  warning("Diseases outside the cause-group map: ",
          paste(unique(pif2_avoided_deaths$disease[pif2_avoided_deaths$category == "Uncategorized"]),
                collapse = ", "))
}
pif2_message("[deaths] Averted-death rows: %s | diseases joined: %s of %s in the PIF grid",
             nrow(pif2_avoided_deaths),
             dplyr::n_distinct(pif2_avoided_deaths$disease),
             dplyr::n_distinct(pif2_pif_results$disease))
pif2_message("[deaths] SCOPE: the 100%%-attributable causes are NOT in the PIF grid, so every")
pif2_message("[deaths]        averted-deaths total below EXCLUDES them (see the note under this section).")

message(sprintf(
  "pif2-deaths-bridge elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[deaths] Total deaths recovered for 1188 cells | max non-integer residual: 2.27e-13
[deaths] Averted-death rows: 8112 | diseases joined: 23 of 23 in the PIF grid
[deaths] SCOPE: the 100%-attributable causes are NOT in the PIF grid, so every
[deaths]        averted-deaths total below EXCLUDES them (see the note under this section).
pif2-deaths-bridge elapsed minutes: 0.00


In [19]:
#| label: pif2-tbl-averted-by-category
#| results: "hold"

.t0 <- Sys.time()
# Aggregation by cause group (the same five groups as the figures of expand_pif.ipynb).
#
# A PIF CANNOT BE AVERAGED across diseases: the unweighted mean of a set of PIFs has no
# epidemiological meaning. The only correct aggregate is death-weighted,
#       PIF_group = sum(averted deaths) / sum(total deaths),
# which is what is computed here. The denominator covers only the causes the scenario
# can act on (an HED scenario reaches no cancer at all, and only 2 of the 4
# cardiovascular causes), so the coverage table below reports that explicitly -- and it
# is why the numbers in one row are NOT comparable across scenario columns.
pif2_pif_category <- pif2_avoided_deaths |>
  dplyr::filter(year == pif2_report_year) |>
  dplyr::group_by(category, gender, age_group, scenario_id, scenario_label) |>
  dplyr::summarise(
    causes       = dplyr::n_distinct(disease[!is.na(deaths_total)]),
    deaths_total = sum(deaths_total, na.rm = TRUE),
    deaths_att   = sum(deaths_attributable, na.rm = TRUE),
    averted      = sum(avoided_deaths, na.rm = TRUE),
    averted_low  = sum(avoided_deaths_low, na.rm = TRUE),
    averted_up   = sum(avoided_deaths_up, na.rm = TRUE),
    .groups      = "drop"
  ) |>
  dplyr::mutate(
    pif_weighted = dplyr::if_else(deaths_total > 0, averted / deaths_total, NA_real_),
    scenario_id  = factor(scenario_id, levels = pif2_scenario_grid$scenario_id),
    Sex          = dplyr::recode(gender, "Hombre" = "Men", "Mujer" = "Women"),
    `Age group`  = unname(pif2_age_labels[as.character(age_group)])
  )
pif2_cat_pif_wide <- pif2_pif_category |>
  dplyr::mutate(cell = sprintf("%.3f", pif_weighted)) |>
  dplyr::select(Category = category, Sex, `Age group`, scenario_id, cell) |>
  tidyr::pivot_wider(names_from = scenario_id, values_from = cell, names_sort = TRUE) |>
  dplyr::arrange(Category, Sex, `Age group`)
pif2_cat_deaths_wide <- pif2_pif_category |>
  dplyr::mutate(cell = sprintf("%.0f (%.0f, %.0f)", averted, averted_low, averted_up)) |>
  dplyr::select(Category = category, Sex, `Age group`, scenario_id, cell) |>
  tidyr::pivot_wider(names_from = scenario_id, values_from = cell, names_sort = TRUE) |>
  dplyr::arrange(Category, Sex, `Age group`)
# Coverage: how many causes of the group each scenario actually reaches, and how many
# deaths therefore sit in its denominator. Both change with the scenario, which is
# exactly why they cannot live inside the two wide tables above.
pif2_cat_coverage <- pif2_pif_category |>
  dplyr::group_by(Category = category, Scenario = scenario_id) |>
  dplyr::summarise(
    `Causes reached`        = max(causes),
    `Deaths in denominator` = sum(deaths_total),
    `Attributable deaths`   = sum(deaths_att),
    .groups = "drop"
  ) |>
  dplyr::arrange(Category, Scenario)
message(sprintf(
  "pif2-tbl-averted-by-category elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))
pif2_html_tables(
  pif2_html_table(
    pif2_cat_pif_wide,
    caption = sprintf("Death-weighted PIF by cause group, sex and age group -- wave %s", pif2_report_year),
    note = paste(
      "PIF_group = sum(averted deaths) / sum(total deaths of the causes the scenario reaches).",
      "This is NOT an average of the disease-level PIFs, which would have no meaning. A blank cell",
      "means the scenario reaches no cause in that group (e.g. HED scenarios and cancers). Negative",
      "values on Cardiovascular are the J-curve: a volume cut removes cardioprotection."
    ),
    max_height = 550,
    round_numeric = FALSE
  ),
  pif2_html_table(
    pif2_cat_deaths_wide,
    caption = sprintf("Averted deaths by cause group, sex and age group -- wave %s (point estimate and 95%% CI)",
                      pif2_report_year),
    note = paste(
      "Averted deaths = TOTAL deaths x PIF, summed over the causes of the group. The interval sums",
      "the per-cell bounds, which assumes the cells are perfectly correlated: it is a conservative",
      "(wide) envelope, not a Monte Carlo interval for the sum."
    ),
    max_height = 550,
    round_numeric = FALSE
  ),
  pif2_html_table(
    pif2_cat_coverage,
    caption = sprintf("Scenario coverage by cause group -- wave %s", pif2_report_year),
    note = paste(
      "A scenario enters the aggregate only for the causes whose RR structure it can act on: the HED",
      "and combined scenarios reach no cancer at all, and only the two J-curve causes (ischaemic",
      "heart disease, ischaemic stroke) of the cardiovascular group. Because the denominator changes",
      "with the scenario, the cells of one row above are NOT comparable across scenario columns."
    ),
    max_height = 500,
    digits = 0
  )
)

pif2-tbl-averted-by-category elapsed minutes: 0.00


Category,Sex,Age group,baseline,volume_reduction_10,volume_reduction_20,volume_reduction_30,hed_reduction_10,hed_reduction_25,hed_reduction_50,combined_v10_h25,combined_v20_h50,combined_v30_h50,hed_reduction_10_rt,hed_reduction_25_rt,hed_reduction_50_rt,combined_v10_h25_rt,combined_v20_h50_rt,combined_v30_h50_rt
Cancer,Men,15-29,0.000,0.003,0.006,0.009,,,,,,,,,,,,
Cancer,Men,30-44,0.000,0.005,0.010,0.014,,,,,,,,,,,,
Cancer,Men,45-59,0.000,0.005,0.010,0.015,,,,,,,,,,,,
Cancer,Men,60-65,0.000,0.004,0.007,0.010,,,,,,,,,,,,
Cancer,Women,15-29,0.000,0.003,0.005,0.008,,,,,,,,,,,,
Cancer,Women,30-44,0.000,0.002,0.004,0.006,,,,,,,,,,,,
Cancer,Women,45-59,0.000,0.001,0.003,0.004,,,,,,,,,,,,
Cancer,Women,60-65,0.000,0.001,0.002,0.003,,,,,,,,,,,,
Cardiovascular,Men,15-29,0.000,0.001,0.002,0.003,0.003,0.007,0.014,0.007,0.013,0.012,0.002,0.006,0.011,0.005,0.009,0.008
Cardiovascular,Men,30-44,0.000,0.002,0.004,0.006,0.003,0.008,0.017,0.009,0.017,0.016,0.003,0.007,0.014,0.007,0.012,0.011


In [20]:
#| label: pif2-tbl-averted-deaths
#| results: "hold"

.t0 <- Sys.time()
# (a) Averted deaths by scenario and wave, under the engine-correct convention.
pif2_deaths_by_scenario <- pif2_avoided_deaths |>
  dplyr::group_by(scenario_id, scenario_label, year) |>
  dplyr::summarise(
    deaths_reached = sum(deaths_total, na.rm = TRUE),
    averted        = sum(avoided_deaths, na.rm = TRUE),
    averted_low    = sum(avoided_deaths_low, na.rm = TRUE),
    averted_up     = sum(avoided_deaths_up, na.rm = TRUE),
    averted_att_cv = sum(avoided_deaths_att_conv, na.rm = TRUE),
    .groups        = "drop"
  ) |>
  dplyr::mutate(scenario_id = factor(scenario_id, levels = pif2_scenario_grid$scenario_id))
pif2_deaths_wide <- pif2_deaths_by_scenario |>
  dplyr::mutate(cell = sprintf("%.0f (%.0f, %.0f)", averted, averted_low, averted_up)) |>
  dplyr::select(scenario_id, scenario_label, year, cell) |>
  tidyr::pivot_wider(names_from = year, values_from = cell, names_sort = TRUE) |>
  dplyr::arrange(scenario_id) |>
  dplyr::rename(Scenario = scenario_id, Label = scenario_label)
# (b) The two conventions side by side, so the size of the difference is visible
#     rather than asserted. The ratio is the death-weighted 1/AAF.
pif2_conventions <- pif2_deaths_by_scenario |>
  dplyr::filter(year == pif2_report_year) |>
  dplyr::mutate(ratio = dplyr::if_else(abs(averted_att_cv) > 1e-9, averted / averted_att_cv, NA_real_)) |>
  dplyr::select(
    Scenario = scenario_id, Label = scenario_label,
    `Deaths of the causes this scenario reaches` = deaths_reached,
    `Averted: TOTAL x PIF (correct)` = averted,
    `Averted: attributable x PIF (old convention)` = averted_att_cv,
    Ratio = ratio
  ) |>
  dplyr::arrange(Scenario)
# (c) Numerator and denominator of the "share of the attributable burden" PUBLISHED AS
#     SEPARATE COLUMNS, deliberately without printing the ratio. The denominator nets
#     the NEGATIVE attributable deaths of the cardioprotective causes (ischaemic stroke,
#     DM2 in women) against the positive ones, so it can pass close to zero and the
#     ratio would be unstable and easy to misread. The reader can form it where it is
#     meaningful; the two columns are always well defined.
pif2_share_components <- pif2_avoided_deaths |>
  dplyr::filter(year == pif2_report_year) |>
  dplyr::group_by(Category = category, Scenario = scenario_id) |>
  dplyr::summarise(
    `Averted deaths (numerator)`          = sum(avoided_deaths, na.rm = TRUE),
    `Attributable deaths (denominator)`   = sum(deaths_attributable, na.rm = TRUE),
    .groups = "drop"
  ) |>
  dplyr::mutate(Scenario = factor(Scenario, levels = pif2_scenario_grid$scenario_id)) |>
  dplyr::arrange(Category, Scenario)
message(sprintf(
  "pif2-tbl-averted-deaths elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))
pif2_html_tables(
  pif2_html_table(
    pif2_deaths_wide,
    caption = "Averted deaths by scenario and survey wave, ages 15-65 (point estimate and 95% CI)",
    note = paste(
      "Averted deaths = TOTAL deaths x PIF, summed over the causes each scenario reaches, both sexes",
      "and the four age groups. The volume scenarios reach all 23 pipeline diseases; the HED and",
      "combined scenarios reach only the HED-capable ones, so the two families are summed over",
      "DIFFERENT populations. The 100%-attributable causes are not in the PIF grid and are excluded."
    ),
    max_height = 450,
    round_numeric = FALSE
  ),
  pif2_html_table(
    pif2_conventions,
    caption = sprintf("The two conventions compared -- wave %s", pif2_report_year),
    note = paste(
      "The engine's PIF is a share of the TOTAL deaths of the cause (it shares the AAF's denominator),",
      "so TOTAL x PIF is the estimate. Multiplying ATTRIBUTABLE deaths by the PIF understates the",
      "averted burden by a factor equal to the AAF; the Ratio column is that factor (death-weighted 1/AAF)."
    ),
    max_height = 400,
    digits = 1
  ),
  pif2_html_table(
    pif2_share_components,
    caption = sprintf("Share of the attributable burden: numerator and denominator -- wave %s",
                      pif2_report_year),
    note = paste(
      "Published as two columns rather than as a percentage on purpose. The denominator nets the",
      "NEGATIVE attributable deaths of the cardioprotective causes (ischaemic stroke, DM2 in women)",
      "against the positive ones, so it can approach zero and the ratio becomes unstable and easy to",
      "misread. Form the ratio only where the denominator is comfortably positive."
    ),
    max_height = 520,
    digits = 1
  )
)

pif2-tbl-averted-deaths elapsed minutes: 0.00


Scenario,Label,2012,2014,2016,2018,2020,2022,2024
baseline,Baseline (no change),"0 (0, 0)","0 (0, 0)","0 (0, 0)","0 (0, 0)","0 (0, 0)","0 (0, 0)","0 (0, 0)"
volume_reduction_10,Average consumption -10%,"134 (64, 235)","160 (76, 275)","121 (56, 211)","98 (46, 170)","147 (66, 255)","135 (62, 241)","93 (41, 173)"
volume_reduction_20,Average consumption -20%,"247 (119, 427)","294 (141, 498)","223 (103, 384)","182 (85, 312)","270 (121, 462)","249 (115, 437)","173 (77, 317)"
volume_reduction_30,Average consumption -30%,"345 (166, 588)","408 (197, 682)","311 (144, 530)","255 (120, 433)","374 (169, 633)","346 (161, 601)","242 (108, 439)"
hed_reduction_10,HED prevalence -10%,"127 (46, 211)","131 (46, 219)","130 (45, 216)","137 (51, 226)","131 (49, 217)","136 (48, 231)","102 (34, 180)"
hed_reduction_25,HED prevalence -25%,"317 (115, 528)","327 (116, 548)","324 (114, 540)","342 (127, 564)","328 (124, 543)","341 (121, 579)","256 (85, 450)"
hed_reduction_50,HED prevalence -50%,"633 (230, 1056)","653 (231, 1096)","649 (227, 1081)","684 (255, 1128)","655 (247, 1086)","682 (241, 1157)","512 (171, 900)"
combined_v10_h25,"Combined: volume -10%, HED -25%","331 (128, 545)","344 (132, 568)","337 (126, 552)","353 (138, 575)","343 (138, 559)","356 (135, 595)","268 (97, 464)"
combined_v20_h50,"Combined: volume -20%, HED -50%","657 (254, 1082)","683 (261, 1126)","669 (250, 1097)","703 (275, 1144)","681 (273, 1110)","708 (268, 1183)","533 (192, 922)"
combined_v30_h50,"Combined: volume -30%, HED -50%","667 (264, 1093)","695 (273, 1139)","678 (259, 1105)","710 (284, 1150)","692 (284, 1121)","718 (279, 1194)","542 (201, 931)"


### Notes on interpretation

Verified directly against the engine and the saved artifacts on 2026-07-11.

**1. The PIF is a share of TOTAL deaths, not of attributable deaths.** `.aaf_core()` returns
`AAF = num/(num+1)` where `num + 1` is exactly the population risk `R_obs` that `.pif_core()`
divides into: `PIF = 1 - R_cf/R_obs` and `AAF = 1 - 1/R_obs` share the same denominator. So
`averted deaths = TOTAL deaths x PIF`. Multiplying **attributable** deaths by the PIF
understates the averted burden by a factor equal to the AAF itself -- between 2.5x and 5x on
this grid (see "The two conventions compared"). The ratio `PIF / AAF` is the share of the
*attributable* burden a scenario averts: a ratio, not a count.

**2. `PIF(shift -> 0)` does NOT converge to the AAF, and must not be used as a validation
check.** The volume and HED counterfactuals leave the former-drinker term `p_form * RR_FD`
untouched, so driving the exposure to zero still leaves that excess risk standing. The header
comment of `aaf_unified.R` ("PAF = PIF de eliminacion total") holds only for a counterfactual
that *also* removes the former-drinker excess. The identity that *does* hold, to 3e-16 on every
cause tested, is `PIF = (AAF - AAF_cf) / (1 - AAF_cf)`.

**3. Substantive: a volume-only policy leaves most of the binge excess intact for injuries.**
The injury RR carries an explicit HED component whose second beta is volume-independent:
`RR_binge(x = 0) = 2.62`. Driving average consumption to zero therefore does **not** remove the
binge risk -- which is exactly why the HED and combined scenarios avert so much more than the
volume scenarios on injuries, and why the two families are not substitutes.

**4. Scope: the 100%-attributable causes are not in the PIF grid.** The mortality table carries a
"Fully attributable to alcohol" block (alcohol poisoning, alcohol use disorders, alcoholic liver
disease -- 2,628 deaths across the waves, 145 in 2024) that has no dose-response RR and therefore
no AAF row and no PIF. Every averted-deaths total in this notebook **excludes** it. A policy-facing
figure would need those causes added under an explicit assumption about how exposure reduction
maps onto them.

## Full injuries test

This cell runs a complete validation of the injury-related PIF results.

It checks all injury output tables (Road injuries, Unintentional injuries and Intentional injuries, each one by gender= 6) against every policy scenario (10), survey wave (2012-2024=7), and age group (4 groups). In total, it evaluates 1,680 combinations using the same simulation size as the AAF/PAF objects. Because injury causes include both average alcohol volume and HED components, all volume-only, HED-only, and combined scenarios should produce valid results.

The purpose of this test is to confirm that the injury PIF pipeline behaves as expected before relying on the final scenario results. It verifies that every expected combination is covered, that no model inputs are missing, that baseline no-change scenarios return zero effect, and that estimated intervals are finite and correctly ordered.

The test also checks whether stronger reductions in alcohol exposure produce stronger or equal effects, whether combined scenarios are at least as large as their individual components, and whether repeated runs with the same seed reproduce the same results.

Some checks are treated as warnings rather than failures. For example, the comparison against the full grid is only possible if the grid has already been run, and some combined-scenario behavior is evaluated as a useful diagnostic rather than a strict rule.

The cell saves two outputs: one file with the full injury test results and one file with the validation checks.

In [21]:
#| label: pif2-injuries-full-test
#| results: "hold"

.t0 <- Sys.time()
# ===========================================================================
# FULL injuries PIF test (replaces the optional single-cell smoke test).
#
# Scope (no sampling, no shortcuts):
#   * diseases : every EXPLICIT-mode (injury) output table in pif2_output_spec ->
#                ri_fem / ri_male             (Road Injuries,          injuries_MVA)
#                injuries_fem / injuries_male (Unintentional Injuries, injuries_other_unit)
#                violence_fem / violence_male (Intentional Injuries,   injuries_other_int)
#   * scenarios: ALL rows of pif2_scenario_grid. Injuries carry an explicit HED RR
#                component (uses_hed = TRUE), so every volume / HED / combined
#                scenario is applicable -- no NA cells are expected here.
#   * years    : ALL survey waves (pif2_years)
#   * groups   : ALL four age groups (1-4, i.e. 15-64)
#   * depth    : the PAF's own Monte Carlo settings (pif2_aaf_mc), not a toy n_sim
#
# This cell ALWAYS runs: it no longer honours pif2_run_smoke_tests.
#
# It is deliberately self-contained. It reuses the CELL-LEVEL resolvers
# (pif2_scenario_applicability, pif2_resolve_cell_inputs, pif2_build_pif_args,
# pif2_uncertainty_flags) and the engine (pif_confint), but walks the cells with
# its own thin loop, because pif2_run_pif_grid() is defined in the same chunk that
# executes the ~314-minute full grid. The per-cell math is therefore identical to
# the grid's by construction; only the loop lives here. When the full grid HAS
# been run, check `grid_cross_check` below reconciles the two row by row.
# ===========================================================================

# ---------------------------------------------------------------------------
# 1. Run configuration.
#
# Cores: NEVER trust the bundle's n_cores. pif2_aaf_mc$n_cores records the machine
# the PAF ran on, not this one. We ask for more, then clamp to what the OS actually
# reports, keeping headroom for the OS, the IDE and the master R process (which
# stays alive holding the whole job list while the workers run).
#
# Parallelism policy, identical to the grid's: the inner Monte Carlo stays SERIAL
# and parallelism is applied at the CELL level. That preserves the engine's
# serial == parallel invariant (each cell re-seeds its own L'Ecuyer streams from
# the fixed seed, so a scenario and its baseline share the same draws and the Monte
# Carlo noise cancels between them), and it guarantees no nested clusters.
# ---------------------------------------------------------------------------
pif2_inj_resolve_cores <- function(requested, reserve = 4L) {
  avail <- as.integer(parallel::detectCores(logical = TRUE))
  if (!is.finite(avail) || avail < 1L) avail <- 1L
  max(1L, min(as.integer(requested), avail - reserve))
}
pif2_inj_cfg <- list(
  n_sim          = as.integer(pif2_aaf_mc$n_sim),   # 10000: PAF-congruent depth
  n_pca          = as.integer(pif2_aaf_mc$n_pca),   # 1000
  n_cores        = pif2_inj_resolve_cores(requested = 20L, reserve = 4L),
  inner_parallel = FALSE,
  outer_parallel = TRUE
)
# Point estimates are deterministic, so the structural identities and the
# PAF-congruence sweep do not need the full MC depth: they run cheap and serial.
#
# n_pca is a FALLBACK: pif_confint ignores it whenever neff_consumption is supplied
# (aaf_unified.R:860-863), and pif2_build_pif_args always supplies it, so the gamma
# resample size is driven by the survey design, not by this number. It is set to the
# PAF's own n_pca anyway, so that IF a cell ever resolved neff_consumption to NULL the
# fallback would match the PAF instead of silently shrinking the resample.
pif2_inj_cheap_cfg <- list(n_sim = 500L, n_pca = as.integer(pif2_aaf_mc$n_pca),
                           n_cores = 1L, inner_parallel = FALSE, outer_parallel = FALSE)
pif2_inj_tol <- list(
  zero  = 1e-6,   # |PIF| under the no-change baseline
  mono  = 1e-6,   # monotone ladders / dominance slack
  ci    = 1e-9,   # pif_low <= pif <= pif_up
  ident = 1e-9,   # both(vol,1) == volume ; both(1,hed) == hed
  paf   = 0.02,   # my AAF vs the SAVED PAF point estimate (same tolerance as Phase 7)
  grid  = 1e-9    # my PIF vs the full grid's PIF for the same cell
)
pif2_message("[inj-test] logical processors detected: %s | workers: %s | bundle recorded: %s",
             parallel::detectCores(logical = TRUE),
             pif2_inj_cfg$n_cores, pif2_aaf_mc$n_cores)
# ---------------------------------------------------------------------------
# 2. The injury subset of the output spec, DERIVED from the AAF audit (never
#    hard-coded), so this test covers exactly the injury tables the PAF produced.
# ---------------------------------------------------------------------------
pif2_inj_spec   <- pif2_output_spec[pif2_output_spec$mode == "explicit", ]
pif2_inj_years  <- pif2_years
pif2_inj_groups <- 1:4
stopifnot(
  nrow(pif2_inj_spec) > 0L,
  all(pif2_inj_spec$uses_hed),                        # explicit mode implies an HED RR
  all(pif2_inj_spec$registry_scope == "injuries")
)
pif2_inj_expected_cells <- nrow(pif2_inj_spec) * nrow(pif2_scenario_grid) *
  length(pif2_inj_years) * length(pif2_inj_groups)
pif2_message("[inj-test] tables=%s | scenarios=%s | years=%s | groups=%s -> %s cells at n_sim=%s",
             nrow(pif2_inj_spec), nrow(pif2_scenario_grid),
             paste(pif2_inj_years, collapse = ","),
             paste(pif2_inj_groups, collapse = ","),
             pif2_inj_expected_cells, pif2_inj_cfg$n_sim)
pif2_message("[inj-test] tables: %s", paste(pif2_inj_spec$output_name, collapse = ", "))
# ---------------------------------------------------------------------------
# 3. Worker entry point.
#
# Defined at the TOP LEVEL on purpose, NOT inside the orchestrator. R serialises a
# closure together with its enclosing environment, and clusterApplyLB re-sends the
# function with EVERY task. A run_one() defined inside pif2_run_injury_cells() would
# therefore drag that function's whole frame -- including the 1,680-element `jobs`
# list and the exposure objects -- across the wire once per task. A top-level
# function has .GlobalEnv as its environment, which is serialised BY REFERENCE, so
# only the cell's own arguments actually travel.
#
# The job is re-stamped here (inside the worker) rather than in the master, so we
# never materialise a second full copy of the job list. Forcing use_parallel = FALSE
# and n_cores = 1 makes nested parallelism structurally impossible.
# ---------------------------------------------------------------------------
pif2_inj_run_one <- function(args) {
  args <- utils::modifyList(args, list(use_parallel = FALSE, n_cores = 1L))
  tryCatch(do.call(pif_confint, args), error = function(e) e)
}
# ---------------------------------------------------------------------------
# 4. Thin orchestrator over the injury cells. Column layout is IDENTICAL to
#    pif2_run_pif_grid()'s `results`, so the two are directly comparable and
#    bindable. A cell that cannot be resolved becomes an NA row carrying a
#    machine-readable reason -- never a spurious zero, never a crash.
# ---------------------------------------------------------------------------
pif2_inj_row <- function(spec_row, scenario_row, year, group, flags,
                         res = NULL, applicable = TRUE, reason = NA_character_,
                         n_sim = pif2_inj_cfg$n_sim) {
  data.frame(
    disease = spec_row$disease, sex = spec_row$sex, age_group = group, year = year,
    output_name = spec_row$output_name,
    scenario_id = scenario_row$scenario_id, scenario_label = scenario_row$scenario_label,
    engine_scenario = scenario_row$engine_scenario,
    avg_consumption_change_pct = scenario_row$volume_reduction_pct,
    hed_prevalence_change_pct  = scenario_row$hed_reduction_pct,
    pif     = if (is.null(res)) NA_real_    else res$point_estimate,
    pif_low = if (is.null(res)) NA_real_    else res$lower_ci,
    pif_up  = if (is.null(res)) NA_real_    else res$upper_ci,
    n_used  = if (is.null(res)) NA_integer_ else res$n_used,
    applicable = applicable, reason = reason,
    uses_volume_rr = flags$uses_volume_rr, uses_hed_rr = flags$uses_hed_rr,
    uses_fd_rr = flags$uses_fd_rr, uses_beta_vcov = flags$uses_beta_vcov,
    uses_weighted_gamma = flags$uses_weighted_gamma,
    uses_design_variance = flags$uses_design_variance,
    n_sim = n_sim, stringsAsFactors = FALSE)
}
pif2_run_injury_cells <- function(spec, scenarios, exposure, unc, mc, years, groups, run_cfg) {
  jobs <- list(); job_meta <- list(); rows <- list()
  # ---- 4a. Resolve every (table x scenario x year x group) cell up front -----
  for (si in seq_len(nrow(spec))) {
    spec_row <- spec[si, ]
    for (sc in seq_len(nrow(scenarios))) {
      scenario_row <- scenarios[sc, ]
      applic <- pif2_scenario_applicability(spec_row, scenario_row)
      for (year in years) for (group in groups) {

        record <- tryCatch(pif2_lookup_record(spec_row, group), error = function(e) NULL)
        flags  <- if (is.null(record)) {
          list(uses_volume_rr = NA, uses_hed_rr = spec_row$uses_hed, uses_fd_rr = NA,
               uses_beta_vcov = NA, uses_weighted_gamma = TRUE,
               uses_design_variance = is.function(unc$neff))
        } else pif2_uncertainty_flags(spec_row, record)

        if (!applic$applicable) {   # should never fire for injuries; kept for symmetry
          rows[[length(rows) + 1L]] <- pif2_inj_row(spec_row, scenario_row, year, group,
                                                    flags, NULL, FALSE, applic$reason)
          next
        }
        if (is.null(record)) {
          rows[[length(rows) + 1L]] <- pif2_inj_row(spec_row, scenario_row, year, group,
                                                    flags, NULL, TRUE, "record_lookup_failed")
          next
        }
        inputs <- pif2_resolve_cell_inputs(spec_row, record, year, group, exposure)
        if (!isTRUE(inputs$ok)) {
          rows[[length(rows) + 1L]] <- pif2_inj_row(spec_row, scenario_row, year, group, flags,
                                                    NULL, TRUE, paste0("invalid_inputs:", inputs$reason))
          next
        }
        if (identical(scenario_row$scenario_id, "baseline")) {
          rows[[length(rows) + 1L]] <- pif2_inj_row(
            spec_row, scenario_row, year, group, flags,
            list(point_estimate = 0, lower_ci = 0, upper_ci = 0, n_used = 0L),
            TRUE, NA_character_)
          next
        }
        jobs[[length(jobs) + 1L]] <- pif2_build_pif_args(
          spec_row, record, inputs, scenario_row, exposure, unc, mc, year, group, run_cfg)
        job_meta[[length(job_meta) + 1L]] <- list(spec_row = spec_row, scenario_row = scenario_row,
                                                  year = year, group = group, flags = flags)
      }
    }
  }
  # ---- 4b. Run the resolved jobs --------------------------------------------
  # Defensive by construction:
  #   * the number of workers was already clamped to the detected core count;
  #   * a cluster that fails to start degrades to a SERIAL run instead of aborting;
  #   * on.exit() reaps the cluster on normal exit, on error AND on user interrupt,
  #     so a Ctrl-C never strands orphan R processes;
  #   * BLAS threads are pinned to 1 per worker: a multithreaded BLAS (MKL/OpenBLAS)
  #     would otherwise silently multiply n_cores by its own thread pool;
  #   * clusterApplyLB load-balances, which matters on a P-core/E-core CPU where
  #     workers do NOT all run at the same speed;
  #   * an engine error in one cell is caught and becomes a reasoned NA row.
  cl <- NULL
  if (isTRUE(run_cfg$outer_parallel) && length(jobs) > 1L) {
    cl <- tryCatch(parallel::makeCluster(run_cfg$n_cores), error = function(e) {
      warning("makeCluster failed (", conditionMessage(e), "); falling back to a serial run.")
      NULL
    })
  }
  res_list <- if (!is.null(cl)) {
    on.exit(try(parallel::stopCluster(cl), silent = TRUE), add = TRUE)
    engine_file <- normalizePath(file.path(pif2_control_dir, "aaf_unified.R"), winslash = "/")
    parallel::clusterExport(cl, "engine_file", envir = environment())
    parallel::clusterEvalQ(cl, {
      Sys.setenv(OMP_NUM_THREADS = "1", OPENBLAS_NUM_THREADS = "1", MKL_NUM_THREADS = "1")
      source(engine_file)
      TRUE
    })
    parallel::clusterApplyLB(cl, jobs, pif2_inj_run_one)
  } else {
    lapply(jobs, pif2_inj_run_one)
  }
  # ---- 4c. Collect: engine failures become reasoned NA rows, not exceptions --
  for (i in seq_along(res_list)) {
    res <- res_list[[i]]; m <- job_meta[[i]]
    if (inherits(res, "error")) {
      rows[[length(rows) + 1L]] <- pif2_inj_row(m$spec_row, m$scenario_row, m$year, m$group, m$flags,
                                                NULL, TRUE, paste0("engine_error:", conditionMessage(res)))
      next
    }
    rows[[length(rows) + 1L]] <- pif2_inj_row(m$spec_row, m$scenario_row, m$year, m$group,
                                              m$flags, res, TRUE, NA_character_)
  }
  list(results = dplyr::bind_rows(rows), n_jobs = length(jobs))
}
pif2_inj_grid    <- pif2_run_injury_cells(
  spec = pif2_inj_spec, scenarios = pif2_scenario_grid,
  exposure = pif2_exposure_inputs, unc = pif2_aaf_uncertainty, mc = pif2_aaf_mc,
  years = pif2_inj_years, groups = pif2_inj_groups, run_cfg = pif2_inj_cfg)
pif2_inj_results <- pif2_inj_grid$results
# Provenance string, identical in spirit to the grid's uncertainty_components.
pif2_inj_results$uncertainty_components <- with(pif2_inj_results, paste(
  ifelse(uses_weighted_gamma, "weighted_gamma", NA),
  ifelse(uses_design_variance, "design_kish_cluster", NA),
  "dirichlet_prevalence",
  ifelse(uses_fd_rr, "fd_lognormal", NA),
  ifelse(uses_beta_vcov, "beta_mvrnorm", NA), sep = "+"))
pif2_inj_results$uncertainty_components <-
  gsub("(NA\\+)|(\\+NA)", "", pif2_inj_results$uncertainty_components)
pif2_message("[inj-test] MC jobs run: %s | deterministic baselines: %s | rows: %s | estimated: %s | failed: %s",
             pif2_inj_grid$n_jobs,
             sum(pif2_inj_results$scenario_id == "baseline" & pif2_inj_results$n_used == 0L),
             nrow(pif2_inj_results),
             sum(!is.na(pif2_inj_results$pif)), sum(is.na(pif2_inj_results$pif)))
# ===========================================================================
# 5. Assertions. `hard` checks are engine/pipeline invariants: any failure stops
#    the cell. `diagnostic` checks encode expectations that are empirically true
#    for a monotone-increasing injury RR but are NOT engine invariants; they warn
#    instead of stopping, so a surprising-but-legitimate result is surfaced rather
#    than silently swallowed or falsely fatal.
# ===========================================================================
pif2_inj_val <- list()
pif2_inj_check <- function(id, cond, note = "", severity = "hard") {
  pif2_inj_val[[length(pif2_inj_val) + 1L]] <<- data.frame(
    check = id, severity = severity, pass = isTRUE(cond), note = note,
    stringsAsFactors = FALSE)
  invisible(NULL)
}
pif2_inj_aeq <- function(a, b, tol = 1e-9) all(is.finite(a) & is.finite(b) & abs(a - b) <= tol)
.r  <- pif2_inj_results
.ok <- .r[.r$applicable & !is.na(.r$pif), ]
# ---- 5a. Coverage: the full cross-product, each cell exactly once ----------
.keys <- paste(.r$output_name, .r$scenario_id, .r$year, .r$age_group, sep = "|")
pif2_inj_check("coverage_full_crossproduct",
               nrow(.r) == pif2_inj_expected_cells && !anyDuplicated(.keys),
               sprintf("rows=%d expected=%d duplicates=%d",
                       nrow(.r), pif2_inj_expected_cells, anyDuplicated(.keys)))
pif2_inj_check("coverage_all_tables_years_groups",
               setequal(unique(.ok$output_name), pif2_inj_spec$output_name) &&
                 setequal(unique(.ok$year), pif2_inj_years) &&
                 setequal(unique(.ok$age_group), pif2_inj_groups) &&
                 setequal(unique(.ok$scenario_id), pif2_scenario_grid$scenario_id),
               sprintf("tables=%d years=%d groups=%d scenarios=%d",
                       dplyr::n_distinct(.ok$output_name), dplyr::n_distinct(.ok$year),
                       dplyr::n_distinct(.ok$age_group), dplyr::n_distinct(.ok$scenario_id)))
# ---- 5b. No engine errors, no unresolved inputs, everything applicable -----
.bad <- .r$reason[!is.na(.r$reason)]
pif2_inj_check("no_engine_or_input_failures", length(.bad) == 0L,
               if (length(.bad)) paste(utils::head(unique(.bad), 3), collapse = " ; ") else "none")
pif2_inj_check("all_scenarios_applicable_to_injuries",
               all(.r$applicable) && nrow(.ok) == pif2_inj_expected_cells,
               "injuries are HED-capable: volume, HED and combined all apply")
# ---- 5c. Uncertainty wiring: explicit HED RR + beta covariance, no FD RR ----
pif2_inj_check("flags_hed_rr_on",  all(.r$uses_hed_rr), "explicit HED RR component")
pif2_inj_check("flags_fd_rr_off",  all(!.r$uses_fd_rr), "RR_FD = 1 for injuries (fd_uncertainty FALSE)")
pif2_inj_check("flags_beta_vcov_on", all(.r$uses_beta_vcov), "joint mvrnorm draw of the binge betas")
pif2_inj_check("flags_design_variance_on", all(.r$uses_design_variance),
               "Kish n + cluster factor closures reused from the PAF")
# ---- 5d. Numerical validity and bounds -------------------------------------
pif2_inj_check("numerical_validity",
               all(is.finite(.ok$pif)) && all(is.finite(.ok$pif_low)) && all(is.finite(.ok$pif_up)), "")
pif2_inj_check("ci_ordering",
               all(.ok$pif_low <= .ok$pif + pif2_inj_tol$ci) &&
                 all(.ok$pif <= .ok$pif_up + pif2_inj_tol$ci), "pif_low <= pif <= pif_up")
.neg <- sum(.ok$pif < -1e-9)
pif2_inj_check("bounds_le_one",
               all(.ok$pif <= 1 + 1e-9) && all(.ok$pif_up <= 1 + 1e-9),
               sprintf("negative PIFs retained (not clipped): %d", .neg))
# ---- 5e. Baseline identity: shift = 1 -> PIF == 0 by construction -----------
.base <- .ok[.ok$scenario_id == "baseline", ]
pif2_inj_check("baseline_zero",
               nrow(.base) == nrow(pif2_inj_spec) * length(pif2_inj_years) * length(pif2_inj_groups) &&
                 all(abs(.base$pif) < pif2_inj_tol$zero) && all(.base$n_used == 0L),
               sprintf("n=%d max|pif|=%.2e", nrow(.base), max(abs(.base$pif))))
# ---- 5f. Monotone ladders, per (table, year, group) -------------------------
#      Injury RRs increase monotonically in exposure, so a deeper cut can never
#      avert a smaller fraction. Baseline (0%) is the first rung of the volume ladder.
pif2_inj_ladder_bad <- function(df, order_col) {
  d <- df[order(df$output_name, df$year, df$age_group, df[[order_col]]), ]
  k <- paste(d$output_name, d$year, d$age_group, sep = "|")
  bad <- 0L
  for (cell in unique(k)) {
    v <- d$pif[k == cell]
    if (length(v) >= 2L && min(diff(v)) < -pif2_inj_tol$mono) bad <- bad + 1L
  }
  bad
}
.vol_bad <- pif2_inj_ladder_bad(.ok[.ok$engine_scenario == "volume", ], "avg_consumption_change_pct")
.hed_bad <- pif2_inj_ladder_bad(.ok[.ok$engine_scenario == "hed", ],    "hed_prevalence_change_pct")
pif2_inj_check("monotone_volume_ladder", .vol_bad == 0L,
               sprintf("non-monotone cells over 0/10/20/30 pct: %d", .vol_bad))
pif2_inj_check("monotone_hed_ladder", .hed_bad == 0L,
               sprintf("non-monotone cells over 10/25/50 pct: %d", .hed_bad))
# ---- 5g. Combined scenarios dominate their own components -------------------
pif2_inj_pick <- function(sid) {
  s <- .ok[.ok$scenario_id == sid, ]
  stats::setNames(s$pif, paste(s$output_name, s$year, s$age_group, sep = "|"))
}
.v10 <- pif2_inj_pick("volume_reduction_10"); .v20 <- pif2_inj_pick("volume_reduction_20")
.v30 <- pif2_inj_pick("volume_reduction_30"); .h25 <- pif2_inj_pick("hed_reduction_25")
.h50 <- pif2_inj_pick("hed_reduction_50")
.c1  <- pif2_inj_pick("combined_v10_h25");    .c2  <- pif2_inj_pick("combined_v20_h50")
.c3  <- pif2_inj_pick("combined_v30_h50")
.cells <- names(.c1)
.dom <- c(
  all(.c1[.cells] >= pmax(.v10[.cells], .h25[.cells]) - pif2_inj_tol$mono),
  all(.c2[.cells] >= pmax(.v20[.cells], .h50[.cells]) - pif2_inj_tol$mono),
  all(.c3[.cells] >= pmax(.v30[.cells], .h50[.cells]) - pif2_inj_tol$mono))
pif2_inj_check("combined_dominates_components", all(.dom),
               sprintf("checked %d cells x 3 combined scenarios", length(.cells)))
# Sub-additivity: a combined PIF must not exceed the sum of its parts, or the two
# levers would be double-counting the same averted deaths. DIAGNOSTIC, not an
# engine invariant.
.excess <- max(c(.c1[.cells] - (.v10[.cells] + .h25[.cells]),
                 .c2[.cells] - (.v20[.cells] + .h50[.cells]),
                 .c3[.cells] - (.v30[.cells] + .h50[.cells])))
pif2_inj_check("combined_subadditive", .excess <= pif2_inj_tol$mono,
               sprintf("max(pif_comb - pif_vol - pif_hed) = %.3e", .excess),
               severity = "diagnostic")
# ---- 5h. Scenario isolation, on a real injury cell (explicit HED mode) -------
#      The "both" engine path is a strict superset: with one lever neutral it must
#      reproduce the single-lever path EXACTLY. Built through pif2_build_pif_args so
#      the same argument assembly the grid uses is what gets exercised.
.id_spec  <- pif2_inj_spec[pif2_inj_spec$output_name == "ri_male", ]
.id_year  <- as.integer(tail(pif2_inj_years, 1L)); .id_group <- 2L
.id_rec   <- pif2_lookup_record(.id_spec, .id_group)
.id_in    <- pif2_resolve_cell_inputs(.id_spec, .id_rec, .id_year, .id_group, pif2_exposure_inputs)
stopifnot(isTRUE(.id_in$ok))
pif2_inj_scn <- function(engine, sv, sh) data.frame(
  engine_scenario = engine, shift_vol = sv, shift_hed = sh, stringsAsFactors = FALSE)
pif2_inj_run1 <- function(engine, sv, sh) do.call(pif_confint, pif2_build_pif_args(
  .id_spec, .id_rec, .id_in, pif2_inj_scn(engine, sv, sh),
  pif2_exposure_inputs, pif2_aaf_uncertainty, pif2_aaf_mc,
  .id_year, .id_group, pif2_inj_cheap_cfg))
.iso_v  <- pif2_inj_run1("volume", 0.8, 1.0)$point_estimate
.iso_h  <- pif2_inj_run1("hed",    1.0, 0.8)$point_estimate
.iso_bv <- pif2_inj_run1("both",   0.8, 1.0)$point_estimate
.iso_bh <- pif2_inj_run1("both",   1.0, 0.8)$point_estimate
pif2_inj_check("isolation_volume_leaves_hed", pif2_inj_aeq(.iso_bv, .iso_v, pif2_inj_tol$ident),
               sprintf("both(0.8,1)=%.6f volume(0.8)=%.6f", .iso_bv, .iso_v))
pif2_inj_check("isolation_hed_leaves_volume", pif2_inj_aeq(.iso_bh, .iso_h, pif2_inj_tol$ident),
               sprintf("both(1,0.8)=%.6f hed(0.8)=%.6f", .iso_bh, .iso_h))
# ---- 5i. Reproducibility: identical args + fixed seed -> identical summaries -
.rep_a <- pif2_inj_run1("both", 0.9, 0.75); .rep_b <- pif2_inj_run1("both", 0.9, 0.75)
pif2_inj_check("reproducible_seed",
               pif2_inj_aeq(.rep_a$point_estimate, .rep_b$point_estimate, 1e-12) &&
                 pif2_inj_aeq(.rep_a$lower_ci, .rep_b$lower_ci, 1e-12) &&
                 pif2_inj_aeq(.rep_a$upper_ci, .rep_b$upper_ci, 1e-12), "")
# ---- 5j. PAF congruence over EVERY injury cell -------------------------------
#      Strip the scenario/shift arguments from the baseline call and the PIF engine
#      degenerates to the AAF engine. Its point estimate must reproduce the SAVED
#      PAF (family_bundles$injuries) for all tables x years x age groups. Serial on
#      the master by design: the cluster is already closed, and at n_sim = 500 the
#      whole sweep is a couple of minutes.
pif2_inj_paf_dev <- do.call(rbind, lapply(seq_len(nrow(pif2_inj_spec)), function(si) {
  spec_row <- pif2_inj_spec[si, ]
  saved    <- pif2_nested_bundle$family_bundles$injuries$raw_result$tables[[spec_row$output_name]]
  base_scn <- pif2_scenario_grid[pif2_scenario_grid$scenario_id == "baseline", ]
  do.call(rbind, lapply(pif2_inj_years, function(year) {
    do.call(rbind, lapply(pif2_inj_groups, function(group) {
      rec  <- pif2_lookup_record(spec_row, group)
      inp  <- pif2_resolve_cell_inputs(spec_row, rec, year, group, pif2_exposure_inputs)
      args <- pif2_build_pif_args(spec_row, rec, inp, base_scn, pif2_exposure_inputs,
                                  pif2_aaf_uncertainty, pif2_aaf_mc, year, group,
                                  pif2_inj_cheap_cfg)
      aaf_args  <- pif2_as_aaf_args(args)
      mine      <- do.call(aaf_confint, aaf_args)$point_estimate
      saved_val <- saved[saved$Year == year, paste0(spec_row$prefix, group, "_point")]
      data.frame(output_name = spec_row$output_name, year = year, age_group = group,
                 aaf_recomputed = mine, paf_saved = as.numeric(saved_val),
                 abs_dev = abs(mine - as.numeric(saved_val)), stringsAsFactors = FALSE)
    }))
  }))
}))
pif2_inj_check("paf_congruence_all_injury_cells",
               nrow(pif2_inj_paf_dev) == nrow(pif2_inj_spec) * length(pif2_inj_years) * length(pif2_inj_groups) &&
                 all(is.finite(pif2_inj_paf_dev$abs_dev)) &&
                 max(pif2_inj_paf_dev$abs_dev) <= pif2_inj_tol$paf,
               sprintf("n=%d max|AAF-savedPAF|=%.4f (tol %.2f)",
                       nrow(pif2_inj_paf_dev), max(pif2_inj_paf_dev$abs_dev), pif2_inj_tol$paf))
# ---- 5k. Cross-check against the full grid, when it has already been run -----
if (exists("pif2_pif_results", inherits = TRUE) &&
    all(pif2_inj_spec$output_name %in% pif2_pif_results$output_name)) {
  .g <- pif2_pif_results[pif2_pif_results$output_name %in% pif2_inj_spec$output_name, ]
  if (identical(as.integer(unique(.g$n_sim)), as.integer(pif2_inj_cfg$n_sim))) {
    .gk  <- paste(.g$output_name, .g$scenario_id, .g$year, .g$age_group, sep = "|")
    .mk  <- paste(.ok$output_name, .ok$scenario_id, .ok$year, .ok$age_group, sep = "|")
    .i   <- match(.mk, .gk)
    .dev <- max(abs(.ok$pif - .g$pif[.i]), na.rm = TRUE)
    pif2_inj_check("grid_cross_check", all(!is.na(.i)) && .dev <= pif2_inj_tol$grid,
                   sprintf("max|mine - grid| = %.2e over %d shared cells", .dev, sum(!is.na(.i))))
  } else {
    pif2_inj_check("grid_cross_check", TRUE,
                   "skipped: grid ran at a different n_sim", severity = "diagnostic")
  }
} else {
  pif2_inj_check("grid_cross_check", TRUE,
                 "skipped: pif2-run-grid not executed in this session", severity = "diagnostic")
}
# ===========================================================================
# 6. Report, persist, and fail loudly on any HARD check.
# ===========================================================================
pif2_inj_report <- dplyr::bind_rows(pif2_inj_val)
print(knitr::kable(pif2_inj_report,
                   caption = "Full injuries PIF test: engine and pipeline invariants"))
# Compact readout: PIF by scenario x sex, averaged over years and age groups.
pif2_inj_summary <- .ok |>
  dplyr::group_by(scenario_id, scenario_label, sex) |>
  dplyr::summarise(mean_pif = mean(pif), min_pif = min(pif), max_pif = max(pif),
                   n_cells = dplyr::n(), .groups = "drop") |>
  dplyr::arrange(scenario_id, sex)
print(knitr::kable(pif2_inj_summary, digits = 4,
                   caption = "Injuries PIF by scenario and sex (mean over years x age groups)"))
pif2_inj_stamp <- format(Sys.Date(), "%Y%m%d")
pif2_inj_results_path <- file.path(pif2_control_dir,
                                   paste0("pif2_injuries_fulltest_results_", pif2_inj_stamp, ".rds"))
pif2_inj_checks_path  <- file.path(pif2_control_dir,
                                   paste0("pif2_injuries_fulltest_checks_",  pif2_inj_stamp, ".rds"))
saveRDS(pif2_inj_results, pif2_inj_results_path)
saveRDS(list(report = pif2_inj_report, paf_congruence = pif2_inj_paf_dev,
             summary = pif2_inj_summary, config = pif2_inj_cfg), pif2_inj_checks_path)
pif2_message("[inj-test] saved: %s | %s",
             basename(pif2_inj_results_path), basename(pif2_inj_checks_path))
.hard_fail <- pif2_inj_report$check[!pif2_inj_report$pass & pif2_inj_report$severity == "hard"]
.diag_fail <- pif2_inj_report$check[!pif2_inj_report$pass & pif2_inj_report$severity == "diagnostic"]
pif2_message("[inj-test] %d/%d checks passed (%d hard failures, %d diagnostic).",
             sum(pif2_inj_report$pass), nrow(pif2_inj_report),
             length(.hard_fail), length(.diag_fail))
if (length(.diag_fail)) warning("Injuries full test DIAGNOSTIC flags: ",
                                paste(.diag_fail, collapse = ", "))
if (length(.hard_fail)) stop("Injuries full test FAILED: ",
                             paste(.hard_fail, collapse = ", "))

message(sprintf(
  "pif2-injuries-full-test elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[inj-test] logical processors detected: 32 | workers: 20 | bundle recorded: 12
[inj-test] tables=6 | scenarios=16 | years=2012,2014,2016,2018,2020,2022,2024 | groups=1,2,3,4 -> 2688 cells at n_sim=10000
[inj-test] tables: ri_fem, ri_male, injuries_fem, injuries_male, violence_fem, violence_male
[inj-test] MC jobs run: 2520 | deterministic baselines: 168 | rows: 2688 | estimated: 2688 | failed: 0
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[

~ 75 minutes


## YPLL bridge

This section uses a cached YPLL matrix when available. It joins the saved AAFs to YPLL to
obtain attributable YPLL, and multiplies the **total** YPLL by the PIF to estimate avoidable
YPLL (the PIF is a share of the total burden of the cause, not of its attributable part --
see the notes above; multiplying attributable YPLL by the PIF, as this bridge did before
2026-07-11, understates the result by a factor of the AAF).

**No YPLL cache is present in this working copy** (there is no `Mortalidad/Matrices` directory
under the expandPIF root; the saved outputs show one was read from the ACC1240138_private tree on
another machine), so `pif2_ypll_cache` resolves to `NULL`,
`pif2_avoidable_ypll` stays `NULL`, and this section is inert until a YPLL matrix is supplied.
The averted-deaths tables above do not depend on it.


In [22]:
#| label: pif2-ypll-bridge
#| results: "hold"

.t0 <- Sys.time()
pif2_normalise_gender <- function(x) {
  dplyr::recode(
    as.character(x),
    "Men" = "Hombre",
    "Women" = "Mujer",
    "male" = "Hombre",
    "female" = "Mujer",
    .default = as.character(x)
  )
}
# Standardise the long PIF results (schema: disease, sex, year, age_group,
# scenario_id, engine_scenario, pif, pif_low, pif_up) into the join schema.
pif2_standardise_pif_table <- function(pif_tbl) {
  out <- tibble::as_tibble(pif_tbl)
  if (!"gender" %in% names(out) && "sex" %in% names(out)) {
    out$gender <- pif2_normalise_gender(out$sex)
  } else {
    out$gender <- pif2_normalise_gender(out$gender)
  }
  required_columns <- c(
    "year", "gender", "age_group", "disease", "scenario_id",
    "engine_scenario", "pif", "pif_low", "pif_up"
  )
  missing_columns <- setdiff(required_columns, names(out))
  if (length(missing_columns) > 0) {
    stop("PIF table is missing column(s): ", paste(missing_columns, collapse = ", "))
  }
  dplyr::select(out, dplyr::all_of(required_columns), dplyr::everything())
}
pif2_build_attributable_ypll <- function(ypll_cache, aaf_long) {
  if (is.null(ypll_cache)) {
    stop("No cached YPLL object is available. Add a YPLL cache or run the mortality/YPLL builder upstream.")
  }
  aaf_join <- aaf_long |>
    dplyr::select(dplyr::all_of(c("year", "age_group", "gender", "disease", "point", "lower", "upper")))
  ypll_cache |>
    tibble::as_tibble() |>
    dplyr::ungroup() |>
    dplyr::mutate(gender = pif2_normalise_gender(.data$gender)) |>
    dplyr::filter(.data$year %in% unique(aaf_join$year)) |>
    dplyr::left_join(
      aaf_join,
      by = c("year", "age_group", "gender", "disease")
    ) |>
    dplyr::mutate(
      ypll_att = .data$ypll * .data$point,
      ypll_att_low = .data$ypll * .data$lower,
      ypll_att_up = .data$ypll * .data$upper
    )
}
# Avoidable YPLL = TOTAL YPLL x PIF, per scenario. Only applicable PIF rows are joined
# (non-applicable disease x scenario cells carry NA and are dropped).
#
# 2026-07-11 CORRECTION. This used to multiply ypll_att (= ypll x AAF) by the PIF. That is
# wrong: the engine defines the PIF against the SAME population risk R_obs as the AAF
# (aaf_unified.R: .aaf_core's den = num + 1 IS .aaf_pop_R's R_obs), so
#     AAF = 1 - 1/R_obs   and   PIF = 1 - R_cf/R_obs
# are both shares of the TOTAL burden of the cause. Multiplying the ATTRIBUTABLE YPLL by the
# PIF therefore understated the avoidable YPLL by a factor equal to the AAF. The correct
# multiplicand is the total YPLL. avoidable_ypll_att_conv keeps the old quantity so the two
# conventions can be compared, but it is NOT the estimate.
pif2_apply_pif_to_ypll <- function(attributable_ypll, pif_tbl) {
  pif_long <- pif2_standardise_pif_table(pif_tbl)
  pif_long <- pif_long[!is.na(pif_long$pif), , drop = FALSE]
  attributable_ypll |>
    dplyr::inner_join(
      pif_long,
      by = c("year", "gender", "age_group", "disease"),
      relationship = "many-to-many"
    ) |>
    dplyr::mutate(
      avoidable_ypll = .data$ypll * .data$pif,
      avoidable_ypll_low = .data$ypll * .data$pif_low,
      avoidable_ypll_up = .data$ypll * .data$pif_up,
      avoidable_ypll_att_conv = .data$ypll_att * .data$pif
    )
}
pif2_attributable_ypll <- tryCatch(
  pif2_build_attributable_ypll(pif2_ypll_cache, pif2_aaf_long),
  error = function(e) {
    pif2_message("[YPLL] Attributable YPLL not built: %s", conditionMessage(e))
    NULL
  }
)
# Bridge PIF -> avoidable YPLL when both a YPLL cache and PIF results exist.
pif2_avoidable_ypll <- tryCatch(
  if (!is.null(pif2_attributable_ypll) && exists("pif2_pif_results")) {
    pif2_apply_pif_to_ypll(pif2_attributable_ypll, pif2_pif_results)
  } else NULL,
  error = function(e) {
    pif2_message("[YPLL] Avoidable YPLL not built: %s", conditionMessage(e))
    NULL
  }
)
if (!is.null(pif2_attributable_ypll)) {
  pif2_message("[YPLL] Attributable YPLL rows: %s", nrow(pif2_attributable_ypll))
  pif2_message("[YPLL] Missing AAF joins: %s", sum(is.na(pif2_attributable_ypll$point)))
}
if (!is.null(pif2_avoidable_ypll)) {
  pif2_message("[YPLL] Avoidable YPLL rows (by scenario): %s", nrow(pif2_avoidable_ypll))
  pif2_message("[YPLL] Scenarios in avoidable YPLL: %s",
               paste(sort(unique(pif2_avoidable_ypll$scenario_id)), collapse = ", "))
}
message(sprintf(
  "pif2-ypll-bridge elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[YPLL] Attributable YPLL rows: 1188
[YPLL] Missing AAF joins: 0
[YPLL] Avoidable YPLL rows (by scenario): 8112
[YPLL] Scenarios in avoidable YPLL: baseline, combined_v10_h25, combined_v10_h25_rt, combined_v20_h50, combined_v20_h50_rt, combined_v30_h50, combined_v30_h50_rt, hed_reduction_10, hed_reduction_10_rt, hed_reduction_25, hed_reduction_25_rt, hed_reduction_50, hed_reduction_50_rt, volume_reduction_10, volume_reduction_20, volume_reduction_30
pif2-ypll-bridge elapsed minutes: 0.00


We had different life tables based on different sources (`__andres_control/build_ypll.R`). We found that the Human Mortality Databases (`[m|f]per_[1|5]x1.txt`) was the most updated source. We used the **cached YPLL called `YPLL_[last date].rds`**

We computed alcohol-attributable years of life lost using three distinct life-expectancy metrics, each loaded from an external source file and kept in separate columns. For crude YPLL consistent with prior Chilean injury work, we subtracted age at death from national period life expectancy at birth, e(0), using INE Chile base-2024 estimates (2012–2024). For a national YLL estimate, we instead used remaining life expectancy at exact age, e(x), from the Human Mortality Database (HMD) Chile period 1×1 life tables (1992–2024), which account for survival already achieved and therefore avoid the downward bias of the e(0)−age convention, particularly at ages 60 and older. As an external standard for cross-country comparability, we applied the GBD 2019 reference life table (TMRLT) interpolated to single years of age. We did not splice authorities or blend series; the source used for each metric is reported explicitly in the results, and a sensitivity analysis replaces INE e(0) with UN WPP 2024 estimates.

In [23]:
#| label: pif2-ypll-three-metrics
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# THE YPLL CACHE NOW CARRIES THREE METRICS. THEY ARE THREE DIFFERENT QUANTITIES.
#
# Mortalidad/Matrices/YPLL_20260714.rds (built by __andres_control/build_ypll.R):
#   yll_hmd   Chile HMD period life table. yll = e(x | year, sex, EXACT age).
#             The NATIONAL YLL. Primary.
#   yll_gbd   GBD 2019 reference life table (TMRLT). The INTERNATIONALLY COMPARABLE YLL,
#             and what Kilian et al. (Lancet Public Health 2025) use.
#   ypll_ref  Legacy reference-age YPLL, max(e0 - age, 0). The Ruiz-Tagle convention.
#             BIASED DOWNWARD outside injuries: it understates years lost by 9% 
#             at age 50 and 44% at age 65, because someone who has SURVIVED to age
#             x has a longer remaining expectancy than e(0) - x. Kept for continuity.
#   ypll      an ALIAS of yll_hmd, so the existing bridge (which joins on `ypll`) keeps
#             working unchanged. Said out loud here rather than left as a silent default.
#
# The bridge above passes the cache through without selecting columns, so all three
# survive the join -- but it only multiplies ONE of them by the PIF. This cell finishes
# the job. Never add these columns together.
#
# The LEGACY Mortalidad/Matrices/YPLL.rds is still on disk (kept by decision, not used).
# It is a DIFFERENT death set: pre-Shield ICD map, 3 injury causes only, band 4 open at
# 60+. The artifact picker prefers an embedded YYYYMMDD and the legacy has none, so it
# takes the new file -- but we ASSERT that here rather than trusting it.
# ---------------------------------------------------------------------------
if (is.null(pif2_ypll_cache)) {
  pif2_message("[YPLL-3] No YPLL cache loaded; section inert.")
} else {
  .p <- attr(pif2_ypll_cache, "path")
  pif2_message("[YPLL-3] Cache in use: %s", basename(.p))
  # If the picker regressed to the legacy file, STOP. Silently reporting avoidable YLL
  # for 3 injury causes as if it covered 23 would be a paper-level error.
  if (!grepl("YPLL_[0-9]{8}\\.rds$", basename(.p))) {
    stop("[YPLL-3] The artifact picker selected '", basename(.p), "', which is the LEGACY ",
         "YPLL (different death set, 3 injury causes, band 4 open at 60+). Expected a dated ",
         "YPLL_YYYYMMDD.rds. Do not use these numbers.")
  }
  .need <- c("ypll", "yll_hmd", "yll_gbd", "ypll_ref", "deaths")
  .miss <- setdiff(.need, names(pif2_ypll_cache))
  if (length(.miss)) {
    stop("[YPLL-3] The YPLL cache is missing columns: ", paste(.miss, collapse = ", "),
         ". Re-run __andres_control/build_ypll.R.")
  }
  stopifnot(isTRUE(all.equal(pif2_ypll_cache$ypll, pif2_ypll_cache$yll_hmd)))
  pif2_message("[YPLL-3] Cache OK: %d cells, %d diseases, years %d-%d, %d deaths.",
               nrow(pif2_ypll_cache), dplyr::n_distinct(pif2_ypll_cache$disease),
               min(pif2_ypll_cache$year), max(pif2_ypll_cache$year),
               sum(pif2_ypll_cache$deaths))
}
# ---- avoidable YLL under each metric -----------------------------------------
pif2_avoidable_yll3 <- if (!is.null(pif2_avoidable_ypll)) {
  pif2_avoidable_ypll |>
    dplyr::mutate(
      avoidable_yll_hmd  = .data$yll_hmd  * .data$pif,   # national     (== avoidable_ypll)
      avoidable_yll_gbd  = .data$yll_gbd  * .data$pif,   # comparable
      avoidable_ypll_ref = .data$ypll_ref * .data$pif    # legacy / JRT (see above)
    )
} else NULL
if (!is.null(pif2_avoidable_yll3)) {
  stopifnot(isTRUE(all.equal(pif2_avoidable_yll3$avoidable_yll_hmd,
                             pif2_avoidable_yll3$avoidable_ypll)))
  pif2_message("[YPLL-3] Avoidable-YLL rows: %d | diseases: %d | scenarios: %s",
               nrow(pif2_avoidable_yll3),
               dplyr::n_distinct(pif2_avoidable_yll3$disease),
               paste(sort(unique(pif2_avoidable_yll3$scenario_id)), collapse = ", "))
  # THE CHECK THAT MATTERS: the legacy cache reached 3 diseases. This one must reach 23.
  .nd <- dplyr::n_distinct(pif2_avoidable_yll3$disease)
  if (.nd < 20L) {
    stop("[YPLL-3] Avoidable YLL spans only ", .nd, " diseases. The legacy cache reached 3. ",
         "Expected ~23. The YPLL cache or the join is wrong.")
  }
  pif2_message("[YPLL-3] Coverage check PASSED: %d diseases (the legacy cache reached 3).", .nd)
}
message(sprintf("pif2-ypll-three-metrics elapsed minutes: %.2f", pif2_elapsed_min(.t0)))

[YPLL-3] Cache in use: YPLL_20260714.rds
[YPLL-3] Cache OK: 2197 cells, 23 diseases, years 2012-2024, 219338 deaths.
[YPLL-3] Avoidable-YLL rows: 8112 | diseases: 23 | scenarios: baseline, combined_v10_h25, combined_v10_h25_rt, combined_v20_h50, combined_v20_h50_rt, combined_v30_h50, combined_v30_h50_rt, hed_reduction_10, hed_reduction_10_rt, hed_reduction_25, hed_reduction_25_rt, hed_reduction_50, hed_reduction_50_rt, volume_reduction_10, volume_reduction_20, volume_reduction_30
[YPLL-3] Coverage check PASSED: 23 diseases (the legacy cache reached 3).
pif2-ypll-three-metrics elapsed minutes: 0.00


## Validation test harness (Phase 7)

Structural and grid-level checks over the computed PIF results and targeted single cells: baseline identity, HED/FD applicability, correct shift mapping, single application of the survey design (Kish n ÷ cluster factor, never doubled), joint covariance recovery, seed reproducibility, numerical validity, PIF ≤ 1 with negative values retained (not clipped), full disease coverage, **PAF congruence** (inputs reproduce the saved PAF), scenario isolation, and monotonicity only where the RR structure implies it. The cell stops loudly if any check fails.


In [24]:
#| label: pif2-assertion-test-harness
#| results: "hold"

.t0 <- Sys.time()
# ---------------------------------------------------------------------------
# Phase 7 validation harness. Grid-level tests run on pif2_pif_results (the demo
# run); structural tests run targeted single cells at small n_sim so they are fast
# and deterministic. Every check is recorded; the cell stops loudly if any fail.
# ---------------------------------------------------------------------------
pif2_val <- list()
pif2_check <- function(id, cond, note = "") {
  pif2_val[[length(pif2_val) + 1L]] <<- data.frame(
    check = id, pass = isTRUE(cond), note = note, stringsAsFactors = FALSE)
  invisible(NULL)
}
pif2_aeq <- function(a, b, tol = 1e-9) all(is.finite(a) & is.finite(b) & abs(a - b) <= tol)
.exposure <- pif2_exposure_inputs; .unc <- pif2_aaf_uncertainty; .mc <- pif2_aaf_mc
.cfg <- list(n_sim = 300L, n_pca = 200L, n_cores = 1L, inner_parallel = FALSE, outer_parallel = FALSE)
.res <- pif2_pif_results
.appl <- .res[.res$applicable & !is.na(.res$pif), ]
# 1) Identity / zero-change: baselines are deterministic rows, not MC jobs.
.base <- .res[.res$scenario_id == "baseline" & .res$applicable, ]
pif2_check("identity_baseline_zero",
           nrow(.base) > 0L && all(is.finite(.base$pif)) &&
             all(is.finite(.base$pif_low)) && all(is.finite(.base$pif_up)) &&
             all(abs(.base$pif) < 1e-12) && all(abs(.base$pif_low) < 1e-12) &&
             all(abs(.base$pif_up) < 1e-12) && all(.base$n_used == 0L),
           sprintf("%d deterministic baseline rows; max|pif|=%.1e",
                   nrow(.base), max(abs(.base$pif), na.rm = TRUE)))
# 3) Applicability: HED scenarios are non-applicable (NA) for volume-only diseases
.nohed <- pif2_output_spec$output_name[pif2_output_spec$mode == "nohed"]
.hn <- .res[.res$output_name %in% .nohed & .res$engine_scenario == "hed", ]
pif2_check("applicability_hed_on_nohed_NA",
           nrow(.hn) > 0 && all(!.hn$applicable) && all(is.na(.hn$pif)) && all(nzchar(na.omit(.hn$reason))),
           "HED scenarios NA + reason on volume-only causes")
# 3b) HED reduction actually lowers risk for HED-capable diseases (shift mapping)
.hc <- .appl[.appl$scenario_id == "hed_reduction_50", ]
pif2_check("hed_reduces_risk", nrow(.hc) > 0 && all(.hc$pif > 0),
           sprintf("min pif=%.4f over %d cells", suppressWarnings(min(.hc$pif)), nrow(.hc)))
# 4) FD applicability flags: injuries flagged uses_fd_rr = FALSE (RR_FD = 1)
pif2_check("fd_flag_injuries_false",
           isTRUE(all(!.res$uses_fd_rr[.res$output_name == "ri_male"])), "")
# 5) Age consistency + CV age-banding
pif2_check("age_support_subset", all(unique(.res$age_group) %in% 1:4), "")
.g1 <- pif2_lookup_record(pif2_output_spec[pif2_output_spec$output_name == "ihd_male", ], 1)
.g2 <- pif2_lookup_record(pif2_output_spec[pif2_output_spec$output_name == "ihd_male", ], 2)
pif2_check("cv_age_banding", .g1$adam_age_band == "15-34" && .g2$adam_age_band == "35-64", "")
# 7) Survey design applied ONCE (Kish n / cluster factor), not doubled
.np <- .aaf_resolve_cell(.unc$neff, 2022L, 2L, "male")
.df <- .aaf_resolve_cell(.unc$design_factor, 2022L, 2L, "male")
.ne <- .aaf_resolve_neff_eff(.np, .df)
.tbl <- environment(.unc$neff)$tbl
.row <- .tbl[.tbl$year == 2022 & .tbl$age_group == 2 & .tbl$sex == "male" & .tbl$variable == "abs", ]
pif2_check("design_applied_once",
           abs(.ne$abs - .row$neff_corr_engine) / .row$neff_corr_engine < 1e-4 &&
             abs(.ne$abs - .np / (.df$abs^2)) > 1,
           sprintf("engine=%.2f table=%.2f (double-apply=%.1f)", .ne$abs, .row$neff_corr_engine, .np / (.df$abs^2)))
# 8) Covariance: mvrnorm beta draws reproduce a supplied covariance
.covq <- matrix(c(0.010, 0.003, 0.003, 0.020), 2, 2); .bq <- c(0.5, -0.2)
set.seed(9); .draws <- t(replicate(30000, .aaf_draw_beta(.bq, .covq)))
pif2_check("covariance_recovered", max(abs(stats::cov(.draws) - .covq)) < 0.002,
           sprintf("max|emp-cov|=%.1e", max(abs(stats::cov(.draws) - .covq))))
# 9) Reproducibility: identical seed -> identical PIF summaries
.spec_liv <- pif2_output_spec[pif2_output_spec$output_name == "lican_male", ]
.rec_liv  <- pif2_lookup_record(.spec_liv, 2)
.in_liv   <- pif2_resolve_cell_inputs(.spec_liv, .rec_liv, 2022L, 2L, .exposure)
.args_liv <- pif2_build_pif_args(.spec_liv, .rec_liv, .in_liv,
                                 pif2_scenario_grid[pif2_scenario_grid$scenario_id == "volume_reduction_20", ],
                                 .exposure, .unc, .mc, 2022L, 2L, .cfg)
# One serial engine smoke test retains direct coverage of the shift = 1 identity.
.args_baseline <- pif2_build_pif_args(.spec_liv, .rec_liv, .in_liv,
                                      pif2_scenario_grid[pif2_scenario_grid$scenario_id == "baseline", ],
                                      .exposure, .unc, .mc, 2022L, 2L, .cfg)
.baseline_engine <- do.call(pif_confint, .args_baseline)
pif2_check("baseline_engine_identity",
           pif2_aeq(.baseline_engine$point_estimate, 0, 1e-12) &&
             pif2_aeq(.baseline_engine$lower_ci, 0, 1e-12) &&
             pif2_aeq(.baseline_engine$upper_ci, 0, 1e-12) &&
             identical(.baseline_engine$n_used, .cfg$n_sim),
           sprintf("n_used=%d", .baseline_engine$n_used))
.ra <- do.call(pif_confint, .args_liv); .rb <- do.call(pif_confint, .args_liv)
pif2_check("reproducible_seed",
           pif2_aeq(.ra$point_estimate, .rb$point_estimate, 1e-12) &&
             pif2_aeq(.ra$lower_ci, .rb$lower_ci, 1e-12) && pif2_aeq(.ra$upper_ci, .rb$upper_ci, 1e-12), "")
# 10) Numerical validity: finite point + CI for applicable results
pif2_check("numerical_validity",
           all(is.finite(.appl$pif)) && all(is.finite(.appl$pif_low)) && all(is.finite(.appl$pif_up)), "")
# 11) Bounds: PIF <= 1; negative PIFs retained (reported, not clipped)
.neg <- sum(.appl$pif < -1e-9)
pif2_check("bounds_le_one", all(.appl$pif <= 1 + 1e-9) && all(.appl$pif_up <= 1 + 1e-9),
           sprintf("negative PIFs retained (flagged): %d", .neg))
# 12) All diseases: every AAF output table estimated under >= 1 applicable scenario
pif2_check("all_diseases_covered",
           setequal(unique(.appl$output_name), pif2_aaf_audit$output_name),
           sprintf("covered=%d / audit=%d", dplyr::n_distinct(.appl$output_name), nrow(pif2_aaf_audit)))
# 13) PAF congruence: resolved inputs reproduce the SAVED PAF for a cell
.aaf_args <- pif2_as_aaf_args(.args_liv)
.aaf_res  <- do.call(aaf_confint, .aaf_args)
.saved <- pif2_nested_bundle$family_bundles$cancer$raw_result$tables$lican_male
.saved_val <- .saved[.saved$Year == 2022, "Male2_point"]
pif2_check("paf_congruence", pif2_aeq(.aaf_res$point_estimate, .saved_val, 0.02),
           sprintf("my AAF=%.4f saved PAF=%.4f", .aaf_res$point_estimate, .saved_val))
# 14) Scenario isolation via the proven both-superset identities (real IHD cell)
.spec_ihd <- pif2_output_spec[pif2_output_spec$output_name == "ihd_male", ]
.rec_ihd  <- pif2_lookup_record(.spec_ihd, 2)
.in_ihd   <- pif2_resolve_cell_inputs(.spec_ihd, .rec_ihd, 2022L, 2L, .exposure)
.mk <- function(scn, sv, sh) do.call(pif_point, list(
  x = .exposure$x_vals, rr_nhed = .rec_ihd$RRCurrent, beta = .rec_ihd$betaCurrent,
  p_abs = .in_ihd$p_abs, p_form = .in_ihd$p_form, rr_fd = exp(.rec_ihd$lnRRFormer),
  y_nhed = .aaf_gamma_density(.exposure$x_vals, .aaf_gamma_pars(.in_ihd$gamma)),
  p_hed = .in_ihd$p_hed, rr_hed = "cap",
  y_hed = .aaf_gamma_density(.exposure$x_vals, .aaf_gamma_pars(.in_ihd$gamma_hed)),
  scenario = scn, shift = if (scn == "hed") sh else sv, shift_hed = sh))
pif2_check("isolation_volume_leaves_hed", pif2_aeq(.mk("both", 0.8, 1.0), .mk("volume", 0.8, 1.0)), "both(vol,1)==volume")
pif2_check("isolation_hed_leaves_volume", pif2_aeq(.mk("both", 1.0, 0.8), .mk("hed", 1.0, 0.8)), "both(1,hed)==hed")
# 15) Monotonicity ONLY for a monotone-increasing-RR cause (liver cancer)
.liv <- .appl[.appl$output_name == "lican_male" & .appl$engine_scenario == "volume", ]
.liv <- .liv[.liv$age_group == min(.liv$age_group), ]
.liv <- .liv[order(.liv$avg_consumption_change_pct), ]
pif2_check("monotone_increasing_rr", nrow(.liv) >= 2 && all(diff(.liv$pif) >= -1e-6),
           "not asserted for IHD/IS J-curve causes")
# ---- report ----
pif2_validation_report <- dplyr::bind_rows(pif2_val)
pif2_n_fail <- sum(!pif2_validation_report$pass)
print(knitr::kable(pif2_validation_report, caption = "Phase 7 validation harness"))
pif2_message("[validation] %d/%d checks passed.",
             sum(pif2_validation_report$pass), nrow(pif2_validation_report))
if (pif2_n_fail > 0L) {
  message("Phase 7 validation FAILED (J-curve RRs): ",
       paste(pif2_validation_report$check[!pif2_validation_report$pass], collapse = ", "))
}
message(sprintf(
  "pif2-assertion-test-harness elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.


Table: Phase 7 validation harness

|check                         |pass  |note                                                         |
|:-----------------------------|:-----|:------------------------------------------------------------|
|identity_baseline_zero        |TRUE  |1260 deterministic baseline rows; max&#124;pif&#124;=0.0e+00 |
|applicability_hed_on_nohed_NA |TRUE  |HED scenarios NA + reason on volume-only causes              |
|hed_reduces_risk              |TRUE  |min pif=0.0017 over 280 cells                                |
|fd_flag_injuries_false        |TRUE  |                                                             |
|age_support_subset            |TRUE  |                                                             |
|cv_age_banding                |TRUE  |                                                             |
|design_applied_once           |TRUE  |engine=1

## Table 5 (PUC) IHD/IS sensitivity PIF

An **alternative RR source** for Ischaemic Heart Disease and Ischaemic Stroke: the Chile 2014 **Table 5 (PUC study)** sex-specific curves, instead of the WHO/Adam age-banded RRs used above. This reuses the *entire* PIF machinery (same orchestrator, scenario grid, survey-design closures, Monte Carlo settings) — only the RR curves change — and produces separate objects (`pif2_pif_results_table5`) plus a WHO-vs-Table 5 comparison. Table 5 RRs are **cardioprotective (J-curve)**, so volume-reduction PIFs can be negative (removing protection); these are retained and flagged, never clipped, and monotonicity is not imposed.

 It keeps every setting identical to the main PIF run — scenarios, exposure, Monte Carlo configuration, seed, and parallel architecture — but swaps the relative-risk source for ischaemic heart disease (IHD) and ischaemic stroke (IS) to the PUC (Table 5) RR registries.

In [25]:
#| label: pif2-table5-registries
#| results: "hold"

.t0 <- Sys.time()
# ---------------------------------------------------------------------------
# Table 5 (PUC, Chile 2014) sex-specific IHD/IS current-drinker RR curves -- an
# ALTERNATIVE RR source to the WHO/Adam age-banded RRs (GENERAL_ihd_RR_2018 /
# GENERAL_IS_RR_2018). These definitions are byte-identical to the canonical
# __andres_control/aaf_table5_ihd_is_experiment.R; replicated here so the notebook
# stays self-contained. Table 5 is NOT age-banded, so one RR per (disease, sex) is
# repeated across the three Adam age bands. The 'fact' column is stored as metadata
# and is NOT applied as a hidden x-scale (same decision as the Table 5 AAF
# experiment). IMPORTANT: these RRs are cardioprotective (J-curve) at low-to-moderate
# consumption, so under a VOLUME reduction the PIF can be NEGATIVE (removing
# protection). The engine retains negative PIFs; we flag them and do NOT clip or
# impose monotonicity on these causes.
# ---------------------------------------------------------------------------
# Table 5 ln(RR) forms (x = alcohol grams/day):
table5_rr_ihd_male   <- function(x, beta) exp(beta[[1L]] * sqrt(x) + beta[[2L]] * x^3)                 # B1*x^0.5 + B2*x^3
table5_rr_ihd_female <- function(x, beta) exp(beta[[1L]] * x + beta[[2L]] * x * log(x))                # B1*x + B2*x*ln(x)
table5_rr_is_male    <- function(x, beta) exp(beta[[1L]] * sqrt(x) + beta[[2L]] * sqrt(x) * log(x))    # B1*x^0.5 + B2*x^0.5*ln(x)
table5_rr_is_female  <- function(x, beta) exp(beta[[1L]] * sqrt(x) + beta[[2L]] * x)                   # B1*x^0.5 + B2*x
# Approximate log-normal former-drinker variance from a 95% CI.
table5_lognormal_var_from_ci <- function(lower, upper) ((log(upper) - log(lower)) / (2 * stats::qnorm(0.975)))^2
# Hard-coded Table 5 coefficients, SEs, former-drinker RRs (traceable to Table 5).
table5_rr_parameters <- function() data.frame(
  family = c("ihd", "ihd", "is", "is"),
  disease = c("Ischaemic Heart Disease", "Ischaemic Heart Disease", "Ischaemic Stroke", "Ischaemic Stroke"),
  sex = c("male", "female", "male", "female"),
  source_object = c("table5_ihd_male", "table5_ihd_female", "table5_is_male", "table5_is_female"),
  b1 = c(-0.046271, -0.052526, -0.141950, -0.249674), se_b1 = c(0.024037, 0.032510, 0.012866, 0.019163),
  b2 = c(0.000001, 0.014704, 0.039613, 0.037207),     se_b2 = c(0.000000, 0.007925, 0.001782, 0.000523),
  fact = c(1 / 3, 1 / 20, 1, 1), rr_former = c(1.25, 1.54, 0.97, 0.97),
  rr_former_lower = c(1.15, 1.17, 0.83, 0.83), rr_former_upper = c(1.36, 2.03, 1.14, 1.14),
  stringsAsFactors = FALSE)
# One RR record (compatible with compute_cv / the PIF cap mode). Diagonal beta
# covariance (independent betas, as supplied); former-drinker variance from the CI.
table5_make_record <- function(row, age_band) {
  rr_fun <- switch(paste(row$family, row$sex, sep = "_"),
                   ihd_male = table5_rr_ihd_male, ihd_female = table5_rr_ihd_female,
                   is_male = table5_rr_is_male,   is_female = table5_rr_is_female,
                   stop("Unsupported Table 5 RR row: ", row$family, " / ", row$sex))
  list(
    disease = row$disease, pipeline_disease = row$disease, rr_endpoint = row$disease,
    source_note = "Chile 2014 Table 5 (PUC) sex-specific RR; same RR across age bands (not age-banded); 'fact' not applied as x-scale.",
    pipeline_icd10 = if (identical(row$family, "ihd")) "I20-I25" else "G45-G46.8/I63/I65-I66/I67.2-I67.8/I69.3-I69.4",
    sex = row$sex, source_file = "Table 5, Chile 2014 (PUC); see aaf_table5_ihd_is_experiment.R",
    source_object = paste0(row$source_object, "_", gsub("[^0-9A-Za-z]+", "_", age_band)),
    adam_age_band = age_band, RRCurrent = rr_fun, betaCurrent = c(row$b1, row$b2),
    covBetaCurrent = diag(c(row$se_b1^2, row$se_b2^2), 2L),
    lnRRFormer = log(row$rr_former),
    varLnRRFormer = table5_lognormal_var_from_ci(row$rr_former_lower, row$rr_former_upper),
    table5_fact = row$fact, table5_rr_former_ci = c(row$rr_former_lower, row$rr_former_upper))
}
# Registry = 2 sexes x 3 age bands (same RR repeated per band, since Table 5 is not
# age-banded), so compute_cv / the cap-mode lookup work exactly as for WHO IHD/IS.
table5_make_registry <- function(family = c("ihd", "is")) {
  family <- match.arg(family)
  pars <- table5_rr_parameters(); pars <- pars[pars$family == family, , drop = FALSE]
  records <- list()
  for (i in seq_len(nrow(pars))) for (band in c("15-34", "35-64", "65+")) {
    rec <- table5_make_record(pars[i, , drop = FALSE], band)
    records[[paste(rec$sex, rec$source_object, sep = "::")]] <- rec
  }
  class(records) <- c("adam_rr_registry", "list")
  records
}
# Build, validate, and register the Table 5 registries as new scopes so the existing
# pif2_lookup_record()/pif2_run_pif_grid() machinery can use them unchanged. This is
# ADDITIVE: it does not touch the WHO-based pif2_output_spec or pif2_pif_results.
table5_ihd_registry <- table5_make_registry("ihd")
table5_is_registry  <- table5_make_registry("is")
validate_adam_rr_registry(table5_ihd_registry)
validate_adam_rr_registry(table5_is_registry)
pif2_rr_registries$table5_ihd <- table5_ihd_registry
pif2_rr_registries$table5_is  <- table5_is_registry
# Table 5 output spec (4 cap-mode tables), same schema pif2_run_pif_grid expects,
# plus an rr_source tag so results are never confused with the WHO run.
pif2_table5_output_spec <- tibble::tibble(
  output_name    = c("ihd_female", "ihd_male", "is_female", "is_male"),
  disease        = c("Ischaemic Heart Disease", "Ischaemic Heart Disease", "Ischaemic Stroke", "Ischaemic Stroke"),
  sex            = c("female", "male", "female", "male"),
  source_object  = c("table5_ihd_female", "table5_ihd_male", "table5_is_female", "table5_is_male"),
  mode           = "cap", uses_hed = TRUE, age_banded = TRUE, fd_uncertainty = TRUE,
  registry_scope = c("table5_ihd", "table5_ihd", "table5_is", "table5_is"),
  prefix         = c("Fem", "Male", "Fem", "Male"), rr_source = "table5_puc")
pif2_message("[table5] Registered scopes: table5_ihd (%d recs), table5_is (%d recs).",
             length(table5_ihd_registry), length(table5_is_registry))
pif2_message("[table5] Output spec: %s tables (IHD/IS x sex, cap mode).", nrow(pif2_table5_output_spec))
pif2_message("[table5] fact column (stored, NOT applied): %s",
             paste(sprintf("%s=%.4f", table5_rr_parameters()$source_object, table5_rr_parameters()$fact), collapse = ", "))
message(sprintf("pif2-table5-registries elapsed minutes: %.2f", pif2_elapsed_min(.t0)))

[table5] Registered scopes: table5_ihd (6 recs), table5_is (6 recs).
[table5] Output spec: 4 tables (IHD/IS x sex, cap mode).
[table5] fact column (stored, NOT applied): table5_ihd_male=0.3333, table5_ihd_female=0.0500, table5_is_male=1.0000, table5_is_female=1.0000
pif2-table5-registries elapsed minutes: 0.00


### Run + compare (WHO vs Table 5) + validate

We ran the same Monte Carlo setting, with same scenarios, exposures, years, age groups, and uncertainties of the main run. The only thing that changes is the RR source for Ischaemic Heart Disease and Ischaemic Stroke. Then, we calculated the differences in the point estimates. Also the chunk makes minimal checks (baseline scenario close to 0, having the expected IHD/IS tables, available use,  HED and combined scenarios, PIFs below 1 and finite values, congruence between PIF and PAF).

In [26]:
#| label: pif2-table5-run
#| results: "hold"
.t0 <- Sys.time()
# ============================================================================
# 1. Load the latest saved Table 5 AAF artifact.
#
# This artifact is used only for AAF/PAF validation. It is not a substitute for
# the PIF simulations because each PIF requires both the observed and
# counterfactual risks under the same Monte Carlo draw.
# ============================================================================
aaf_table5_result <- pif2_read_latest_artifact(
  directory = pif2_control_dir,
  stem = "aaf_table5_result",
  extension = "rds"
)
pif2_table5_aaf_path <- attr(aaf_table5_result, "path")
pif2_table5_required_aaf_components <- c(
  "metadata",
  "config",
  "raw",
  "by_age_scope"
)
pif2_table5_missing_aaf_components <- setdiff(
  pif2_table5_required_aaf_components,
  names(aaf_table5_result)
)
if (length(pif2_table5_missing_aaf_components) > 0L) {
  stop(
    "The latest Table 5 AAF artifact is missing component(s): ",
    paste(pif2_table5_missing_aaf_components, collapse = ", ")
  )
}
pif2_table5_aaf_long <-
  aaf_table5_result$by_age_scope[["15_64"]]$standard_tables$long
if (is.null(pif2_table5_aaf_long)) {
  stop(
    "The latest Table 5 AAF artifact does not contain ",
    "by_age_scope[['15_64']]$standard_tables$long."
  )
}
pif2_table5_required_aaf_columns <- c(
  "year",
  "age_group",
  "gender",
  "disease",
  "point",
  "lower",
  "upper"
)
pif2_table5_missing_aaf_columns <- setdiff(
  pif2_table5_required_aaf_columns,
  names(pif2_table5_aaf_long)
)
if (length(pif2_table5_missing_aaf_columns) > 0L) {
  stop(
    "The cached Table 5 AAF long table is missing column(s): ",
    paste(pif2_table5_missing_aaf_columns, collapse = ", ")
  )
}
pif2_message(
  "[table5] Loaded cached AAF artifact: %s",
  basename(pif2_table5_aaf_path)
)
# ============================================================================
# 2. Require the complete HED-exit scenario grid.
#
# The pif2-hed-exit-wiring chunk must have run before this chunk. The resulting
# grid contains:
#   - 10 conservative scenarios, lambda = 0;
#   - 6 Ruiz-Tagle twins, lambda = 1.
#
# Baseline and volume-only scenarios are not duplicated for lambda = 1 because
# nobody exits HED in those scenarios and lambda would be a mathematical no-op.
# ============================================================================
pif2_table5_required_scenario_columns <- c(
  "scenario_id",
  "scenario_label",
  "scenario_family",
  "engine_scenario",
  "shift_vol",
  "shift_hed",
  "requires_hed",
  "hed_exit_mix",
  "hed_exit_shift",
  "exit_rule"
)
pif2_table5_missing_scenario_columns <- setdiff(
  pif2_table5_required_scenario_columns,
  names(pif2_scenario_grid)
)
if (length(pif2_table5_missing_scenario_columns) > 0L) {
  stop(
    "The Table 5 run requires the completed HED-exit scenario grid. ",
    "Run pif2-hed-exit-wiring first. Missing column(s): ",
    paste(pif2_table5_missing_scenario_columns, collapse = ", ")
  )
}
pif2_table5_lambda_values <- sort(
  unique(pif2_scenario_grid$hed_exit_mix)
)
if (!identical(pif2_table5_lambda_values, c(0, 1))) {
  stop(
    "Expected hed_exit_mix values lambda = 0 and lambda = 1; found: ",
    paste(pif2_table5_lambda_values, collapse = ", ")
  )
}
# ============================================================================
# 3. Load the latest main PIF artifact when it is not already in memory.
#
# This gives us an executed reference run against which the Table 5 run
# signature can be checked: scenarios, years, age groups, and n_sim.
# ============================================================================
if (!exists("pif2_pif_results") || is.null(pif2_pif_results)) {
  pif2_pif_results <- pif2_read_latest_artifact(
    directory = pif2_control_dir,
    stem = paste0("pif2_pif_results_", pif2_pif_run_mode),
    extension = "rds"
  )
  pif2_message(
    "[table5] Loaded main PIF reference artifact: %s",
    basename(attr(pif2_pif_results, "path"))
  )
}
pif2_main_cv_results <- pif2_pif_results[
  pif2_pif_results$disease %in% c(
    "Ischaemic Heart Disease",
    "Ischaemic Stroke"
  ),
  ,
  drop = FALSE
]
pif2_main_scenarios <- sort(
  unique(pif2_main_cv_results$scenario_id)
)
pif2_table5_scenarios <- sort(
  unique(pif2_scenario_grid$scenario_id)
)
if (!identical(pif2_main_scenarios, pif2_table5_scenarios)) {
  stop(
    "The current scenario grid does not match the executed main PIF run. ",
    "Only in main: ",
    paste(setdiff(pif2_main_scenarios, pif2_table5_scenarios), collapse = ", "),
    ". Only in current grid: ",
    paste(setdiff(pif2_table5_scenarios, pif2_main_scenarios), collapse = ", "),
    "."
  )
}
pif2_main_n_sim <- unique(pif2_main_cv_results$n_sim)
if (
  length(pif2_main_n_sim) != 1L ||
    as.integer(pif2_main_n_sim) != as.integer(pif2_run_cfg$n_sim)
) {
  stop(
    "The main PIF artifact and pif2_run_cfg have different n_sim values. ",
    "Main artifact: ",
    paste(pif2_main_n_sim, collapse = ", "),
    "; current configuration: ",
    pif2_run_cfg$n_sim,
    "."
  )
}
# ============================================================================
# 4. Run Table 5 using the exact main PIF configuration.
#
# This intentionally reuses:
#   - pif2_run_pif_grid();
#   - pif2_scenario_grid, including lambda = 0 and lambda = 1;
#   - pif2_run_cfg;
#   - the same exposure inputs;
#   - the same uncertainty closures;
#   - the same seed;
#   - outer PSOCK parallelization with load balancing.
#
# The only substantive change is pif2_table5_output_spec, which points the IHD
# and ischaemic-stroke cells to the Table 5 PUC RR registries.
# ============================================================================
pif2_table5_run_cfg <- pif2_run_cfg
if (
  !isTRUE(pif2_table5_run_cfg$outer_parallel) ||
    isTRUE(pif2_table5_run_cfg$inner_parallel)
) {
  stop(
    "The Table 5 run must use the same parallel architecture as the main run: ",
    "outer_parallel = TRUE and inner_parallel = FALSE."
  )
}
pif2_message(
  paste0(
    "[table5] Starting full Table 5 PIF run: ",
    "%d scenarios; %d years; %d age groups; ",
    "n_sim=%d; n_pca=%d; cores=%d; outer parallel=%s."
  ),
  nrow(pif2_scenario_grid),
  length(pif2_table5_run_cfg$years),
  length(pif2_table5_run_cfg$groups),
  pif2_table5_run_cfg$n_sim,
  pif2_table5_run_cfg$n_pca,
  pif2_table5_run_cfg$n_cores,
  pif2_table5_run_cfg$outer_parallel
)
pif2_table5_grid <- pif2_run_pif_grid(
  spec = pif2_table5_output_spec,
  scenarios = pif2_scenario_grid,
  exposure = pif2_exposure_inputs,
  unc = pif2_aaf_uncertainty,
  mc = pif2_aaf_mc,
  years = pif2_table5_run_cfg$years,
  groups = pif2_table5_run_cfg$groups,
  run_cfg = pif2_table5_run_cfg
)
pif2_pif_results_table5 <- pif2_table5_grid$results
pif2_pif_audit_table5 <- pif2_table5_grid$audit
# ============================================================================
# 5. Attach scenario and RR provenance.
# ============================================================================
pif2_table5_scenario_metadata <- pif2_scenario_grid |>
  dplyr::select(
    .data$scenario_id,
    .data$hed_exit_mix,
    .data$hed_exit_shift,
    .data$exit_rule,
    .data$scale
  )
pif2_pif_results_table5 <- pif2_pif_results_table5 |>
  dplyr::left_join(
    pif2_table5_scenario_metadata,
    by = "scenario_id",
    relationship = "many-to-one"
  ) |>
  dplyr::mutate(rr_source = "table5_puc")
pif2_pif_audit_table5 <- pif2_pif_audit_table5 |>
  dplyr::left_join(
    pif2_table5_scenario_metadata,
    by = "scenario_id",
    relationship = "many-to-one"
  ) |>
  dplyr::mutate(rr_source = "table5_puc")
# Attach the same uncertainty provenance used by the main PIF result.
pif2_pif_results_table5$uncertainty_components <- with(
  pif2_pif_results_table5,
  paste(
    ifelse(uses_weighted_gamma, "weighted_gamma", NA),
    ifelse(uses_design_variance, "design_kish_cluster", NA),
    "dirichlet_prevalence",
    ifelse(uses_fd_rr, "fd_lognormal", NA),
    ifelse(uses_beta_vcov, "beta_mvrnorm", NA),
    sep = "+"
  )
)
pif2_pif_results_table5$uncertainty_components <- gsub(
  "(NA\\+)|(\\+NA)",
  "",
  pif2_pif_results_table5$uncertainty_components
)
# The implied consumption change depends on the exposure distributions and exit
# rule, not on the RR source. Therefore, the values computed for the main PIF
# workflow are also valid for the Table 5 RR sensitivity analysis.
if (exists("pif2_implied_vol") && !is.null(pif2_implied_vol)) {
  pif2_table5_implied_vol <- pif2_implied_vol |>
    dplyr::select(
      .data$output_name,
      .data$sex,
      .data$year,
      .data$age_group,
      .data$scenario_id,
      .data$policy_vol_lever_pct,
      .data$implied_vol_change_pct
    )
  pif2_pif_results_table5 <- pif2_pif_results_table5 |>
    dplyr::left_join(
      pif2_table5_implied_vol,
      by = c(
        "output_name",
        "sex",
        "year",
        "age_group",
        "scenario_id"
      ),
      relationship = "many-to-one"
    )
}
# ============================================================================
# 6. Save both results and audit, following the main-run convention.
# ============================================================================
pif2_table5_stamp <- format(Sys.Date(), "%Y%m%d")
pif2_table5_results_path <- file.path(
  pif2_control_dir,
  paste0(
    "pif2_pif_results_table5_",
    pif2_pif_run_mode,
    "_",
    pif2_table5_stamp,
    ".rds"
  )
)
pif2_table5_audit_path <- file.path(
  pif2_control_dir,
  paste0(
    "pif2_pif_audit_table5_",
    pif2_pif_run_mode,
    "_",
    pif2_table5_stamp,
    ".rds"
  )
)
base::saveRDS(
  pif2_pif_results_table5,
  pif2_table5_results_path
)
base::saveRDS(
  pif2_pif_audit_table5,
  pif2_table5_audit_path
)
# ============================================================================
# 7. WHO/Adam versus Table 5 comparison.
# ============================================================================
pif2_cv_who <- pif2_main_cv_results[
  pif2_main_cv_results$applicable &
    !is.na(pif2_main_cv_results$pif),
  c(
    "disease",
    "sex",
    "age_group",
    "year",
    "scenario_id",
    "pif",
    "pif_low",
    "pif_up"
  )
]
pif2_cv_table5 <- pif2_pif_results_table5[
  pif2_pif_results_table5$applicable &
    !is.na(pif2_pif_results_table5$pif),
  c(
    "disease",
    "sex",
    "age_group",
    "year",
    "scenario_id",
    "pif",
    "pif_low",
    "pif_up"
  )
]
pif2_cv_compare <- pif2_cv_who |>
  dplyr::inner_join(
    pif2_cv_table5,
    by = c(
      "disease",
      "sex",
      "age_group",
      "year",
      "scenario_id"
    ),
    suffix = c("_who", "_table5"),
    relationship = "one-to-one"
  ) |>
  dplyr::left_join(
    pif2_table5_scenario_metadata,
    by = "scenario_id",
    relationship = "many-to-one"
  ) |>
  dplyr::mutate(
    pif_diff = .data$pif_table5 - .data$pif_who
  )
# ============================================================================
# 8. Validation.
#
# The previous expensive aaf_confint() and compute_cv_aaf_from_registry() calls
# are removed. Validation now reads the saved AAF artifact.
# ============================================================================
t5_val <- list()
t5_check <- function(id, cond, note = "") {
  t5_val[[length(t5_val) + 1L]] <<- data.frame(
    check = id,
    pass = isTRUE(cond),
    note = note,
    stringsAsFactors = FALSE
  )
  invisible(NULL)
}
.t5_appl <- pif2_pif_results_table5[
  pif2_pif_results_table5$applicable &
    !is.na(pif2_pif_results_table5$pif),
  ,
  drop = FALSE
]
.t5_base <- pif2_pif_results_table5[
  pif2_pif_results_table5$scenario_id == "baseline" &
    pif2_pif_results_table5$applicable,
  ,
  drop = FALSE
]
.t5_expected_rows <-
  nrow(pif2_table5_output_spec) *
  nrow(pif2_scenario_grid) *
  length(pif2_table5_run_cfg$years) *
  length(pif2_table5_run_cfg$groups)
t5_check(
  "identity_baseline_zero",
  nrow(.t5_base) > 0L &&
    all(abs(.t5_base$pif) < 1e-6, na.rm = TRUE)
)
t5_check(
  "all_4_cv_tables_covered",
  setequal(
    unique(.t5_appl$output_name),
    pif2_table5_output_spec$output_name
  )
)
t5_check(
  "same_scenarios_as_main",
  setequal(
    unique(.t5_appl$scenario_id),
    unique(pif2_main_cv_results$scenario_id)
  ),
  sprintf(
    "Table 5=%d scenarios; main=%d scenarios",
    dplyr::n_distinct(.t5_appl$scenario_id),
    dplyr::n_distinct(pif2_main_cv_results$scenario_id)
  )
)
t5_check(
  "lambda_0_and_1_computed",
  all(c(0, 1) %in% unique(.t5_appl$hed_exit_mix)),
  paste(
    "lambda values:",
    paste(sort(unique(.t5_appl$hed_exit_mix)), collapse = ", ")
  )
)
t5_check(
  "same_n_sim_as_main",
  length(unique(.t5_appl$n_sim)) == 1L &&
    unique(.t5_appl$n_sim) == pif2_main_n_sim,
  sprintf("n_sim=%d", unique(.t5_appl$n_sim))
)
t5_check(
  "complete_result_grid",
  nrow(pif2_pif_results_table5) == .t5_expected_rows,
  sprintf(
    "observed=%d; expected=%d",
    nrow(pif2_pif_results_table5),
    .t5_expected_rows
  )
)
t5_check(
  "hed_and_combined_computed",
  any(.t5_appl$engine_scenario == "hed") &&
    any(.t5_appl$engine_scenario == "both")
)
t5_check(
  "bounds_le_one",
  all(.t5_appl$pif <= 1 + 1e-9) &&
    all(.t5_appl$pif_up <= 1 + 1e-9)
)
t5_check(
  "numerical_validity",
  all(is.finite(.t5_appl$pif)) &&
    all(is.finite(.t5_appl$pif_low)) &&
    all(is.finite(.t5_appl$pif_up))
)
t5_check(
  "cached_aaf_numerical_validity",
  nrow(pif2_table5_aaf_long) > 0L &&
    all(is.finite(pif2_table5_aaf_long$point)) &&
    all(is.finite(pif2_table5_aaf_long$lower)) &&
    all(is.finite(pif2_table5_aaf_long$upper)) &&
    all(pif2_table5_aaf_long$lower <= pif2_table5_aaf_long$point) &&
    all(pif2_table5_aaf_long$point <= pif2_table5_aaf_long$upper),
  basename(pif2_table5_aaf_path)
)
# Negative PIFs are valid for the cardioprotective IHD/IS curves. Do not assert
# that lambda = 1 must increase the PIF: the J-shaped curves make its direction
# genuinely ambiguous.
.t5_neg <- sum(.t5_appl$pif < -1e-9)
pif2_table5_validation <- dplyr::bind_rows(t5_val)
pif2_table5_n_fail <- sum(!pif2_table5_validation$pass)
print(
  knitr::kable(
    pif2_table5_validation,
    caption = "Table 5 PUC IHD/IS PIF validation"
  )
)
print(
  knitr::kable(
    utils::head(
      dplyr::arrange(
        pif2_cv_compare,
        .data$disease,
        .data$sex,
        .data$scenario_id
      ),
      16L
    ),
    digits = 4,
    caption = paste(
      "WHO/Adam versus Table 5 PUC PIF:",
      "same cells, scenarios, and Monte Carlo configuration"
    )
  )
)
pif2_message(
  paste0(
    "[table5] %d/%d checks passed | ",
    "scenarios=%d | lambda values=%s | ",
    "negative PIFs retained=%d | results=%s | audit=%s"
  ),
  sum(pif2_table5_validation$pass),
  nrow(pif2_table5_validation),
  dplyr::n_distinct(.t5_appl$scenario_id),
  paste(sort(unique(.t5_appl$hed_exit_mix)), collapse = ","),
  .t5_neg,
  basename(pif2_table5_results_path),
  basename(pif2_table5_audit_path)
)
if (pif2_table5_n_fail > 0L) {
  stop(
    "Table 5 PIF validation failed: ",
    paste(
      pif2_table5_validation$check[
        !pif2_table5_validation$pass
      ],
      collapse = ", "
    )
  )
}
message(
  sprintf(
    "pif2-table5-run elapsed minutes: %.2f",
    pif2_elapsed_min(.t0)
  )
)

[latest-file] Directory: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control
[latest-file] Pattern: ^aaf_table5_result_\d{8}\.rds$
[latest-file] Selection rule: embedded YYYYMMDD date, then modified time.
[latest-file] Candidate count: 1
[latest-file] Selected: aaf_table5_result_20260715.rds
[read-artifact] Reading: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control/aaf_table5_result_20260715.rds
[table5] Loaded cached AAF artifact: aaf_table5_result_20260715.rds
[table5] Starting full Table 5 PIF run: 16 scenarios; 7 years; 4 age groups; n_sim=10000; n_pca=1000; cores=12; outer parallel=TRUE.


Warning messages:
1: Use of .data in tidyselect expressions was deprecated in tidyselect 1.2.0.
ℹ Please use `"scenario_id"` instead of `.data$scenario_id`
This warning is displayed once per session.
Call ]8;;x-r-run:lifecycle::last_lifecycle_warnings()lifecycle::last_lifecycle_warnings()]8;; to see where this warning was
generated. 
2: Use of .data in tidyselect expressions was deprecated in tidyselect 1.2.0.
ℹ Please use `"hed_exit_mix"` instead of `.data$hed_exit_mix`
This warning is displayed once per session.
Call ]8;;x-r-run:lifecycle::last_lifecycle_warnings()lifecycle::last_lifecycle_warnings()]8;; to see where this warning was
generated. 
3: Use of .data in tidyselect expressions was deprecated in tidyselect 1.2.0.
ℹ Please use `"hed_exit_shift"` instead of `.data$hed_exit_shift`
This warning is displayed once per session.
Call ]8;;x-r-run:lifecycle::last_lifecycle_warnings()lifecycle::last_lifecycle_warnings()]8;; to see where this warning was
generated. 
4: Use o



Table: Table 5 PUC IHD/IS PIF validation

|check                         |pass |note                                    |
|:-----------------------------|:----|:---------------------------------------|
|identity_baseline_zero        |TRUE |                                        |
|all_4_cv_tables_covered       |TRUE |                                        |
|same_scenarios_as_main        |TRUE |Table 5=16 scenarios; main=16 scenarios |
|lambda_0_and_1_computed       |TRUE |lambda values: 0, 1                     |
|same_n_sim_as_main            |TRUE |n_sim=10000                             |
|complete_result_grid          |TRUE |observed=1792; expected=1792            |
|hed_and_combined_computed     |TRUE |                                        |
|bounds_le_one                 |TRUE |                                        |
|numerical_validity            |TRUE |                                        |
|cached_aaf_numerical_validity |TRUE |aaf_table5_result_20260715.rds        

~ 51 minutes

In [27]:
#| label: pif2-table5-paf-congruence-extra
#| results: "hold"

.t0 <- Sys.time()
# run codex resume 019f664d-4158-7f83-8913-b2c03297e01c
# ---------------------------------------------------------------------------
# ADDITIONAL PAF-CONGRUENCE CHECKS FOR THE TABLE 5 (PUC) IHD/IS REGISTRIES
#
# WHAT THIS CHUNK DOES, AND WHY IT IS CHEAP
#   It does NOT re-run pif2_run_pif_grid() (the ~51-minute Table 5 grid), and it
#   does not touch pif2_pif_results_table5. For a handful of INDIVIDUAL cells it
#   only re-derives the two quantities that must agree by construction:
#
#     (a) "my" engine : aaf_confint() fed with exactly the inputs that
#                       pif2_run_pif_grid() gives the cell, with the scenario
#                       arguments removed -> the BASELINE AAF of the cell, i.e.
#                       its PAF.
#     (b) reference   : compute_cv_aaf_from_registry(), the independent wide-table
#                       AAF producer used elsewhere in the project.
#
#   If (a) != (b), the Table 5 PIFs are standing on a PAF that the reference
#   implementation does not reproduce.
#
# WHY THE SAVED RDS IS NOT ENOUGH
#   pif2_pif_results_table5_*.rds stores pif / pif_low / pif_up only; it carries
#   NO baseline-AAF column. So this check cannot be done by reading the saved
#   results. Both sides must be re-derived -- but for single cells (seconds),
#   never for the whole grid.
# ---------------------------------------------------------------------------
# ---- Fact 1: the SHAPE of a compute_cv table (this is where a naive check dies)
#   compute_cv_aaf_from_registry() returns WIDE tables: one row per Year, one
#   column per age group, named "<prefix><group>_point" / "_lower" / "_upper",
#   plus a `disease` column. See .aaf_make_output_df() in aaf_unified.R:
#       out <- data.frame(Year = years)
#       out[[paste0(prefix, g, "_point")]] <- NA_real_
#   There is NO age_group column. The age group is encoded in the COLUMN NAME.
#   Filtering on `tbl$age_group == "..."` yields NULL == "..." -> logical(0) ->
#   a 0-row subset -> numeric(0): a check that neither passes nor fails honestly.

# ---- Fact 2: the pipeline's four age groups (ENPG edad_tramo, 15-64/15-65) ----
#   group 1 = 15-29    group 2 = 30-44    group 3 = 45-59    group 4 = 60-65
#   Source: enpg_cluster_factors_by_year_variable_tramo.csv (columns age_group /
#   age_group_label). There is NO "35-44" group in this pipeline: the cell that
#   "Male2_point" refers to is group 2 = 30-44.
pif2_t5_age_labels <- c("1" = "15-29", "2" = "30-44", "3" = "45-59", "4" = "60-65")
# ---- Fact 3: compute_cv indexes the HED prevalence POSITIONALLY ---------------
#   pif2_resolve_cell_inputs() reads p_hed at year_index = which(pif2_years == year).
#   .aaf_run_hed_table() reads it at row_index = i, the position of the year INSIDE
#   the `years` argument it was handed:
#       for (i in seq_along(years)) { y <- years[[i]]
#         p_hed <- .aaf_prop_from_group_list(p_hed_list, i, g, y) }
#   Therefore compute_cv MUST be called with the FULL pif2_years vector. Calling it
#   with years = 2018 alone would set i = 1 and silently read the 2012 HED
#   prevalence: a reference value that is wrong but perfectly plausible.

# ---- Fact 4: compute_cv CAPS the stored point estimate at 1 -------------------
#   .aaf_run_hed_table() stores min(res$point_estimate, 1), while "my" side returns
#   the raw aaf_confint()$point_estimate. We compare RAW on purpose: an AAF > 1
#   would then fail this check loudly instead of being silently capped into
#   agreement -- and an IHD/IS AAF > 1 is anomalous in the first place. (For the
#   same reason we only compare _point: the stored _lower/_upper are additionally
#   reshaped by min(lower, point) / min(max(upper, point), 1) and are not raw.)

# ---- Monte Carlo settings: identical to the original paf_congruence check -----
.t5_cfg <- list(n_sim = 200L, n_pca = 200L, n_cores = 1L,
                inner_parallel = FALSE, outer_parallel = FALSE)

# ---- Validation harness ------------------------------------------------------
# Re-uses t5_val / t5_check from the Table 5 run chunk when they are still in
# memory, so the extra checks land in the same pif2_table5_validation table.
if (!exists("t5_val", inherits = TRUE)) t5_val <- list()
if (!exists("t5_check", inherits = TRUE)) {
  t5_check <- function(id, cond, note = "") {
    t5_val[[length(t5_val) + 1L]] <<- data.frame(
      check = id, pass = isTRUE(cond), note = note, stringsAsFactors = FALSE)
  }
}
# Idempotency: re-running this chunk must REPLACE its own rows, never append
# duplicates to t5_val.
t5_drop_check <- function(id) {
  if (!length(t5_val)) return(invisible(NULL))
  t5_val <<- t5_val[!vapply(t5_val, function(r) identical(r$check, id), logical(1))]
  invisible(NULL)
}
# ---- Read ONE point estimate out of a wide compute_cv table, LOUDLY -----------
# Uses tbl[[col]][row], never tbl[row, col]: the latter returns a 1-column tibble
# if the table is ever a tibble, and numeric(0) when the row filter matches
# nothing -- both of which turn a real failure into a silent one.
pif2_t5_cv_point <- function(cv_obj, output_name, prefix, group, year) {
  tbl <- cv_obj$tables[[output_name]]
  if (is.null(tbl)) {
    stop(sprintf(
      "compute_cv table '%s' is absent. Present: %s. (Was it listed in target_output_names?)",
      output_name, paste(names(cv_obj$tables), collapse = ", ")))
  }
  col <- paste0(prefix, group, "_point")
  if (!col %in% names(tbl)) {
    stop(sprintf("Column '%s' absent from compute_cv table '%s'. Present: %s",
                 col, output_name, paste(names(tbl), collapse = ", ")))
  }
  row <- which(tbl$Year == year)
  if (length(row) != 1L) {
    stop(sprintf("Year %s matched %d rows in compute_cv table '%s' (years present: %s).",
                 as.character(year), length(row), output_name,
                 paste(tbl$Year, collapse = ", ")))
  }
  val <- tbl[[col]][row]
  if (!is.numeric(val) || length(val) != 1L || !is.finite(val)) {
    stop(sprintf("compute_cv %s[Year=%s, %s] is not a finite scalar (got: %s).",
                 output_name, as.character(year), col, paste(format(val), collapse = ",")))
  }
  val
}
# ---- Reference AAF tables: ONE compute_cv run per output_name, then reused ----
# The compute_cv run is the only non-trivial cost in this chunk (7 years x 4
# groups at n_sim = 200), so every (group, year) check for the same table reuses
# it instead of recomputing.
pif2_t5_cv_cache <- list()
pif2_t5_cv_get <- function(output_name) {
  if (!is.null(pif2_t5_cv_cache[[output_name]])) return(pif2_t5_cv_cache[[output_name]])
  spec <- pif2_table5_output_spec[pif2_table5_output_spec$output_name == output_name, ]
  if (nrow(spec) != 1L) stop("'", output_name, "' is not a Table 5 output_name.")
  cv <- compute_cv_aaf_from_registry(
    registry         = pif2_rr_registries[[spec$registry_scope]],
    g_fem_hed_list   = pif2_exposure_inputs$g_fem_hed_list,
    g_male_hed_list  = pif2_exposure_inputs$g_male_hed_list,
    p_abs_list_fem   = pif2_exposure_inputs$p_abs_list_fem,
    p_abs_list_male  = pif2_exposure_inputs$p_abs_list_male,
    p_form_list_fem  = pif2_exposure_inputs$p_form_list_fem,
    p_form_list_male = pif2_exposure_inputs$p_form_list_male,
    p_hed_list_fem   = pif2_exposure_inputs$p_hed_list_fem,
    p_hed_list_male  = pif2_exposure_inputs$p_hed_list_male,
    x_vals           = pif2_exposure_inputs$x_vals,
    years            = pif2_years,   # FULL vector -- see Fact 3. Do NOT narrow this.
    age_groups       = 1:4,
    age_scope        = "15_64",
    prev_method               = pif2_aaf_uncertainty$prev_method,
    neff                      = pif2_aaf_uncertainty$neff,
    design_factor             = pif2_aaf_uncertainty$design_factor,
    neff_consumption          = pif2_aaf_uncertainty$neff_consumption,
    design_factor_consumption = pif2_aaf_uncertainty$design_factor_consumption,
    fd_uncertainty   = TRUE,
    n_sim            = .t5_cfg$n_sim,
    n_pca            = .t5_cfg$n_pca,
    seed             = pif2_aaf_mc$seed,
    use_parallel     = FALSE,
    target_output_names = output_name)
  pif2_t5_cv_cache[[output_name]] <<- cv
  cv
}
# ---- "My" side: the cell's BASELINE AAF, built exactly as the PIF grid does ----
# The scenario row is only supplied because pif2_build_pif_args() requires one;
# dropping scenario / shift / shift_hed from the argument list turns a pif_confint()
# input into an aaf_confint() input, which is the PAF of the cell. This mirrors the
# original paf_congruence check on purpose.
pif2_t5_my_paf <- function(output_name, group, year) {
  spec  <- pif2_table5_output_spec[pif2_table5_output_spec$output_name == output_name, ]
  if (nrow(spec) != 1L) stop("'", output_name, "' is not a Table 5 output_name.")
  group <- as.integer(group)
  year  <- as.integer(year)
  if (!year %in% pif2_years) {
    stop(sprintf("Year %d is not a survey wave. pif2_years = %s.",
                 year, paste(pif2_years, collapse = ", ")))
  }
  rec <- pif2_lookup_record(spec, group)                      # age_scope = "15_64"
  inp <- pif2_resolve_cell_inputs(spec, rec, year, group, pif2_exposure_inputs)
  if (!isTRUE(inp$ok)) {
    stop(sprintf("Cell %s / group %d / year %d has no usable inputs: %s",
                 output_name, group, year, inp$reason))
  }
  args <- pif2_build_pif_args(
    spec, rec, inp,
    pif2_scenario_grid[pif2_scenario_grid$scenario_id == "volume_reduction_20", ],
    pif2_exposure_inputs, pif2_aaf_uncertainty, pif2_aaf_mc,
    year, group, .t5_cfg)
  args <- pif2_as_aaf_args(args)
  do.call(aaf_confint, args)$point_estimate
}
# ---- One congruence check for one (table, group, year) cell -------------------
pif2_t5_congruence <- function(output_name, group, year, tol = 1e-9) {
  spec  <- pif2_table5_output_spec[pif2_table5_output_spec$output_name == output_name, ]
  group <- as.integer(group)
  year  <- as.integer(year)
  mine  <- pif2_t5_my_paf(output_name, group, year)
  ref   <- pif2_t5_cv_point(pif2_t5_cv_get(output_name), output_name, spec$prefix, group, year)
  id    <- sprintf("paf_congruence_%s_g%d_%d", output_name, group, year)
  t5_drop_check(id)
  t5_check(
    id,
    isTRUE(abs(mine - ref) < tol),
    sprintf("age %s (group %d, col %s%d_point) | my AAF=%.5f compute_cv=%.5f | |diff|=%.2e%s",
            pif2_t5_age_labels[[as.character(group)]], group, spec$prefix, group,
            mine, ref, abs(mine - ref),
            if (mine > 1) " | WARNING: raw AAF > 1, compute_cv caps at 1 (see Fact 4)" else ""))
  invisible(NULL)
}
# ---- The cells to check ------------------------------------------------------
# Row 1 is the cell you asked for: IHD, men, year 2018, age group 2 = 30-44.
# The remaining rows widen coverage to all four Table 5 tables, to both Adam RR
# bands, and to the first year (i = 1) and the last year (i = length(pif2_years)),
# which is exactly where the positional p_hed alignment of Fact 3 would break first.
#
# Stated honestly: the Table 5 registries repeat the SAME RR across the three Adam
# bands (Table 5 is not age-banded -- see table5_make_registry), so the group-1 row
# does NOT exercise a different RR curve here. What it exercises is the age-band
# record lookup and, above all, the exposure-cell + positional-year resolution.
pif2_t5_congruence_cells <- data.frame(
  output_name = c("ihd_male", "ihd_male", "ihd_female", "is_male", "is_female"),
  group       = c(2L,         1L,         4L,           3L,        2L),
  year        = c(2018L,      2012L,      2024L,        2016L,     2020L),
  stringsAsFactors = FALSE)

pif2_message("[table5-congruence] Re-deriving %d single cells (NOT the PIF grid); %d compute_cv table(s) at n_sim=%d.",
             nrow(pif2_t5_congruence_cells),
             length(unique(pif2_t5_congruence_cells$output_name)), .t5_cfg$n_sim)

for (i in seq_len(nrow(pif2_t5_congruence_cells))) {
  pif2_t5_congruence(pif2_t5_congruence_cells$output_name[[i]],
                     pif2_t5_congruence_cells$group[[i]],
                     pif2_t5_congruence_cells$year[[i]])
}
# ---- Refresh the master Table 5 validation table -----------------------------
pif2_table5_validation <- dplyr::bind_rows(t5_val)
pif2_table5_n_fail     <- sum(!pif2_table5_validation$pass)
print(knitr::kable(pif2_table5_validation,
                   caption = "Table 5 (PUC) IHD/IS PIF validation (incl. extra PAF-congruence cells)")) |> print()
pif2_message("[table5-congruence] %d/%d checks passed (%d extra PAF-congruence cells added).",
             sum(pif2_table5_validation$pass), nrow(pif2_table5_validation),
             nrow(pif2_t5_congruence_cells))
if (pif2_table5_n_fail > 0L) {
  stop("Table 5 PIF validation FAILED: ",
       paste(pif2_table5_validation$check[!pif2_table5_validation$pass], collapse = ", "))
}
message(sprintf("pif2-table5-paf-congruence-extra elapsed minutes: %.2f", pif2_elapsed_min(.t0)))

[table5-congruence] Re-deriving 5 single cells (NOT the PIF grid); 4 compute_cv table(s) at n_sim=200.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.
[aaf-args] Dropped 3 PIF-only argument(s) before calling aaf_confint(): scenario, shift, shift_hed.


Table: Table 5 (PUC) IHD/IS PIF validation (incl. extra PAF-congruence cells)

|check                             |pass |note                                                                                                             |
|:---------------------------------|:----|:----------------------------------------------------------------------------------------------------------------|
|i

~ 51 minutes


## Session info


In [28]:

#| label: pif2-session-info
#| echo: true
#| results: "hold"

.t0 <- Sys.time()

message(paste0("R library: ", Sys.getenv("R_LIBS_USER")))
message(paste0("Date: ", withr::with_locale(new = c("LC_TIME" = "C"), code = Sys.time())))
message(paste0("Editor context: ", getwd()))
print(utils::sessionInfo())
sesion_info <- devtools::session_info()
message(sprintf(
  "pif2-session-info elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


tabla_pkg <- dplyr::select(
  tibble::as_tibble(sesion_info$packages),
  package,
  loadedversion,
  source
)

tabla_pkg <- tibble::rowid_to_column(tabla_pkg, var = "row_number")

names(tabla_pkg) <- c("Row number", "Package", "Version", "Source")

htmltools::browsable(
  htmltools::tags$div(
    style = "
      max-height: 420px;
      overflow: auto;
      border: 1px solid #ddd;
      font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
      font-size: 70%;
      line-height: 0.75em;
      width: 100%;
    ",
    htmltools::tags$caption(
      style = "
        caption-side: top;
        text-align: left;
        display: block;
        padding: 6px 4px;
        font-size: 120%;
        line-height: 1.2em;
      ",
      htmltools::em("R packages")
    ),
    htmltools::tags$table(
      style = "
        border-collapse: collapse;
        width: max-content;
        min-width: 100%;
        white-space: nowrap;
      ",
      htmltools::tags$thead(
        htmltools::tags$tr(
          lapply(names(tabla_pkg), function(nm) {
            htmltools::tags$th(
              style = "
                position: sticky;
                top: 0;
                z-index: 2;
                background: #f3f3f3;
                border-bottom: 1px solid #ccc;
                padding: 4px 8px;
                text-align: left;
                white-space: nowrap;
              ",
              nm
            )
          })
        )
      ),
      htmltools::tags$tbody(
        lapply(seq_len(nrow(tabla_pkg)), function(i) {
          htmltools::tags$tr(
            lapply(tabla_pkg[i, ], function(x) {
              htmltools::tags$td(
                style = "
                  border-bottom: 1px solid #eee;
                  padding: 3px 8px;
                  white-space: nowrap;
                ",
                as.character(x)
              )
            })
          )
        })
      )
    )
  )
)

R library: C:\Users\nDP\AppData\Local/R/win-library/4.4
Date: 2026-07-21 05:30:14.042342
Editor context: c:/Users/nDP/Desktop/ACC1240138_private/__andres_control
R version 4.4.1 (2024-06-14 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 26200)

Matrix products: default


Random number generation:
 RNG:     L'Ecuyer-CMRG 
 Normal:  Inversion 
 Sample:  Rejection 
 
locale:
[1] LC_COLLATE=Spanish_Chile.utf8  LC_CTYPE=Spanish_Chile.utf8   
[3] LC_MONETARY=Spanish_Chile.utf8 LC_NUMERIC=C                  
[5] LC_TIME=Spanish_Chile.utf8    

time zone: America/Santiago
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] digest_0.6.39    tidyr_1.3.2      R6_2.6.1         fastmap_1.2.0   
 [5] xfun_0.57        cellranger_1.1.0 tidyselect_1.2.1 readxl_1.4.5    
 [9] magrittr_2.0.4   glue_1.8.0       tibble_3.3.1     parallel_4.4.1  
[13] knitr_1.5

Warning message:
In system2("quarto", "-V", stdout = TRUE, env = paste0("TMPDIR=",  :
  running command '"quarto" TMPDIR=C:/Users/nDP/AppData/Local/Temp/Rtmp4Y125N/file7ba87adc2da4 -V' had status 1


pif2-session-info elapsed minutes: 0.04


1,cachem,1.1.0,CRAN (R 4.4.3)
2,cellranger,1.1.0,CRAN (R 4.4.3)
3,cli,3.6.5,CRAN (R 4.4.3)
4,devtools,2.5.0,CRAN (R 4.4.3)
5,digest,0.6.39,CRAN (R 4.4.3)
6,dplyr,1.2.0,CRAN (R 4.4.3)
7,ellipsis,0.3.3,CRAN (R 4.4.3)
8,evaluate,1.0.5,CRAN (R 4.4.3)
9,fastmap,1.2.0,CRAN (R 4.4.3)
10,fs,2.1.0,CRAN (R 4.4.3)
11,generics,0.1.4,CRAN (R 4.4.3)
